In [ ]:
# DARK-ENTRAIN: one-cell numerical reproduction of the principal slow encounter.
# MODE = "replay" uses the supplied saved fields; MODE = "fresh" evolves both runs from -50 Myr.
# If the notebook is downloaded alone, the default automatically selects a fresh run.
from pathlib import Path
import os, sys, base64, shutil, subprocess, tempfile, zipfile, importlib.util
for name in ("numpy", "scipy", "pandas", "matplotlib"):
    if importlib.util.find_spec(name) is None:
        raise RuntimeError("Install the dependencies in requirements.txt first: " + name)
source = Path.cwd()
packs = [source/f"DM{dm}_data_{i:02d}.zip" for dm in (0,1) for i in range(1,5)]
if any(p.exists() for p in packs) and not all(p.exists() for p in packs):
    raise RuntimeError("Incomplete saved-data release: all eight DM data archives are required.")
packed = all(p.exists() for p in packs)
mode = "replay" if packed or all((source/f"full_DM{dm}"/f"frame_+70_{dm}_+050.0.npz").exists() for dm in (0,1)) else "fresh"
root = Path(tempfile.mkdtemp(prefix="dark_entrain_", dir=source))
embedded = {'CP29_SOURCE.py': '...', 'recovery_fast.py': 'IiIiQ29tcGlsZWQgc2NhbGFyLWxvb3AgYmFja2VuZCBmb3IgcmVjb3ZlcmVkIENQMjkgZGlzY3JldGl6YXRpb24uIE5vIGZhc3RtYXRoLiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBudW1iYSBpbXBvcnQgbmppdAppbXBvcnQgQ1AyOV9TT1VSQ0UgYXMgcwpHPXMuR0FNTUEKQG5qaXQoaW5saW5lPSdhbHdheXMnKQpkZWYgcChyLG0sbixlKTpyZXR1cm4gKEctMSkqKGUtLjUqKG0qbStuKm4pL21heChyLDFlLTEyKSkKQG5qaXQoY2FjaGU9VHJ1ZSkKZGVmIHNsb3BlcyhVLGUpOgogbixtPVUuc2hhcGVbMTpdOyBTPW5wLnplcm9zX2xpa2UoVSkKIGZvciBpIGluIHJhbmdlKDEsbi0xKToKICB3PWVbaSsxXS1lW2ldOyBhPS41KihlW2krMV0tZVtpLTFdKTtiPS41KihlW2krMl0tZVtpXSkKICBmb3IgaiBpbiByYW5nZShtKToKICAgZm9yIGsgaW4gcmFuZ2UoNSk6CiAgICBsPShVW2ssaSxqXS1VW2ssaS0xLGpdKS9hO3I9KFVbayxpKzEsal0tVVtrLGksal0pL2IKICAgIGlmIGwqcj4wOlNbayxpLGpdPW5wLnNpZ24obCkqbWluKGFicyhsKSxhYnMocikpKncKICAgZm9yIGsgaW4gcmFuZ2UoMTYpOgogICAgcj1VWzAsaSxqXTtteD1VWzEsaSxqXTtteT1VWzIsaSxqXTtlbj1VWzMsaSxqXQogICAgZHI9LjUqU1swLGksal07ZHg9LjUqU1sxLGksal07ZHk9LjUqU1syLGksal07ZGU9LjUqU1szLGksal0KICAgIGlmIG1pbihyLWRyLHIrZHIpPjFlLTkgYW5kIG1pbihwKHItZHIsbXgtZHgsbXktZHksZW4tZGUpLHAocitkcixteCtkeCxteStkeSxlbitkZSkpPjFlLTk6YnJlYWsKICAgIGZvciB2IGluIHJhbmdlKDUpOlNbdixpLGpdKj0uNQogICBlbHNlOnJhaXNlIFZhbHVlRXJyb3IoJ3JlY29uc3RydWN0aW9uJykKIHJldHVybiBTCkBuaml0KGNhY2hlPVRydWUpCmRlZiBmYWNlcyhVLGUpOgogbixtPVUuc2hhcGVbMTpdO1M9c2xvcGVzKFUsZSk7Rj1ucC5lbXB0eSgoNSxuKzEsbSkpO25iPTAKIGZvciBpIGluIHJhbmdlKG4rMSk6CiAgaWw9bWF4KDAsaS0xKTtpcj1taW4obi0xLGkpCiAgZm9yIGogaW4gcmFuZ2UobSk6CiAgIHJsPVVbMCxpbCxqXSsuNSpTWzAsaWwsal07cnI9VVswLGlyLGpdLS41KlNbMCxpcixqXQogICBtbD1VWzEsaWwsal0rLjUqU1sxLGlsLGpdO21yPVVbMSxpcixqXS0uNSpTWzEsaXIsal0KICAgbmw9VVsyLGlsLGpdKy41KlNbMixpbCxqXTtucj1VWzIsaXIsal0tLjUqU1syLGlyLGpdCiAgIGVsPVVbMyxpbCxqXSsuNSpTWzMsaWwsal07ZXI9VVszLGlyLGpdLS41KlNbMyxpcixqXQogICB3bD1VWzQsaWwsal0rLjUqU1s0LGlsLGpdO3dyPVVbNCxpcixqXS0uNSpTWzQsaXIsal0KICAgcGw9cChybCxtbCxubCxlbCk7cHI9cChycixtcixucixlcikKICAgaWYgbWluKHJsLHJyLHBsLHByKTw9MDpyYWlzZSBWYWx1ZUVycm9yKCdiYWQgZmFjZScpCiAgIHVsPW1sL3JsO3VyPW1yL3JyO3ZsPW5sL3JsO3ZyPW5yL3JyO2NsPW5wLnNxcnQoRypwbC9ybCk7Y3I9bnAuc3FydChHKnByL3JyKQogICBzbD1taW4odWwtY2wsdXItY3IpO3NyPW1heCh1bCtjbCx1citjcikKICAgc209KHByLXBsK3JsKnVsKihzbC11bCktcnIqdXIqKHNyLXVyKSkvKHJsKihzbC11bCktcnIqKHNyLXVyKSkKICAgcHM9LjUqKChwbCtybCooc2wtdWwpKihzbS11bCkpKyhwcityciooc3ItdXIpKihzbS11cikpKQogICByc2w9cmwqKHNsLXVsKS8oc2wtc20pO3Jzcj1yciooc3ItdXIpLyhzci1zbSkKICAgZmw9KG1sLG1sKnVsK3BsLG5sKnVsLChlbCtwbCkqdWwsd2wqdWwpO2ZyPShtcixtcip1citwcixucip1ciwoZXIrcHIpKnVyLHdyKnVyKQogICBMPShybCxtbCxubCxlbCx3bCk7Uj0ocnIsbXIsbnIsZXIsd3IpCiAgIGlmIG5vdCBucC5pc2Zpbml0ZShzbSkgb3Igbm90IG5wLmlzZmluaXRlKHBzKSBvciBwczw9MCBvciByc2w8PTAgb3IgcnNyPD0wIG9yIHNsPnNtIG9yIHNtPnNyOgogICAgZGlzcz1tYXgoYWJzKHVsKStjbCxhYnModXIpK2NyKTtuYis9MQogICAgZm9yIGsgaW4gcmFuZ2UoNSk6RltrLGksal09LjUqKGZsW2tdK2ZyW2tdKS0uNSpkaXNzKihSW2tdLUxba10pCiAgIGVsaWYgc2w+PTA6CiAgICBmb3IgayBpbiByYW5nZSg1KTpGW2ssaSxqXT1mbFtrXQogICBlbGlmIHNtPj0wOgogICAgZXM9KChzbC11bCkqZWwtcGwqdWwrcHMqc20pLyhzbC1zbSk7c3Q9KHJzbCxyc2wqc20scnNsKnZsLGVzLHdsKnJzbC9ybCkKICAgIGZvciBrIGluIHJhbmdlKDUpOkZbayxpLGpdPWZsW2tdK3NsKihzdFtrXS1MW2tdKQogICBlbGlmIHNyPjA6CiAgICBlcz0oKHNyLXVyKSplci1wcip1citwcypzbSkvKHNyLXNtKTtzdD0ocnNyLHJzcipzbSxyc3IqdnIsZXMsd3IqcnNyL3JyKQogICAgZm9yIGsgaW4gcmFuZ2UoNSk6RltrLGksal09ZnJba10rc3IqKHN0W2tdLVJba10pCiAgIGVsc2U6CiAgICBmb3IgayBpbiByYW5nZSg1KTpGW2ssaSxqXT1mcltrXQogcmV0dXJuIEYsbmIKQG5qaXQoY2FjaGU9VHJ1ZSkKZGVmIGNvb2xfa2VybmVsKFUsZHQsVHcsUDAsbncsY2EsZ2gsVE1JTik6CiBBPWR0KmNhKmdoCiBmb3IgaSBpbiByYW5nZShVLnNoYXBlWzFdKToKICBmb3IgaiBpbiByYW5nZShVLnNoYXBlWzJdKToKICAgcj1VWzAsaSxqXTtwMD1tYXgocChyLFVbMSxpLGpdLFVbMixpLGpdLFVbMyxpLGpdKSwxZS04KTtvbGQ9VHcqcDAvKFAwKnIpO249bncqcgogICBsbz1UTUlOO2hpPW9sZCtBO3o9bWluKG1heChvbGQsbG8pLGhpKQogICBmb3IgayBpbiByYW5nZSgxNik6CiAgICBMMT0xZTcqbnAuZXhwKC0xMTg0MDAvKHorMTAwMCkpO0wyPS4wMTQqbnAuc3FydCh6KSpucC5leHAoLTkyL3opO2Y9ei1vbGQtQSooMS1uKihMMStMMikpO2Rlcj0xK0EqbiooTDEqMTE4NDAwLyh6KzEwMDApKioyK0wyKiguNS96KzkyL3oqKjIpKQogICAgaWYgYWJzKGYpLygxK29sZCtBKTwyZS0xMjpicmVhawogICAgaWYgZjwwOmxvPXoKICAgIGVsaWYgZj4wOmhpPXoKICAgIG5ldz16LWYvZGVyO3o9bmV3IGlmIGxvPD1uZXc8PWhpIGVsc2UgLjUqKGxvK2hpKQogICBlbHNlOnJldHVybiBGYWxzZQogICBVWzMsaSxqXSs9KHIqUDAqei9Udy1wMCkvKEctMSkKIHJldHVybiBUcnVlCm9yaWdfY29vbGluZz1zLmNvb2xpbmdfaW1wbGljaXQKb3JpZ19yaHM9cy5yaHMyNgpkZWYgY29vbChVLGR0KToKIFY9VS5jb3B5KCkKIGlmIGNvb2xfa2VybmVsKFYsZHQscy5UdyxzLlAwX1JBRCxzLm53LHMuQ09PTF9BLHMuR0FNTUFfSEVBVCxzLlRNSU4pOnJldHVybiBWCiByZXR1cm4gb3JpZ19jb29saW5nKFUsZHQpCmRlZiByaHMoVSxYLFkseCx5LHQsd2l0aF9kbSk6CiBmeCxieD1mYWNlcyhVLHgpO2Z5LGJ5PWZhY2VzKG5wLmFzY29udGlndW91c2FycmF5KFVbWzAsMiwxLDMsNF1dLnRyYW5zcG9zZSgwLDIsMSkpLHkpO2Z5PWZ5W1swLDIsMSwzLDRdXS50cmFuc3Bvc2UoMCwyLDEpCiBkPS0oZnhbOiwxOiw6XS1meFs6LDotMSw6XSkvbnAuZGlmZih4KVtOb25lLDosTm9uZV0tKGZ5WzosOiwxOl0tZnlbOiw6LDotMV0pL25wLmRpZmYoeSlbTm9uZSxOb25lLDpdCiByaG8sdngsdnkscHIsYT1zLnByaW1pdGl2ZXMoVSk7YXg9MipzLk9NRUdBKnZ5KzIqcy5RKnMuT01FR0Eqcy5PTUVHQSpYO2F5PS0yKnMuT01FR0EqdngKIGlmIHdpdGhfZG06CiAgbXg9bnAuaW50ZXJwKHQscy5mdWxsX3R0LHMuZnVsbF90cmFqWzosMF0pO215PW5wLmludGVycCh0LHMuZnVsbF90dCxzLmZ1bGxfdHJhals6LDFdKTtzeD1YLW14O3N5PVktbXk7cnI9bnAuc3FydChzeCpzeCtzeSpzeSsuNSoqMik7bW09bnAuaW50ZXJwKHJyLHMucmdyaWQscy5tZ3JpZCxsZWZ0PTAuLHJpZ2h0PXMubWdyaWRbLTFdKTthYT0tcy5HMiptbS8ocnIqKjMpO2F4Kz1hYSpzeDtheSs9YWEqc3kKIGRbMV0rPXJobypheDtkWzJdKz1yaG8qYXk7ZFszXSs9cmhvKih2eCpheCt2eSpheSkKIHJldHVybiBkLGZ4WzRdLGZ5WzRdLGJ4K2J5Cg==', 'CP36_RUN.py': 'IiIiSW5kZXBlbmRlbnQgZnVsbC1kdXJhdGlvbiB3aWRlLXN0cmlwIHBhaXI7IG5vIHJlc3RhcnQgZnJvbSBDUDM1LiIiIgppbXBvcnQgc3lzLCBvcywganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBDUDI5X1NPVVJDRSBhcyBzClJPT1Q9UGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudAppZiBvcy5lbnZpcm9uLmdldCgnQ1AzNl9OVU1QWScpIT0nMSc6CiB0cnk6CiAgaW1wb3J0IHJlY292ZXJ5X2Zhc3QgYXMgZmFzdAogIHMucmhzMjY9ZmFzdC5yaHM7IHMuY29vbGluZ19pbXBsaWNpdD1mYXN0LmNvb2wKIGV4Y2VwdCBJbXBvcnRFcnJvcjogcGFzcwoKZGVmIHNldHVwKCk6CiB3aXRoIG5wLmxvYWQoUk9PVC8nR1JJRC5ucHonKSBhcyBnOiB4PWdbJ3gnXTsgeT1nWyd5J10KIHMuU1RBUlQ9LTUwLjtzLkVORD01MC47cy5WUio9LjU7cy5mdWxsX3R0LHMuZnVsbF90cmFqLHMuZnVsbF9wb3M9cy5vcmJpdF9mdWxsX3JhbmdlKCkKIHMuU0FNUExFUz10dXBsZShucC5hcmFuZ2UoLTUwLDUxLDUuKSk7cy5ncmlkMjY9bGFtYmRhIGg6KHgseSkKIHJldHVybiB4LHkKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiBkbT1pbnQoc3lzLmFyZ3ZbMV0pO3gseT1zZXR1cCgpO1UsWCxZPXMuaW5pdDI2KHgseSw3MC4pCiBQYXRoKCdjb25maWd1cmF0aW9uLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdChkbT1kbSxzdGFydD0tNTAsZW5kPTUwLG1lc2g9bGlzdChYLnNoYXBlKSxpbml0aWFsaXphdGlvbj0nYW5hbHl0aWMgY2VsbCBhdmVyYWdlcycsbWluX2R4PWZsb2F0KG5wLmRpZmYoeCkubWluKCkpLG1pbl9keT1mbG9hdChucC5kaWZmKHkpLm1pbigpKSxpbml0aWFsX21hc3M9ZmxvYXQobnAuc3VtKFVbNF0qbnAuZGlmZih4KVs6LE5vbmVdKm5wLmRpZmYoeSlbTm9uZSw6XSkpKSxpbmRlbnQ9MikpCiBzLmludGVncmF0ZTI2KDEuLDcwLixkbSxlbmQ9ZmxvYXQob3MuZW52aXJvbi5nZXQoJ0NQMzZfRU5EJywnNTAnKSkpCg==', 'CP37_ANALYZE.py': 'IiIiSW5kZXBlbmRlbnQgc25hcHNob3QgY2hlY2tzLCBwYWlyZWQgY29tcGFyaXNvbnMsIGFuZCBzaGFwZS1wcmVzZXJ2aW5nIGRpc3BsYXkuCk5vIGV4dHJhcG9sYXRpb24gYmV5b25kIHRoZSBsYXN0IGNvbW1vbiBzYXZlZCB0aW1lLiBVc2VzIGFyY2hpdmVkIENQMzEgYW5kIENQMzUgdGFibGVzLgoiIiIKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBqc29uLCBzeXMKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKZnJvbSBzY2lweS5pbnRlZ3JhdGUgaW1wb3J0IGN1bXVsYXRpdmVfdHJhcGV6b2lkCmZyb20gc2NpcHkuaW50ZXJwb2xhdGUgaW1wb3J0IFBjaGlwSW50ZXJwb2xhdG9yCmltcG9ydCBtYXRwbG90bGliCm1hdHBsb3RsaWIudXNlKCdBZ2cnKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CmltcG9ydCBDUDI5X1NPVVJDRSBhcyBzCmZyb20gQ1AzNl9SVU4gaW1wb3J0IHNldHVwCnJvb3Q9UGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudApzZXR1cCgpCm90PXMuZnVsbF90dDsgb3A9cy5mdWxsX3RyYWo7IG92PW5wLmdyYWRpZW50KG9wLG90LGF4aXM9MCkKcj1ucC5nZW9tc3BhY2UoLjUscy5ydCw0MDAwMCk7IGludGU9Y3VtdWxhdGl2ZV90cmFwZXpvaWQobnAuaW50ZXJwKHIscy5yZ3JpZCxzLm1ncmlkKS9yKioyLHIsaW5pdGlhbD0wKTsgTT1zLm1ncmlkWy0xXTsgcG90PS1zLkcyKihNL3MucnQraW50ZVstMV0taW50ZSkKcm93cz1bXTsgY29tcGFyaXNvbnM9W10KZm9yIGRtIGluICgwLDEpOgogICAgZm9sZGVyPXJvb3QvZidmdWxsX0RNe2RtfSc7IGQ9cGQucmVhZF9jc3YoZm9sZGVyL2YnaDFfcGhhc2UrNzBfZG17ZG19LmNzdicpLnNvcnRfdmFsdWVzKCd0X015cicpOyBvbGQ9cGQucmVhZF9jc3Yocm9vdC9mJ29yaWdpbmFsX0RNe2RtfS5jc3YnKTsgaW5pdD1Ob25lOyBzdGFydD17fQogICAgZm9yIF8sYSBpbiBkLml0ZXJyb3dzKCk6CiAgICAgICAgdD1mbG9hdChhLnRfTXlyKQogICAgICAgIHdpdGggbnAubG9hZChmb2xkZXIvZidmcmFtZV8rNzBfe2RtfV97dDorMDYuMWZ9Lm5weicpIGFzIHo6CiAgICAgICAgICAgIFU9elsnVSddO3g9elsneCddO3k9elsneSddOyBhc3NlcnQgYWJzKGZsb2F0KHpbJ3QnXSktdCk8MWUtOAogICAgICAgIFgsWT1ucC5tZXNoZ3JpZCgoeFs6LTFdK3hbMTpdKS8yLCh5WzotMV0reVsxOl0pLzIsaW5kZXhpbmc9J2lqJyk7IEE9bnAuZGlmZih4KVs6LE5vbmVdKm5wLmRpZmYoeSlbTm9uZSw6XTsgVz1VWzRdKkE7IHRvdGFsPWZsb2F0KFcuc3VtKCkpOyByaG89VVswXTsgdng9VVsxXS9yaG87IHZ5PVVbMl0vcmhvOyBQPShzLkdBTU1BLTEpKihVWzNdLS41KihVWzFdKioyK1VbMl0qKjIpL3Jobyk7IFQ9cy5UdypQLyhzLlAwX1JBRCpyaG8pCiAgICAgICAgYXNzZXJ0IG5wLmlzZmluaXRlKFUpLmFsbCgpIGFuZCByaG8ubWluKCk+MCBhbmQgUC5taW4oKT4wIGFuZCBXLm1pbigpPj0tMWUtMTIKICAgICAgICBpZiBpbml0IGlzIE5vbmU6IGluaXQ9dG90YWw7c3RhcnQ9e2M6ZmxvYXQoV1tYPGNdLnN1bSgpKSBmb3IgYyBpbiAoLTQwLDAsNDApfQogICAgICAgIG14LG15PVtucC5pbnRlcnAodCxvdCxvcFs6LGldKSBmb3IgaSBpbiAoMCwxKV07IHV4LHV5PVtucC5pbnRlcnAodCxvdCxvdls6LGldKSBmb3IgaSBpbiAoMCwxKV07IGR4PVgtbXg7ZHk9WS1teTsgUj1ucC5oeXBvdChkeCxkeSk7cnI9bnAuc3FydChSKioyKy41KioyKTtwaGk9bnAud2hlcmUocnI8PXMucnQsbnAuaW50ZXJwKHJyLHIscG90KSwtcy5HMipNL3JyKTsgQj0uNSooKHZ4LXV4LXMuT01FR0EqZHkpKioyKyh2eS11eStzLk9NRUdBKmR4KSoqMikrcy5HQU1NQS8ocy5HQU1NQS0xKSpQL3JobytwaGkKICAgICAgICBpaT1ucC5hcmdzb3J0KFQucmF2ZWwoKSk7IHRjPVQucmF2ZWwoKVtpaV07IHdjPW5wLmN1bXN1bShXLnJhdmVsKClbaWldKTsgcTUwPWZsb2F0KHRjW25wLnNlYXJjaHNvcnRlZCh3YywuNSp0b3RhbCldKTtxOTA9ZmxvYXQodGNbbnAuc2VhcmNoc29ydGVkKHdjLC45KnRvdGFsKV0pCiAgICAgICAgST1YPC00MAogICAgICAgIHE9ZGljdChkbT1kbSx0X015cj10LG1hc3M9dG90YWwsbWFzc19kcmlmdD10b3RhbC9pbml0LTEsbWluX3Jobz1mbG9hdChyaG8ubWluKCkpLG1pbl9wPWZsb2F0KFAubWluKCkpLGV4Y2Vzc190YWc9ZmxvYXQoKG5wLm1heGltdW0oVVs0XS1yaG8sMCkqQSkuc3VtKCkvdG90YWwpLGNlbnRyb2lkPWZsb2F0KChXKlgpLnN1bSgpL3RvdGFsKSxKPWZsb2F0KChXKih2eSsyKnMuT01FR0EqWCkpLnN1bSgpL3RvdGFsKSxtZWFuX1Q9ZmxvYXQoKFcqVCkuc3VtKCkvdG90YWwpLFQ1MD1xNTAsVDkwPXE5MCxmaW5lX2ZyYWN0aW9uPWZsb2F0KFdbKG5wLmRpZmYoeClbOixOb25lXTw9MS4wMDAwMSkmKG5wLmRpZmYoeSlbTm9uZSw6XTw9Mi41MDAwMSldLnN1bSgpL3RvdGFsKSxuZWFyNDA9ZmxvYXQoV1tSPDQwXS5zdW0oKS90b3RhbCksbmVhcl9ydD1mbG9hdChXW1I8cy5ydF0uc3VtKCkvdG90YWwpLEJuZWc0MD1mbG9hdChXWyhSPDQwKSYoQjwwKV0uc3VtKCkvdG90YWwpLGlubmVyX2ZhcjQwPWZsb2F0KFdbSSYoUj49NDApXS5zdW0oKS90b3RhbCksaW5uZXJfZmFyX3J0PWZsb2F0KFdbSSYoUj49cy5ydCldLnN1bSgpL3RvdGFsKSxzdGVwcz1pbnQoYS5zdGVwX2NvdW50KSkKICAgICAgICBmb3IgYyBpbiAoLTQwLDAsNDApOgogICAgICAgICAgICBrPSd4JytzdHIoYyk7IGludmVudG9yeT1mbG9hdChXW1g8Y10uc3VtKCkpOyBxW2YnaW52ZW50b3J5X3tjfSddPWludmVudG9yeS90b3RhbDsgcVtmJ2N1bV9pbl97Y30nXT1mbG9hdChhWydjdW1faW5fJytrXSk7IHFbZidpbmRlcGVuZGVudF9idWRnZXRfe2N9J109KGludmVudG9yeS1zdGFydFtjXSkvaW5pdC0oZmxvYXQoYVsnY3VtX2luXycra10pK2Zsb2F0KGFbJ2xlZnRfZmx1eF8nK2tdKS1mbG9hdChhWyd5X2xvc3NfJytrXSkpOyBhc3NlcnQgYWJzKHFbZidpbmRlcGVuZGVudF9idWRnZXRfe2N9J10pPDFlLTEwCiAgICAgICAgYXNzZXJ0IGFicyhxWydtZWFuX1QnXS1hLnRhZ19tZWFuX1RfSyk8MWUtOCBhbmQgYWJzKHFbJ2NlbnRyb2lkJ10tYS50YWdfY2VudGVyX3hfcGMpPDFlLTkKICAgICAgICByb3dzLmFwcGVuZChxKQogICAgcD1kLm1lcmdlKG9sZCxvbj0ndF9NeXInLHN1ZmZpeGVzPSgnX3dpZGUnLCdfb3JpZ2luYWwnKSkKICAgIGZvciBfLGEgaW4gcC5pdGVycm93cygpOgogICAgICAgIHJvdz1kaWN0KGRtPWRtLHRfTXlyPWZsb2F0KGEudF9NeXIpKQogICAgICAgIGZvciBrIGluICgnY3VtX2luX3gtNDAnLCdjdW1faW5feDAnLCdjdW1faW5feDQwJywndGFnX2NlbnRlcl94X3BjJywndGFnX21lYW5fVF9LJywnZGVsdGFfSicpOgogICAgICAgICAgICByb3dbaysnX3dpZGUnXT1mbG9hdChhW2srJ193aWRlJ10pO3Jvd1trKydfb3JpZ2luYWwnXT1mbG9hdChhW2srJ19vcmlnaW5hbCddKTtyb3dbaysnX2RpZmYnXT1mbG9hdChhW2srJ193aWRlJ10tYVtrKydfb3JpZ2luYWwnXSkKICAgICAgICBjb21wYXJpc29ucy5hcHBlbmQocm93KQpEPXBkLkRhdGFGcmFtZShyb3dzKS5zb3J0X3ZhbHVlcyhbJ2RtJywndF9NeXInXSk7IEM9cGQuRGF0YUZyYW1lKGNvbXBhcmlzb25zKTsgRC50b19jc3Yocm9vdC8nQ1AzN19GSUVMRFMuY3N2JyxpbmRleD1GYWxzZSk7IEMudG9fY3N2KHJvb3QvJ0NQMzdfT1JJR0lOQUxfQ09NUEFSSVNPTi5jc3YnLGluZGV4PUZhbHNlKQpwYWlyZWQ9W10KZm9yIHQsZyBpbiBELmdyb3VwYnkoJ3RfTXlyJyk6CiAgICBpZiBsZW4oZykhPTI6IGNvbnRpbnVlCiAgICBkPWcuc2V0X2luZGV4KCdkbScpO2E9ZC5sb2NbMV07Yz1kLmxvY1swXTtyb3c9eyd0X015cic6ZmxvYXQodCksKip7azpmbG9hdChhW2tdKSBmb3IgayBpbiAoJ2NlbnRyb2lkJywnSicsJ21lYW5fVCcsJ1Q1MCcsJ1Q5MCcsJ25lYXI0MCcsJ25lYXJfcnQnLCdCbmVnNDAnLCdpbm5lcl9mYXI0MCcsJ2lubmVyX2Zhcl9ydCcsJ2ZpbmVfZnJhY3Rpb24nLCdpbnZlbnRvcnlfLTQwJyl9fQogICAgZm9yIGN1dCBpbiAoLTQwLDAsNDApOiByb3dbZidleGNlc3Nfe2N1dH0nXT1mbG9hdChhW2YnY3VtX2luX3tjdXR9J10tY1tmJ2N1bV9pbl97Y3V0fSddKTsgcm93W2YnY29udHJvbF9pbnZlbnRvcnlfe2N1dH0nXT1mbG9hdChjW2YnaW52ZW50b3J5X3tjdXR9J10pCiAgICBwYWlyZWQuYXBwZW5kKHJvdykKUD1wZC5EYXRhRnJhbWUocGFpcmVkKTsgUC50b19jc3Yocm9vdC8nQ1AzN19QQUlSRUQuY3N2JyxpbmRleD1GYWxzZSk7IGxhc3Q9ZmxvYXQoUC50X015ci5tYXgoKSk7IGVuZD1QLmlsb2NbLTFdLnRvX2RpY3QoKTsgb2xkPUQuZ3JvdXBieSgnZG0nKS50X015ci5tYXgoKS50b19kaWN0KCkKc3VtbWFyeT17J2xhc3RfYnlfZG0nOntzdHIoayk6ZmxvYXQodikgZm9yIGssdiBpbiBvbGQuaXRlbXMoKX0sJ2NvbW1vbl90aW1lJzpsYXN0LCdmdWxsX3BhaXJfY29tcGxldGUnOmJvb2wobGFzdD09NTApLCdmaWVsZHMnOmxlbihEKSwnbWF4X2J1ZGdldCc6ZmxvYXQoRC5maWx0ZXIocmVnZXg9J15pbmRlcGVuZGVudF9idWRnZXRfJykuYWJzKCkubWF4KCkubWF4KCkpLCdtYXhfbWFzc19kcmlmdCc6ZmxvYXQoRC5tYXNzX2RyaWZ0LmFicygpLm1heCgpKSwnbWF4X2V4Y2Vzc190YWcnOmZsb2F0KEQuZXhjZXNzX3RhZy5tYXgoKSksJ21pbl9yaG8nOmZsb2F0KEQubWluX3Joby5taW4oKSksJ21pbl9wJzpmbG9hdChELm1pbl9wLm1pbigpKSwnY29tbW9uX2VuZHBvaW50JzplbmR9CmlmIGxhc3Q+PTIwOgogICAgYj1QW1AudF9NeXI9PTIwXS5pbG9jWzBdO3N1bW1hcnlbJ2xhdGVfZXhjZXNzX21pbnVzNDAnXT1mbG9hdChlbmRbJ2V4Y2Vzc18tNDAnXS1iWydleGNlc3NfLTQwJ10pCihyb290LydDUDM3X1JFU1VMVFMuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyhzdW1tYXJ5LGluZGVudD0yKSk7cHJpbnQoanNvbi5kdW1wcyhzdW1tYXJ5LGluZGVudD0yKSkKcGx0LnJjUGFyYW1zLnVwZGF0ZSh7J2ZvbnQuZmFtaWx5Jzonc2VyaWYnLCdmb250LnNlcmlmJzpbJ0xpYmVyYXRpb24gU2VyaWYnLCdUaW1lcyBOZXcgUm9tYW4nLCdEZWphVnUgU2VyaWYnXSwnbWF0aHRleHQuZm9udHNldCc6J3N0aXgnLCdmb250LnNpemUnOjEyLCdheGVzLmxhYmVsc2l6ZSc6MTIsJ2xlZ2VuZC5mb250c2l6ZSc6MTAsJ3BkZi5mb250dHlwZSc6NDIsJ2F4ZXMuc3BpbmVzLnRvcCc6RmFsc2UsJ2F4ZXMuc3BpbmVzLnJpZ2h0JzpGYWxzZX0pCmZpZyxheHM9cGx0LnN1YnBsb3RzKDIsMixmaWdzaXplPSg5LDYuNyksbGF5b3V0PSdjb25zdHJhaW5lZCcpOyBmaWcuZ2V0X2xheW91dF9lbmdpbmUoKS5zZXQocmVjdD0oMCwuMTEsMSwxKSk7IGNvbG9ycz1bJyMwMDcyQjInLCcjRDU1RTAwJywnIzAwOUU3MyddCk89cGQucmVhZF9jc3Yocm9vdC8nb3JpZ2luYWxfRE0xLmNzdicpOyBPYz1wZC5yZWFkX2Nzdihyb290LydvcmlnaW5hbF9ETTAuY3N2Jyk7IE89Ty5tZXJnZShPYyxvbj0ndF9NeXInLHN1ZmZpeGVzPSgnXzEnLCdfMCcpKTsgTz1PW08udF9NeXI8PWxhc3RdCkw9cGQucmVhZF9jc3Yocm9vdC8nQ1AzNV9QQUlSRUQuY3N2Jyk7TD1MWyhMLm1lc2g9PSdleHBhbmRlZCcpJihMLnRfTXlyPD1sYXN0KV0uY29weSgpCiMgRm9yIENQMzUgdXNlIGEgc2VwYXJhdGUgbGF0ZSBjcm9zc2luZyByZWxhdGl2ZSB0byB0aGUgc2FtZSB0PTIwOyBubyBjb25jYXRlbmF0aW9uIGludG8gaW5kZXBlbmRlbnQgaGlzdG9yaWVzLgpmb3IgYXgsayxrbyxtdWx0LGxhYiBpbiB6aXAoYXhzLmZsYXQsWydleGNlc3NfLTQwJywnY2VudHJvaWQnLCdtZWFuX1QnLCdmaW5lX2ZyYWN0aW9uJ10sW05vbmUsJ3RhZ19jZW50ZXJfeF9wY18xJywndGFnX21lYW5fVF9LXzEnLE5vbmVdLFsxMDAsMSwxLDEwMF0sWydFeGNlc3MgaW53YXJkIGNyb3NzaW5nICglKScsJ1JhZGlhbCBjZW50cm9pZCAocGMpJywnTWVhbiB0YWdnZWQgdGVtcGVyYXR1cmUgKEspJywnVGFnIG9uIGZpbmUgY2VsbHMgKCUpJ10pOgogICAgc2V0cz1bKFAudF9NeXIudG9fbnVtcHkoKSxQW2tdLnRvX251bXB5KCkqbXVsdCwnSW5kZXBlbmRlbnQgd2lkZSBzdHJpcCcsY29sb3JzWzBdKV0KICAgIGlmIGs9PSdleGNlc3NfLTQwJzogc2V0cy5hcHBlbmQoKE8udF9NeXIudG9fbnVtcHkoKSwoT1snY3VtX2luX3gtNDBfMSddLU9bJ2N1bV9pbl94LTQwXzAnXSkudG9fbnVtcHkoKSoxMDAsJ09yaWdpbmFsIHN0cmlwJyxjb2xvcnNbMV0pKQogICAgZWxpZiBrbzogc2V0cy5hcHBlbmQoKE8udF9NeXIudG9fbnVtcHkoKSxPW2tvXS50b19udW1weSgpKm11bHQsJ09yaWdpbmFsIHN0cmlwJyxjb2xvcnNbMV0pKQogICAgaWYgbGVuKEwpPjEgYW5kIGsgaW4gKCdjZW50cm9pZCcsJ21lYW5fVCcsJ2ZpbmVfZnJhY3Rpb24nKToKICAgICAgICBxPXsnY2VudHJvaWQnOid4X2NlbnRyb2lkJywnbWVhbl9UJzonbWVhbl9UJywnZmluZV9mcmFjdGlvbic6J2ZpbmVfZnJhY3Rpb24nfVtrXTtzZXRzLmFwcGVuZCgoTC50X015ci50b19udW1weSgpLExbcV0udG9fbnVtcHkoKSptdWx0LCdMYXRlIHJlbWFwcGVkIHN0cmlwJyxjb2xvcnNbMl0pKQogICAgZm9yIHQsdixsLGNvbCBpbiBzZXRzOgogICAgICAgIGlmIGxlbih0KT4xOgogICAgICAgICAgICB0dD1ucC5saW5zcGFjZSh0Lm1pbigpLHQubWF4KCksMjAwMSk7dnY9UGNoaXBJbnRlcnBvbGF0b3IodCx2KSh0dCk7YXNzZXJ0IHZ2Lm1pbigpPj12Lm1pbigpLTFlLTggYW5kIHZ2Lm1heCgpPD12Lm1heCgpKzFlLTg7YXgucGxvdCh0dCx2dixsdz0xLjgsYz1jb2wsbGFiZWw9bCkKICAgICAgICBheC5zY2F0dGVyKHQsdixzPTEwLGM9Y29sLHpvcmRlcj0zKQogICAgYXguc2V0KHhsYWJlbD0nVGltZSAoTXlyKScseWxhYmVsPWxhYik7YXguZ3JpZChhbHBoYT0uMTQpCmhhbmRsZXM9e30KZm9yIGF4IGluIGF4cy5mbGF0OgogICAgZm9yIGgsbCBpbiB6aXAoKmF4LmdldF9sZWdlbmRfaGFuZGxlc19sYWJlbHMoKSk6aGFuZGxlc1tsXT1oCmZpZy5sZWdlbmQoaGFuZGxlcy52YWx1ZXMoKSxoYW5kbGVzLmtleXMoKSxsb2M9J2xvd2VyIGNlbnRlcicsYmJveF90b19hbmNob3I9KC41LC4wMTUpLG5jb2w9MyxmcmFtZW9uPVRydWUpO2ZpZy5zYXZlZmlnKHJvb3QvJ0NQMzdfZnVsbF9oaXN0b3J5X2NvbXBhcmlzb24ucGRmJyk7cGx0LmNsb3NlKGZpZykK', 'CP38_KINEMATICS.py': 'IiIiQ2VsbC1hcmVhLXdlaWdodGVkIGluc3RhbnRhbmVvdXMga2luZW1hdGljczsgbm8gcHJlZGljdGlvbiBvZiBmdXR1cmUgdHJhamVjdG9yaWVzLiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wLCBjc3YsIGpzb24KUk9PVD1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CiMgVXNlIHRoZSBzYW1lIGFuYWx5dGljIGhvc3QgcGFyYW1ldGVycyBhbmQgcGh5c2ljYWwgdW5pdHMgYXMgcHJvZHVjdGlvbi4KaW1wb3J0IENQMjlfU09VUkNFIGFzIHMKcm93cz1bXQpmb3IgZG0gaW4gKDAsMSk6CiBmb3IgcCBpbiBzb3J0ZWQoKFJPT1QvZidmdWxsX0RNe2RtfScpLmdsb2IoJ2ZyYW1lXyoubnB6JykpOgogIHdpdGggbnAubG9hZChwKSBhcyB6OgogICBVPXpbJ1UnXTsgeD16Wyd4J107IHk9elsneSddOyB0PWZsb2F0KHpbJ3QnXSkKICBYPSh4WzotMV0reFsxOl0pWzosTm9uZV0vMjsgQT1ucC5kaWZmKHgpWzosTm9uZV0qbnAuZGlmZih5KVtOb25lLDpdOyBXPVVbNF0qQTsgTT1XLnN1bSgpOyB2eD1VWzFdL1VbMF07IHZ5PVVbMl0vVVswXTsgaj12eSsyKnMuT01FR0EqWDsgeGc9ai8oKDItcy5RKSpzLk9NRUdBKQogIGF2Zz1sYW1iZGEgdjogZmxvYXQobnAuc3VtKFcqdikvTSkKICByb3c9ZGljdChkbT1kbSx0X015cj10LHhfcGM9YXZnKFgpLHZ4X3BjX015cj1hdmcodngpLGpfcGNfTXlyPWF2ZyhqKSx4Z19wYz1hdmcoeGcpLHhfbWludXNfeGdfcGM9YXZnKFgteGcpLGlubmVyX2ZyYWN0aW9uPWF2ZyhYPC00MCksaW5uZXJfcG9zaXRpdmVfeGc9YXZnKChYPC00MCkmKHhnPjApKSxpbm5lcl9pbndhcmRfdmVsb2NpdHk9YXZnKChYPC00MCkmKHZ4PDApKSxmcmVlX2VwaWN5Y2xpY19hY2NlbGVyYXRpb25fcGNfTXlyMj1hdmcoLTIqKDItcy5RKSpzLk9NRUdBKioyKihYLXhnKSkpCiAgYXNzZXJ0IGFicyhyb3dbJ3hnX3BjJ10tcm93WydqX3BjX015ciddLygoMi1zLlEpKnMuT01FR0EpKTwxZS0xMAogIHJvd3MuYXBwZW5kKHJvdykKcm93cy5zb3J0KGtleT1sYW1iZGEgcjooclsnZG0nXSxyWyd0X015ciddKSkKd2l0aCAoUk9PVC8nQ1AzOF9LSU5FTUFUSUNTLmNzdicpLm9wZW4oJ3cnLG5ld2xpbmU9JycpIGFzIGY6CiB3PWNzdi5EaWN0V3JpdGVyKGYsZmllbGRuYW1lcz1saXN0KHJvd3NbMF0pKTt3LndyaXRlaGVhZGVyKCk7dy53cml0ZXJvd3Mocm93cykKIyBDb21wYXJlIHRoZSBzZXBhcmF0ZWx5IGFyY2hpdmVkIGRpYWdub3N0aWNzIHdoZW4gdGhlIGZ1bGwgb3V0cHV0IGlzIHByZXNlbnQuCndpdGggKFJPT1QvJ0NQMzdfRklFTERTLmNzdicpLm9wZW4oKSBhcyBmOiBvbGQ9eyhpbnQoclsnZG0nXSksZmxvYXQoclsndF9NeXInXSkpOnIgZm9yIHIgaW4gY3N2LkRpY3RSZWFkZXIoZil9CmZvciByIGluIHJvd3M6CiBhPW9sZC5nZXQoKHJbJ2RtJ10sclsndF9NeXInXSkpCiBpZiBhOgogIGFzc2VydCBhYnMoclsnal9wY19NeXInXS1mbG9hdChhWydKJ10pKTwxZS0xMAogIGFzc2VydCBhYnMoclsneF9wYyddLWZsb2F0KGFbJ2NlbnRyb2lkJ10pKTwxZS0xMApwcmludChqc29uLmR1bXBzKFtyIGZvciByIGluIHJvd3MgaWYgclsndF9NeXInXSBpbiAoMjAsMzAsNDAsNTApXSxpbmRlbnQ9MikpCg==', 'CP38_PLOT.py': 'ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucCxwYW5kYXMgYXMgcGQsbWF0cGxvdGxpYgptYXRwbG90bGliLnVzZSgnQWdnJykKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApmcm9tIHNjaXB5LmludGVycG9sYXRlIGltcG9ydCBQY2hpcEludGVycG9sYXRvcgpyb290PVBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKUD1wZC5yZWFkX2Nzdihyb290LydDUDM3X1BBSVJFRC5jc3YnKTsgbGFzdD1QLnRfTXlyLm1heCgpCk89cGQucmVhZF9jc3Yocm9vdC8nb3JpZ2luYWxfRE0xLmNzdicpLm1lcmdlKHBkLnJlYWRfY3N2KHJvb3QvJ29yaWdpbmFsX0RNMC5jc3YnKSxvbj0ndF9NeXInLHN1ZmZpeGVzPSgnXzEnLCdfMCcpKTtPPU9bTy50X015cjw9bGFzdF0KTD1wZC5yZWFkX2Nzdihyb290LydDUDM1X1BBSVJFRC5jc3YnKTtMPUxbTC50X015cjw9bGFzdF07THc9TFtMLm1lc2g9PSdleHBhbmRlZCddO0xvPUxbTC5tZXNoPT0nb3JpZ2luYWwnXQpwbHQucmNQYXJhbXMudXBkYXRlKHsnZm9udC5mYW1pbHknOidzZXJpZicsJ2ZvbnQuc2VyaWYnOlsnTGliZXJhdGlvbiBTZXJpZicsJ1RpbWVzIE5ldyBSb21hbicsJ0RlamFWdSBTZXJpZiddLCdtYXRodGV4dC5mb250c2V0Jzonc3RpeCcsJ2ZvbnQuc2l6ZSc6MTIsJ2F4ZXMubGFiZWxzaXplJzoxMiwnbGVnZW5kLmZvbnRzaXplJzoxMCwncGRmLmZvbnR0eXBlJzo0MiwnYXhlcy5zcGluZXMudG9wJzpGYWxzZSwnYXhlcy5zcGluZXMucmlnaHQnOkZhbHNlfSkKZmlnLGF4cz1wbHQuc3VicGxvdHMoMiwyLGZpZ3NpemU9KDksNi45KSk7ZmlnLnN1YnBsb3RzX2FkanVzdChsZWZ0PS4xMixyaWdodD0uOTgsYm90dG9tPS4xNix0b3A9Ljk2LGhzcGFjZT0uMzcsd3NwYWNlPS4zOSkKY29sb3JzPVsnIzAwNzJCMicsJyNENTVFMDAnLCcjMDA5RTczJ107IGxhYmVscz1bJ0luZGVwZW5kZW50IHdpZGUgc3RyaXAnLCdPcmlnaW5hbCBzdHJpcCcsJ0xhdGUgcmVtYXBwZWQgc3RyaXAnXTsgc3R5bGVzPVsnLScsJy0tJywnLS4nXTtoYW5kbGVzPXt9CmZvciBheCxrLGtvLG11bHQsbGFiIGluIHppcChheHMuZmxhdCxbJ2V4Y2Vzc18tNDAnLCdjZW50cm9pZCcsJ21lYW5fVCcsJ2ZpbmVfZnJhY3Rpb24nXSxbTm9uZSwndGFnX2NlbnRlcl94X3BjXzEnLCd0YWdfbWVhbl9UX0tfMScsTm9uZV0sWzEwMCwxLDEsMTAwXSxbJ0V4Y2VzcyBpbndhcmQgY3Jvc3NpbmcgKCUpJywnUmFkaWFsIGNlbnRyb2lkIChwYyknLCdNZWFuIHRhZ2dlZCB0ZW1wZXJhdHVyZSAoSyknLCdUYWcgb24gZmluZSBjZWxscyAoJSknXSk6CiBzZXRzPVsoUC50X015cixQW2tdKm11bHQsMCldCiBpZiBrPT0nZXhjZXNzXy00MCc6IHNldHMuYXBwZW5kKChPLnRfTXlyLChPWydjdW1faW5feC00MF8xJ10tT1snY3VtX2luX3gtNDBfMCddKSoxMDAsMSkpCiBlbGlmIGtvOnNldHMuYXBwZW5kKChPLnRfTXlyLE9ba29dKm11bHQsMSkpCiBlbGlmIGs9PSdmaW5lX2ZyYWN0aW9uJyBhbmQgbGVuKExvKTpzZXRzLmFwcGVuZCgoTG8udF9NeXIsTG8uZmluZV9mcmFjdGlvbioxMDAsMSkpCiBpZiBrIT0nZXhjZXNzXy00MCcgYW5kIGxlbihMdyk6c2V0cy5hcHBlbmQoKEx3LnRfTXlyLEx3W3snY2VudHJvaWQnOid4X2NlbnRyb2lkJywnbWVhbl9UJzonbWVhbl9UJywnZmluZV9mcmFjdGlvbic6J2ZpbmVfZnJhY3Rpb24nfVtrXV0qbXVsdCwyKSkKIGFsbHY9W10KIGZvciB0LHYsaSBpbiBzZXRzOgogIHQ9bnAuYXNhcnJheSh0KTt2PW5wLmFzYXJyYXkodik7YXNzZXJ0IG5wLmlzZmluaXRlKHYpLmFsbCgpO2FsbHYuZXh0ZW5kKHYudG9saXN0KCkpCiAgaWYgbGVuKHQpPjE6CiAgIHR0PW5wLmxpbnNwYWNlKHQubWluKCksdC5tYXgoKSwyNTAxKTt2dj1QY2hpcEludGVycG9sYXRvcih0LHYpKHR0KTthc3NlcnQgdnYubWluKCk+PXYubWluKCktMWUtOCBhbmQgdnYubWF4KCk8PXYubWF4KCkrMWUtOAogICBsaW5lLD1heC5wbG90KHR0LHZ2LGx3PTEuOCxjPWNvbG9yc1tpXSxscz1zdHlsZXNbaV0sbGFiZWw9bGFiZWxzW2ldKTtoYW5kbGVzW2xhYmVsc1tpXV09bGluZQogIGF4LnNjYXR0ZXIodCx2LHM9MTEsYz1jb2xvcnNbaV0sem9yZGVyPTMpCiBsbyxoaT1taW4oYWxsdiksbWF4KGFsbHYpOyBtYXJnaW49bWF4KChoaS1sbykqLjA1NSwxZS01KTtheC5zZXRfeWxpbShsby1tYXJnaW4saGkrbWFyZ2luKTtheC5zZXRfeGxpbSgtNTMsbWF4KC00OSxsYXN0KSszKTthc3NlcnQgYWxsKGF4LmdldF95bGltKClbMF08PXY8PWF4LmdldF95bGltKClbMV0gZm9yIHYgaW4gYWxsdik7YXguc2V0KHhsYWJlbD0nVGltZSAoTXlyKScseWxhYmVsPWxhYik7YXguZ3JpZChhbHBoYT0uMTQpCmZpZy5sZWdlbmQoW2hhbmRsZXNba10gZm9yIGsgaW4gbGFiZWxzIGlmIGsgaW4gaGFuZGxlc10sW2sgZm9yIGsgaW4gbGFiZWxzIGlmIGsgaW4gaGFuZGxlc10sbG9jPSdsb3dlciBjZW50ZXInLGJib3hfdG9fYW5jaG9yPSguNSwuMDI1KSxuY29sPTMsZnJhbWVvbj1UcnVlKTtmaWcuc2F2ZWZpZyhyb290LydDUDM4X2Z1bGxfaGlzdG9yeV9jb21wYXJpc29uLnBkZicpO3BsdC5jbG9zZShmaWcpCg==', 'CP39_JOINT_CHECK.py': 'IiIiUG9zdC1wcm9jZXNzIHNhdmVkIGZ1bGwgZW5jb3VudGVyIHdpdGhvdXQgYWR2YW5jaW5nIHRoZSBoeWRybyBzb2x2ZXIuCkRpcmVjdCB0YWdnZWQgbWFzcyBpbnRlcnNlY3Rpb25zLCBmaXhlZCBpbml0aWFsIG1hc3MgZGVub21pbmF0b3IsIGluZGVwZW5kZW50IGFuZ3VsYXIgbW9tZW50dW0gY2hlY2suCiIiIgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IGNzdiwgbWF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IENQMjlfU09VUkNFIGFzIHMKUk9PVD1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CiMgUmVnZW5lcmF0ZSBwcmVzY3JpYmVkIHNsb3cgdHJhamVjdG9yeSBmcm9tIHBhc3NhZ2Ugc3RhdGUsIGluZGVwZW5kZW50IG9mIHJlbWFwcGluZy4Kcy5TVEFSVD0tNTAuOyBzLkVORD01MC47IHMuVlIqPS41CnRpbWVzLHRyYWNrLF89cy5vcmJpdF9mdWxsX3JhbmdlKCkKa2FwcGEyPTIqKDItcy5RKSpzLk9NRUdBKioyCnJvd3M9W10KZm9yIGRtIGluICgwLDEpOgogcGF0aHM9c29ydGVkKChST09UL2YnZnVsbF9ETXtkbX0nKS5nbG9iKCdmcmFtZV8qLm5weicpKQogaWYgbm90IHBhdGhzOiByYWlzZSBSdW50aW1lRXJyb3IoJ01pc3Npbmcgc25hcHNob3RzJykKIGZzPVtdCiBmb3IgcCBpbiBwYXRoczoKICB3aXRoIG5wLmxvYWQocCkgYXMgejogZnMuYXBwZW5kKChmbG9hdCh6Wyd0J10pLHApKQogZnMuc29ydCgpCiBmb3IgdCxwIGluIGZzOgogIHdpdGggbnAubG9hZChwKSBhcyB6OiBVPXpbJ1UnXTsgeD16Wyd4J107IHk9elsneSddCiAgWD0oeFs6LTFdK3hbMTpdKVs6LE5vbmVdLzI7IFk9KHlbOi0xXSt5WzE6XSlbTm9uZSw6XS8yCiAgVz1VWzRdKm5wLmRpZmYoeClbOixOb25lXSpucC5kaWZmKHkpW05vbmUsOl07IG1hc3M9Vy5zdW0oKQogIGlmIHQ9PWZzWzBdWzBdOiBtMD1tYXNzCiAgYXNzZXJ0IGFicyhtYXNzL20wLTEpPDFlLTEyIGFuZCBucC5pc2Zpbml0ZShVKS5hbGwoKQogIHZ4PVVbMV0vVVswXTsgdnk9VVsyXS9VWzBdOyBqPXZ5KzIqcy5PTUVHQSpYCiAgbXgsbXk9W25wLmludGVycCh0LHRpbWVzLHRyYWNrWzosaV0pIGZvciBpIGluICgwLDEpXQogIFI9bnAuaHlwb3QoWC1teCxZLW15KTsgST1ucC5icm9hZGNhc3RfdG8oWDwtNDAsVy5zaGFwZSk7IEo9aj4wCiAgZj1sYW1iZGEgbWFzazogZmxvYXQoV1ttYXNrXS5zdW0oKS9tMCkKICBtZWFuPWxhbWJkYSBmaWVsZDogZmxvYXQobnAuc3VtKFcqZmllbGQpL21hc3MpCiAgYSxiLGM9ZihJKSxmKEkmKFI+PTQwKSksZihJJkopCiAgdHJpcGxlPWYoSSYoUj49NDApJkopOyB0cmlwbGVfcnQ9ZihJJihSPj1zLnJ0KSZKKQogIGFzc2VydCBtYXgoMCxiK2MtYSktMWUtMTI8PXRyaXBsZTw9bWluKGIsYykrMWUtMTIKICBhc3NlcnQgYWJzKG1lYW4oLWthcHBhMiooWC1qLygoMi1zLlEpKnMuT01FR0EpKSktbWVhbigyKnMuT01FR0EqdnkrMipzLlEqcy5PTUVHQSoqMipYKSk8MWUtMTIKICByb3dzLmFwcGVuZChkaWN0KGRtPWRtLHRfTXlyPXQsaW5uZXI9YSxpbm5lcl9mYXI0MD1iLGlubmVyX3Bvc2l0aXZlX2o9Yyxpbm5lcl9mYXI0MF9wb3NpdGl2ZV9qPXRyaXBsZSxpbm5lcl9mYXJfcnRfcG9zaXRpdmVfaj10cmlwbGVfcnQsbWVhbl9mcmVlX3JhZGlhbF9hY2NlbD1tZWFuKDIqcy5PTUVHQSp2eSsyKnMuUSpzLk9NRUdBKioyKlgpLG1lYW5feD1tZWFuKFgpLG1lYW5faj1tZWFuKGopLG1lYW5fdng9bWVhbih2eCkscGVyaW9kX015cj0yKm1hdGgucGkvbWF0aC5zcXJ0KGthcHBhMikpKQp3aXRoIChST09ULydDUDM5X0pPSU5UX0NIRUNLLmNzdicpLm9wZW4oJ3cnLG5ld2xpbmU9JycpIGFzIGY6CiB3PWNzdi5EaWN0V3JpdGVyKGYsZmllbGRuYW1lcz1saXN0KHJvd3NbMF0pKTt3LndyaXRlaGVhZGVyKCk7dy53cml0ZXJvd3Mocm93cykKaWYgbGVuKHJvd3MpPT00MjoKIHdpdGggKFJPT1QvJ0NQMzhfS0lORU1BVElDUy5jc3YnKS5vcGVuKCkgYXMgZjogb2xkPXsoaW50KHJbJ2RtJ10pLGZsb2F0KHJbJ3RfTXlyJ10pKTpyIGZvciByIGluIGNzdi5EaWN0UmVhZGVyKGYpfQogd2l0aCAoUk9PVC8nQ1AzN19GSUVMRFMuY3N2Jykub3BlbigpIGFzIGY6IG9sZDI9eyhpbnQoclsnZG0nXSksZmxvYXQoclsndF9NeXInXSkpOnIgZm9yIHIgaW4gY3N2LkRpY3RSZWFkZXIoZil9CiBmb3IgciBpbiByb3dzOgogIGtleT0oclsnZG0nXSxyWyd0X015ciddKTsgYXNzZXJ0IGFicyhyWydpbm5lcl9wb3NpdGl2ZV9qJ10tZmxvYXQob2xkW2tleV1bJ2lubmVyX3Bvc2l0aXZlX3hnJ10pKTwxZS0xMTsgYXNzZXJ0IGFicyhyWydpbm5lcl9mYXI0MCddLWZsb2F0KG9sZDJba2V5XVsnaW5uZXJfZmFyNDAnXSkpPDFlLTExCmZvciByIGluIHJvd3M6CiBpZiByWyd0X015ciddPT01MDogcHJpbnQocikK', 'CP40_VERIFY.py': 'IiIiUmVjaGVjayBzYXZlZCBDUDM3IGZpZWxkcyB1c2luZyByZWNvcmRlZCB0cmFqZWN0b3J5IGFuZCBpbmRlcGVuZGVudCBtYXNrZWQgcXVhZHJhdHVyZTsgbm8gaHlkcm8uIiIiCmltcG9ydCBjc3YsIG1hdGgsIGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucApyb290PVBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKb2xkPXsoaW50KHJbJ2RtJ10pLGZsb2F0KHJbJ3RfTXlyJ10pKTpyIGZvciByIGluIGNzdi5EaWN0UmVhZGVyKChyb290LydDUDM5X0pPSU5UX0NIRUNLLmNzdicpLm9wZW4oKSl9CiMgVXNlIHJlY29yZGVkIHRyYWplY3RvcnkgYW5kIGluZmVyIHRoZSBleGFjdCBPbWVnYSBmcm9tIGFyY2hpdmVkIHBlcmlvZCBhbmQgcSAocm91bmRlZCBxIGJlbG93IGlzIG5vdCB1c2VkKS4KaW1wb3J0IENQMjlfU09VUkNFIGFzIHMKb21lZ2E9cy5PTUVHQQpyZXN1bHRzPVtdOyBtYXhkaWZmPTA7IG1heGRyaWZ0PTAKZm9yIGRtIGluICgwLDEpOgogZm9sZGVyPXJvb3QvZidmdWxsX0RNe2RtfScKIGhpc3Q9e2Zsb2F0KHJbJ3RfTXlyJ10pOnIgZm9yIHIgaW4gY3N2LkRpY3RSZWFkZXIobmV4dChmb2xkZXIuZ2xvYignaDFfcGhhc2UqLmNzdicpKS5vcGVuKCkpfQogZnJhbWVzPVtdCiBmb3IgZiBpbiBmb2xkZXIuZ2xvYignZnJhbWVfKi5ucHonKToKICB3aXRoIG5wLmxvYWQoZikgYXMgejogZnJhbWVzLmFwcGVuZCgoZmxvYXQoelsndCddKSxmKSkKIGZvciB0LGYgaW4gc29ydGVkKGZyYW1lcyk6CiAgd2l0aCBucC5sb2FkKGYpIGFzIHo6IFU9elsnVSddOyB4PXpbJ3gnXTsgeT16Wyd5J10KICByaG8sbXgsbXksRSx0YWc9VQogIGFyZWE9bnAub3V0ZXIobnAuZGlmZih4KSxucC5kaWZmKHkpKTsgd2VpZ2h0cz1hcmVhKnRhZwogIG1hc3M9bWF0aC5mc3VtKHdlaWdodHMucmF2ZWwoKSk7CiAgaWYgdD09LTUwOiBtMD1tYXNzCiAgbWF4ZHJpZnQ9bWF4KG1heGRyaWZ0LGFicyhtYXNzL20wLTEpKQogIFA9KDUvMy0xKSooRS0obXgqbXgrbXkqbXkpLygyKnJobykpCiAgYXNzZXJ0IG5wLmlzZmluaXRlKFUpLmFsbCgpIGFuZCByaG8ubWluKCk+MCBhbmQgUC5taW4oKT4wIGFuZCB0YWcubWluKCk+PS0xZS0xMiBhbmQgKHRhZy1yaG8pLm1heCgpPD0xZS0xMgogIFgsWT1ucC5tZXNoZ3JpZCgoeFs6LTFdK3hbMTpdKS8yLCh5WzotMV0reVsxOl0pLzIsaW5kZXhpbmc9J2lqJykKICAjIERldGVybWluZSBzaWduIHdpdGhvdXQgZGl2aWRpbmcgYnkgcmhvOyB1c2Ugc3F1YXJlZCBkaXN0YW5jZSByYXRoZXIgdGhhbiBoeXBvdC4KICBqcG9zaXRpdmU9bXkrMipvbWVnYSpYKnJobz4wCiAgaD1oaXN0W3RdOyBSMj0oWC1mbG9hdChoWydkbV94X3BjJ10pKSoqMisoWS1mbG9hdChoWydkbV95X3BjJ10pKSoqMgogIEk9WDwtNDAKICBmcmFjPWxhbWJkYSBtYXNrOiBtYXRoLmZzdW0od2VpZ2h0c1ttYXNrXS5yYXZlbCgpKS9tMAogIHZhbHM9eydpbm5lcic6ZnJhYyhJKSwnaW5uZXJfZmFyNDAnOmZyYWMoSSYoUjI+PTE2MDApKSwgJ2lubmVyX3Bvc2l0aXZlX2onOmZyYWMoSSZqcG9zaXRpdmUpLCdpbm5lcl9mYXI0MF9wb3NpdGl2ZV9qJzpmcmFjKEkmanBvc2l0aXZlJihSMj49MTYwMCkpLCAnaW5uZXJfZmFyX3J0X3Bvc2l0aXZlX2onOmZyYWMoSSZqcG9zaXRpdmUmKFIyPj1zLnJ0KioyKSl9CiAgZm9yIGssdiBpbiB2YWxzLml0ZW1zKCk6IG1heGRpZmY9bWF4KG1heGRpZmYsYWJzKHYtZmxvYXQob2xkWyhkbSx0KV1ba10pKSkKICByZXN1bHRzLmFwcGVuZChkaWN0KGRtPWRtLHQ9dCwqKnZhbHMpKQphc3NlcnQgbGVuKHJlc3VsdHMpPT00MiBhbmQgbWF4ZGlmZjwxZS0xMSBhbmQgbWF4ZHJpZnQ8MWUtMTIKb3V0PXsnZnJhbWVzJzpsZW4ocmVzdWx0cyksJ21heF9mcmFjdGlvbl9kaWZmZXJlbmNlJzptYXhkaWZmLCdtYXhfbWFzc19kcmlmdCc6bWF4ZHJpZnQsJ2FsbF9wb3NpdGl2ZV9kZW5zaXR5X3ByZXNzdXJlX2FuZF9ib3VuZGVkX3RhZyc6VHJ1ZSwnZW5kcG9pbnQnOltyIGZvciByIGluIHJlc3VsdHMgaWYgclsndCddPT01MF19Cihyb290LydDUDQwX1ZFUklGWV9SRVNVTFRTLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMob3V0LGluZGVudD0yKSsnXG4nKTtwcmludChqc29uLmR1bXBzKG91dCxpbmRlbnQ9MikpCg==', 'CP46_DIAGNOSTICS.py': 'IiIiRmluaXRlLXdpbmRvdyBncmF2aXR5LW9ubHkgY29tcGFyaXNvbiBhbmQgc2F2ZWQtZmllbGQgZGlhZ25vc3RpY3MuIE5vIG5ldyBoeWRybyBldm9sdXRpb24uIiIiCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAsIHBhbmRhcyBhcyBwZCwganNvbgppbXBvcnQgQ1AyOV9TT1VSQ0UgYXMgcwpyb290PVBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKcy5TVEFSVD0tNTAuO3MuRU5EPTUwLjtzLlZSKj0uNQp0dCx0cmFjayxfPXMub3JiaXRfZnVsbF9yYW5nZSgpOyBrMj0yKigyLXMuUSkqcy5PTUVHQSoqMgpyaj0ocy5HMipzLm1ncmlkWy0xXS8oMipzLlEqcy5PTUVHQSoqMikpKiooMS8zKQpkZWYgZm9yY2UodCxYLFkpOgogbXgsbXk9W25wLmludGVycCh0LHR0LHRyYWNrWzosaV0pIGZvciBpIGluICgwLDEpXTsgZHg9WC1teDtkeT1ZLW15O3JyPW5wLnNxcnQoZHgqZHgrZHkqZHkrLjI1KTtmYWM9LXMuRzIqbnAuaW50ZXJwKHJyLHMucmdyaWQscy5tZ3JpZCxsZWZ0PTAscmlnaHQ9cy5tZ3JpZFstMV0pL3JyKiozCiByZXR1cm4gZmFjKmR4LGZhYypkeSxucC5oeXBvdChkeCxkeSkKZGVmIHF1YW50KHYsdyxxcz0oLjEsLjUsLjkpKToKIGE9bnAuYXJnc29ydCh2LnJhdmVsKCkpO3JldHVybiBucC5pbnRlcnAocXMsbnAuY3Vtc3VtKHcucmF2ZWwoKVthXSkvdy5zdW0oKSx2LnJhdmVsKClbYV0pCnJvd3M9W10KZm9yIGRtIGluICgwLDEpOgogZm9yIHAgaW4gc29ydGVkKChyb290L2YnZnVsbF9ETXtkbX0nKS5nbG9iKCdmcmFtZV8qLm5weicpKToKICB3aXRoIG5wLmxvYWQocCkgYXMgejogVT16WydVJ107eD16Wyd4J107eT16Wyd5J107dD1mbG9hdCh6Wyd0J10pCiAgeGM9KHhbMTpdK3hbOi0xXSkvMjt5Yz0oeVsxOl0reVs6LTFdKS8yO1gsWT1ucC5tZXNoZ3JpZCh4Yyx5YyxpbmRleGluZz0naWonKTtyaG89VVswXTt2eD1VWzFdL3Jobzt2eT1VWzJdL3JobztQPShzLkdBTU1BLTEpKihVWzNdLS41KnJobyoodngqdngrdnkqdnkpKTsgVz1VWzRdKm5wLmRpZmYoeClbOixOb25lXSpucC5kaWZmKHkpW05vbmUsOl07Vy89Vy5zdW0oKTt4Zz0odnkrMipzLk9NRUdBKlgpLygoMi1zLlEpKnMuT01FR0EpO2F4LGF5LFI9Zm9yY2UodCxYLFkpO2ZyZWU9LWsyKihYLXhnKTtwcmVzc3VyZT0tbnAuZ3JhZGllbnQoUCx4YyxheGlzPTAsZWRnZV9vcmRlcj0yKS9yaG87IHZyZWw9bnAuaHlwb3QodngtbnAuaW50ZXJwKHQsdHQsbnAuZ3JhZGllbnQodHJhY2ssdHQsYXhpcz0wKVs6LDBdKSx2eS1ucC5pbnRlcnAodCx0dCxucC5ncmFkaWVudCh0cmFjayx0dCxheGlzPTApWzosMV0pKTsgbXgsbXk9W25wLmludGVycCh0LHR0LHRyYWNrWzosaV0pIGZvciBpIGluICgwLDEpXTt1eCx1eT1bbnAuaW50ZXJwKHQsdHQsbnAuZ3JhZGllbnQodHJhY2ssdHQsYXhpcz0wKVs6LGldKSBmb3IgaSBpbiAoMCwxKV07dnJlbF9pbmVydGlhbD1ucC5oeXBvdCh2eC11eC1zLk9NRUdBKihZLW15KSx2eS11eStzLk9NRUdBKihYLW14KSk7IFQ9cy5UdypQLyhzLlAwX1JBRCpyaG8pCiAgYXZnPWxhbWJkYSBhOmZsb2F0KG5wLnN1bShXKmEpKTsgZj1sYW1iZGEgYTpmbG9hdChXW2FdLnN1bSgpKTtkPWRpY3QoZG09ZG0sdD10LHg9YXZnKFgpLHhnPWF2Zyh4Zyksdng9YXZnKHZ4KSxmcmVlPWF2ZyhmcmVlKSxzdWJoYWxvPWRtKmF2ZyhheCkscHJlc3N1cmU9YXZnKHByZXNzdXJlKSxUbWVhbj1hdmcoVCkpO2RbJ3RvdGFsJ109ZFsnZnJlZSddK2RbJ3N1YmhhbG8nXStkWydwcmVzc3VyZSddCiAgZm9yIGEgaW4gKDIwLDQwLDgwLDEyMCwxNTApOiBkW2YnaW5fe2F9J109ZihYPC1hKQogIGZvciBhLG4gaW4gWyg0MCwnNDAnKSwoODAsJzgwJyksKDEyMCwnMTIwJyksKHMucnQsJ3J0JyksKHJqLCdyaicpXTpkWyduZWFyXycrbl09ZihSPGEpO2RbJ2lubmVyX2Zhcl8nK25dPWYoKFg8LTQwKSYoUj49YSkpO2RbJ2pvaW50Xycrbl09ZigoWDwtNDApJihSPj1hKSYoeGc+MCkpO2RbJ2pvaW50NDBfJytuXT1mKChYPC00MCkmKFI+PWEpJih4Zz40MCkpCiAgZm9yIG5hbWUsZmllbGQgaW4gWygneCcsWCksKCd4ZycseGcpLCgndnJlbF9yb3QnLHZyZWwpLCgndnJlbF9pbmVydGlhbCcsdnJlbF9pbmVydGlhbCksKCdUJyxUKSwoJ2NvbmNlbnRyYXRpb24nLFVbNF0vcmhvKV06CiAgIGZvciBxLHYgaW4gemlwKCgxMCw1MCw5MCkscXVhbnQoZmllbGQsVykpOmRbZid7bmFtZX1fcXtxfSddPXYKICBtYXNrPVVbNF0vcmhvPi41O2RbJ2hpZ2hfY19tYXNzJ109ZihtYXNrKTtkWydoaWdoX2NfVG1lYW4nXT1mbG9hdChucC5zdW0oV1ttYXNrXSpUW21hc2tdKS9XW21hc2tdLnN1bSgpKSBpZiBtYXNrLmFueSgpIGVsc2UgZmxvYXQoJ25hbicpO3Jvd3MuYXBwZW5kKGQpCkQ9cGQuRGF0YUZyYW1lKHJvd3MpLnNvcnRfdmFsdWVzKFsnZG0nLCd0J10pOyBELnRvX2Nzdihyb290LydDUDQ2X0ZJRUxEUy5jc3YnLGluZGV4PUZhbHNlKQojIFdlaWdodGVkIHBhcnRpY2xlcyBhdCBjZWxsIGNlbnRlcnM7IGluaXRpYWwgdmVsb2NpdGllcyBleGFjdGx5IGZyb20gdGhlIGh5ZHJvZHluYW1pYyBpbml0aWFsIHN0YXRlLgp3aXRoIG5wLmxvYWQocm9vdC8nZnVsbF9ETTEvZnJhbWVfKzcwXzFfLTA1MC4wLm5weicpIGFzIHo6VT16WydVJ107eD16Wyd4J107eT16Wyd5J10KWCxZPW5wLm1lc2hncmlkKCh4WzE6XSt4WzotMV0pLzIsKHlbMTpdK3lbOi0xXSkvMixpbmRleGluZz0naWonKTt3PVVbNF0qbnAuZGlmZih4KVs6LE5vbmVdKm5wLmRpZmYoeSlbTm9uZSw6XTt3Lz13LnN1bSgpO21hc2s9dz4xZS0xMjtkaXNjYXJkPTEtd1ttYXNrXS5zdW0oKTt3PXdbbWFza107dy89dy5zdW0oKTtpbml0PW5wLmFycmF5KFtYW21hc2tdLFlbbWFza10sKFVbMV0vVVswXSlbbWFza10sKFVbMl0vVVswXSlbbWFza11dKQpkZWYgcmhzKHQseixkbSk6CiBYLFksdSx2PXo7YXgsYXksXz1mb3JjZSh0LFgsWSk7cmV0dXJuIG5wLmFycmF5KFt1LHYsMipzLk9NRUdBKnYrMipzLlEqcy5PTUVHQSoqMipYK2RtKmF4LC0yKnMuT01FR0EqdStkbSpheV0pClA9W10KZm9yIGR0IGluICguMSwuMDUpOgogZm9yIGRtIGluICgwLDEpOgogIHo9aW5pdC5jb3B5KCk7bj1yb3VuZCgxMDAvZHQpCiAgZm9yIGkgaW4gcmFuZ2UobisxKToKICAgdD0tNTAraSpkdAogICBpZiBpJXJvdW5kKDUvZHQpPT0wOgogICAgWCxZLHUsdj16O3hnPSh2KzIqcy5PTUVHQSpYKS8oKDItcy5RKSpzLk9NRUdBKTtfLF8sUj1mb3JjZSh0LFgsWSk7cD1kaWN0KGR0PWR0LGRtPWRtLHQ9dCx4PWZsb2F0KHdAWCkseGc9ZmxvYXQod0B4Zyksdng9ZmxvYXQod0B1KSxpbl80MD1mbG9hdCh3W1g8LTQwXS5zdW0oKSksaW5uZXJfZmFyX3J0PWZsb2F0KHdbKFg8LTQwKSYoUj49cy5ydCldLnN1bSgpKSxpbm5lcl9mYXJfcmo9ZmxvYXQod1soWDwtNDApJihSPj1yaildLnN1bSgpKSxqb2ludF9ydD1mbG9hdCh3WyhYPC00MCkmKFI+PXMucnQpJih4Zz4wKV0uc3VtKCkpKTtQLmFwcGVuZChwKQogICBpZiBpPT1uOmJyZWFrCiAgIGE9cmhzKHQseixkbSk7Yj1yaHModCtkdC8yLHorZHQqYS8yLGRtKTtjPXJocyh0K2R0LzIseitkdCpiLzIsZG0pO2Q9cmhzKHQrZHQseitkdCpjLGRtKTt6Kz1kdCooYSsyKmIrMipjK2QpLzYKUD1wZC5EYXRhRnJhbWUoUCk7UC50b19jc3Yocm9vdC8nQ1A0Nl9QQVJUSUNMRVMuY3N2JyxpbmRleD1GYWxzZSkKcHJpbnQoJ3JqJyxyaiwncGFydGljbGVzJyxsZW4odyksJ2Rpc2NhcmRlZCcsZGlzY2FyZCkKcHJpbnQoRFsoRC5kbT09MSkmKEQudD49MCldW1sndCcsJ3gnLCd4ZycsJ3Z4JywnZnJlZScsJ3N1YmhhbG8nLCdwcmVzc3VyZScsJ2luXzQwJywnaW5uZXJfZmFyX3J0JywnaW5uZXJfZmFyX3JqJywnam9pbnQ0MF9yaicsJ1RfcTUwJywnVF9xOTAnXV0udG9fc3RyaW5nKGluZGV4PUZhbHNlKSkKcHJpbnQoUFsoUC50PT01MCldLnRvX3N0cmluZyhpbmRleD1GYWxzZSkpCnByaW50KCdtYXggcGFydGljbGUgZHQgZGlmZicsUFtQLmR0PT0uMV0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKS5pbl80MC5zdWIoUFtQLmR0PT0uMDVdLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkuaW5fNDApLmFicygpLm1heCgpKQo=', 'CP46_VALIDATE.py': 'IiIiSW5kZXBlbmRlbnQgcGFydGljbGUgaW50ZWdyYXRvciBhbmQgaW5pdGlhbCBzcGF0aWFsIHNhbXBsaW5nIHNlbnNpdGl2aXR5LiIiIgppbXBvcnQgbnVtcHkgYXMgbnAscGFuZGFzIGFzIHBkCmZyb20gc2NpcHkuaW50ZWdyYXRlIGltcG9ydCBzb2x2ZV9pdnAKaW1wb3J0IENQMjlfU09VUkNFIGFzIHMKcy5TVEFSVD0tNTAuO3MuRU5EPTUwLjtzLlZSKj0uNTt0dCx0cmFjayxfPXMub3JiaXRfZnVsbF9yYW5nZSgpCndpdGggbnAubG9hZCgnZnVsbF9ETTEvZnJhbWVfKzcwXzFfLTA1MC4wLm5weicpIGFzIHo6IFU9elsnVSddO3g9elsneCddO3k9elsneSddCnhjPSh4WzotMV0reFsxOl0pLzI7eWM9KHlbOi0xXSt5WzE6XSkvMjtYLFk9bnAubWVzaGdyaWQoeGMseWMsaW5kZXhpbmc9J2lqJyk7dz1VWzRdKm5wLmRpZmYoeClbOixOb25lXSpucC5kaWZmKHkpW05vbmUsOl07dy89dy5zdW0oKTttYXNrPXc+MWUtMTI7d2VpZ2h0cz13W21hc2tdO3dlaWdodHMvPXdlaWdodHMuc3VtKCk7Wj1ucC5hcnJheShbWFttYXNrXSxZW21hc2tdLG5wLnplcm9zKG1hc2suc3VtKCkpLC1zLlEqcy5PTUVHQSpYW21hc2tdXSkKZGVmIHJocyh0LHopOgogWCxZLHUsdj16LnJlc2hhcGUoNCwtMSk7ZHg9WC1ucC5pbnRlcnAodCx0dCx0cmFja1s6LDBdKTtkeT1ZLW5wLmludGVycCh0LHR0LHRyYWNrWzosMV0pO3JyPW5wLnNxcnQoZHgqZHgrZHkqZHkrLjI1KTtmPS1zLkcyKm5wLmludGVycChycixzLnJncmlkLHMubWdyaWQsbGVmdD0wLHJpZ2h0PXMubWdyaWRbLTFdKS9ycioqMztyZXR1cm4gbnAuYXJyYXkoW3UsdiwyKnMuT01FR0EqdisyKnMuUSpzLk9NRUdBKioyKlgrZipkeCwtMipzLk9NRUdBKnUrZipkeV0pLnJlc2hhcGUoei5zaGFwZSkKZGVmIHJrKHosZHQpOgogej16LmNvcHkoKQogZm9yIGkgaW4gcmFuZ2Uocm91bmQoMTAwL2R0KSk6CiAgdD0tNTAraSpkdDthPXJocyh0LHopO2I9cmhzKHQrZHQvMix6K2R0KmEvMik7Yz1yaHModCtkdC8yLHorZHQqYi8yKTtkPXJocyh0K2R0LHorZHQqYyk7eis9ZHQqKGErMipiKzIqYytkKS82CiByZXR1cm4gegojIENvbXBhcmUgaW5kZXBlbmRlbnQgYWRhcHRpdmUgaW50ZWdyYXRpb24gZm9yIGEgZGV0ZXJtaW5pc3RpYyBzYW1wbGUgb2YgMjUgbWFzcyBxdWFudGlsZXMuCmlkcz1ucC51bmlxdWUobnAuc2VhcmNoc29ydGVkKG5wLmN1bXN1bSh3ZWlnaHRzKSxucC5saW5zcGFjZSguMDIsLjk4LDI1KSkpO3owPVpbOixpZHNdO3NvbD1zb2x2ZV9pdnAocmhzLCgtNTAsNTApLHowLnJhdmVsKCksbWV0aG9kPSdET1A4NTMnLHJ0b2w9MWUtOSxhdG9sPTFlLTksbWF4X3N0ZXA9LjEpO2Fzc2VydCBzb2wuc3VjY2VzcztsYXN0PXNvbC55WzosLTFdLnJlc2hhcGUoejAuc2hhcGUpO3I9cmsoejAsLjA1KTtlcnI9bnAubWF4KG5wLmFicyhsYXN0LXIpLGF4aXM9MSk7cHJpbnQoJ0luZGVwZW5kZW50IGludGVncmF0b3IgY29tcG9uZW50IG1heCBlcnJvcnMnLGVycik7YXNzZXJ0IG1heChlcnJbOjJdKTwuMDIgYW5kIG1heChlcnJbMjpdKTwuMDAyCiMgVHdvLXBvaW50IEdhdXNzaWFuIHF1YWRyYXR1cmUgaW4gZWFjaCBpbml0aWFsIGNlbGwgd2l0aCBhbmFseXRpYyB0YWcgd2VpZ2h0cyB3aXRoaW4gY2VsbCwgbm9ybWFsaXplZCB0byB0aGUgb3JpZ2luYWwgY2VsbCBtYXNzLgpEWCxEWT1ucC5tZXNoZ3JpZChucC5kaWZmKHgpLG5wLmRpZmYoeSksaW5kZXhpbmc9J2lqJyk7YXJyPVtdO3dzPVtdCmZvciBpIGluICgtMSwxKToKIGZvciBqIGluICgtMSwxKToKICB4eD1aWzBdK2kqRFhbbWFza10vKDIqbnAuc3FydCgzKSk7eXk9WlsxXStqKkRZW21hc2tdLygyKm5wLnNxcnQoMykpO2Fyci5hcHBlbmQobnAuYXJyYXkoW3h4LHl5LG5wLnplcm9zX2xpa2UoeHgpLC1zLlEqcy5PTUVHQSp4eF0pKTt3cy5hcHBlbmQobnAuZXhwKC0uNSooKHh4LzMpKioyKygoeXktNzApLzM1KSoqMikpKQp3cz1ucC5hcnJheSh3cyk7d3M9d3Mvd3Muc3VtKGF4aXM9MCkqd2VpZ2h0czsgeno9cmsobnAuY29uY2F0ZW5hdGUoYXJyLGF4aXM9MSksLjA1KTt3dz13cy5yYXZlbCgpO3h4LHl5LHUsdj16ejt4Zz0odisyKnMuT01FR0EqeHgpLygoMi1zLlEpKnMuT01FR0EpO3ByaW50KCdxdWFkcmF0dXJlJyxsZW4od3cpLHd3QHh4LHd3QHhnLHd3W3h4PC00MF0uc3VtKCkpO3ByaW50KCdjZWxsIGNlbnRlcicscGQucmVhZF9jc3YoJ0NQNDZfUEFSVElDTEVTLmNzdicpLnF1ZXJ5KCdkdD09MC4wNSBhbmQgZG09PTEgYW5kIHQ9PTUwJykudG9fZGljdCgncmVjb3JkcycpKQphc3NlcnQgYWJzKHd3W3h4PC00MF0uc3VtKCktLjgxODc3MSk8LjAyIGFuZCBhYnMod3dAeGcrNjUuODU4KTwyCg==', 'CP46_PLOT.py': 'ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBwYW5kYXMgYXMgcGQsbnVtcHkgYXMgbnAsbWF0cGxvdGxpYgptYXRwbG90bGliLnVzZSgnQWdnJykKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApmcm9tIG1hdHBsb3RsaWIuY29sb3JzIGltcG9ydCBMb2dOb3JtCmltcG9ydCBDUDI5X1NPVVJDRSBhcyBzCmZyb20gbWF0cGxvdGxpYi5wYXRjaGVzIGltcG9ydCBDaXJjbGUKcj1QYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50O209ci8nbWFudXNjcmlwdCc7RD1wZC5yZWFkX2NzdihyLydDUDQ2X0ZJRUxEUy5jc3YnKTtQPXBkLnJlYWRfY3N2KHIvJ0NQNDZfUEFSVElDTEVTLmNzdicpO3A9UFsoUC5kdD09LjA1KSYoUC5kbT09MSldO2c9RFtELmRtPT0xXTtjPURbRC5kbT09MF0KcGx0LnJjUGFyYW1zLnVwZGF0ZSh7J2ZvbnQuZmFtaWx5Jzonc2VyaWYnLCdmb250LnNlcmlmJzpbJ0xpYmVyYXRpb24gU2VyaWYnLCdUaW1lcyBOZXcgUm9tYW4nLCdEZWphVnUgU2VyaWYnXSwnbWF0aHRleHQuZm9udHNldCc6J3N0aXgnLCdmb250LnNpemUnOjEyLCdwZGYuZm9udHR5cGUnOjQyLCdheGVzLnNwaW5lcy50b3AnOkZhbHNlLCdheGVzLnNwaW5lcy5yaWdodCc6RmFsc2V9KQpmaWcsYXg9cGx0LnN1YnBsb3RzKDIsMixmaWdzaXplPSg5LDYuOCksbGF5b3V0PSdjb25zdHJhaW5lZCcpCmE9YXhbMCwwXTthLnBsb3QoZy50LGcueCwnby0nLG1zPTMsbGFiZWw9J0dhcyBjZW50cm9pZCcpO2EuZmlsbF9iZXR3ZWVuKGcudC50b19udW1weSgpLGcueF9xMTAudG9fbnVtcHkoKSxnLnhfcTkwLnRvX251bXB5KCksYWxwaGE9LjE1KTthLnBsb3QocC50LHAueCwncy0tJyxtcz0zLGxhYmVsPSdQcmVzc3VyZWxlc3MgY2VudHJvaWQnKTthLmF4aGxpbmUoLTQwLGNvbG9yPScuNScsbHM9JzonKTthLnNldF95bGFiZWwoJ1JhZGlhbCBwb3NpdGlvbiAocGMpJyk7YS5sZWdlbmQoZm9udHNpemU9OSxsb2M9J2xvd2VyIGxlZnQnLGZyYW1lYWxwaGE9Ljg1KQphPWF4WzAsMV07YS5wbG90KGcudCxnLnhnLCdvLScsbXM9MyxsYWJlbD0nR2FzJyk7YS5maWxsX2JldHdlZW4oZy50LnRvX251bXB5KCksZy54Z19xMTAudG9fbnVtcHkoKSxnLnhnX3E5MC50b19udW1weSgpLGFscGhhPS4xNSk7YS5wbG90KHAudCxwLnhnLCdzLS0nLG1zPTMsbGFiZWw9J1ByZXNzdXJlbGVzcycpO2EuYXhobGluZSgwLGNvbG9yPScuNScsbHM9JzonKTthLnNldF95bGFiZWwoJ0d1aWRpbmctY2VudHJlIHByb3h5IChwYyknKTthLmxlZ2VuZChmb250c2l6ZT05LGxvYz0nbG93ZXIgcmlnaHQnLGZyYW1lYWxwaGE9Ljg1KQphPWF4WzEsMF07YS5wbG90KGcudCwxMDAqKGcuaW5fNDAudG9fbnVtcHkoKS1jLmluXzQwLnRvX251bXB5KCkpLCdvLScsbXM9MyxsYWJlbD0nR2FzLCBleGNlc3MgaW52ZW50b3J5Jyk7YS5wbG90KHAudCwxMDAqcC5pbl80MCwncy0tJyxtcz0zLGxhYmVsPSdQcmVzc3VyZWxlc3MnKTthLnBsb3QoZy50LDEwMCpnLmlubmVyX2Zhcl9yaiwnXi0nLG1zPTMsbGFiZWw9J0dhcywgb3V0c2lkZSBIaWxsIHJhZGl1cycpO2EucGxvdChnLnQsMTAwKmcuam9pbnRfcmosJ2QtJyxtcz0zLGxhYmVsPSdBbHNvIHBvc2l0aXZlIHByb3h5Jyk7YS5zZXRfeWxhYmVsKCdJbml0aWFsIHRhZ2dlZCBtYXNzICglKScpO2EubGVnZW5kKGZvbnRzaXplPTguNSxsb2M9J3VwcGVyIGxlZnQnLGZyYW1lYWxwaGE9Ljg1KQphPWF4WzEsMV07CmZvciBrZXksbGFiZWwgaW4gWygnZnJlZScsJ0VwaWN5Y2xpYycpLCgnc3ViaGFsbycsJ1JlbW5hbnQnKSwoJ3ByZXNzdXJlJywnUHJlc3N1cmUgZXN0aW1hdGUnKSwoJ3RvdGFsJywnU3VtJyldOmEucGxvdChnLnQsZ1trZXldLCcuLScsbXM9MyxsYWJlbD1sYWJlbCkKYS5heGhsaW5lKDAsY29sb3I9Jy41Jyxscz0nOicpO2Euc2V0X3lsYWJlbChyJ1JhZGlhbCBhY2NlbGVyYXRpb24gKHBjIE15ciReey0yfSQpJyk7YS5sZWdlbmQoZm9udHNpemU9OC41LGxvYz0nbG93ZXIgcmlnaHQnLGZyYW1lYWxwaGE9Ljg1KQpmb3IgYSBpbiBheC5mbGF0OmEuc2V0X3hsYWJlbCgnVGltZSAoTXlyKScpO2EuYXh2bGluZSgwLGNvbG9yPScuNicsbHM9JzonLGx3PS43KTthLmdyaWQoYWxwaGE9LjE1KQpmaWcuc2F2ZWZpZyhtLydnYXNfcGFydGljbGVfY29tcGFyaXNvbi5wZGYnKTtwbHQuY2xvc2UoZmlnKQojIFJlZ2VuZXJhdGUgc2xvdyB0aGVybWFsIG1hcHMgYW5kIGpvaW50IG1hc3MgZGlzdHJpYnV0aW9ucyBmcm9tIGluZGVwZW5kZW50bHkgZXZvbHZlZCB3aWRlIG1lc2guCmRlZiBmaWVsZHMoZG0sdCk6CiB3aXRoIG5wLmxvYWQoci9mJ2Z1bGxfRE17ZG19L2ZyYW1lXys3MF97ZG19X3t0OiswNi4xZn0ubnB6JykgYXMgejpVPXpbJ1UnXTt4PXpbJ3gnXTt5PXpbJ3knXTtteD1mbG9hdCh6WydkbV94J10pO215PWZsb2F0KHpbJ2RtX3knXSkKIHJobz1VWzBdO3A9KHMuR0FNTUEtMSkqKFVbM10tLjUqKFVbMV0qKjIrVVsyXSoqMikvcmhvKTtUPXMuVHcqcC8ocy5QMF9SQUQqcmhvKTtXPVVbNF0qbnAuZGlmZih4KVs6LE5vbmVdKm5wLmRpZmYoeSlbTm9uZSw6XTtyZXR1cm4gVSx4LHksVCxXLG14LG15CmZpZyxheD1wbHQuc3VicGxvdHMoMiwyLGZpZ3NpemU9KDguNCw3LjEpLGxheW91dD0nY29uc3RyYWluZWQnKTtmaWVsZHM0PVtmaWVsZHMoMSx0KSBmb3IgdCBpbiAoMCwyMCwzMCw1MCldO2xvPW1pbih2WzNdLm1pbigpIGZvciB2IGluIGZpZWxkczQpO2hpPW1heCh2WzNdLm1heCgpIGZvciB2IGluIGZpZWxkczQpCmZvciBhLHQsKFUseCx5LFQsVyxteCxteSkgaW4gemlwKGF4LmZsYXQsKDAsMjAsMzAsNTApLGZpZWxkczQpOgogaW09YS5wY29sb3JtZXNoKHgseSxULlQsbm9ybT1Mb2dOb3JtKGxvLGhpKSxjbWFwPSdpbmZlcm5vJyxzaGFkaW5nPSdmbGF0JyxyYXN0ZXJpemVkPVRydWUpO2EuY29udG91cigoeFsxOl0reFs6LTFdKS8yLCh5WzE6XSt5WzotMV0pLzIsVVs0XS5ULGxldmVscz1bLjEsMSwxMF0sY29sb3JzPVsnIzljZWNmYicsJyMzMmM5ZGYnLCcjMDBhNmNmJ10sbGluZXdpZHRocz0uOCk7YS5hZGRfcGF0Y2goQ2lyY2xlKChteCxteSksNDAsZmlsbD1GYWxzZSxlZGdlY29sb3I9J3doaXRlJyxsdz0uOSkpO2EucGxvdChteCxteSwndysnLG1zPTcpO2EuYXh2bGluZSgtNDAsYz0nd2hpdGUnLGxzPSctLScsbHc9LjgpO2Euc2V0KHhsYWJlbD0neCAocGMpJyx5bGFiZWw9J3kgKHBjKScsdGl0bGU9Zid0ID0ge3R9IE15cicseGxpbT0oLTUwMCw1MDApLHlsaW09KC01MDAsNTAwKSxhc3BlY3Q9J2VxdWFsJykKZmlnLmNvbG9yYmFyKGltLGF4PWF4LnJhdmVsKCkudG9saXN0KCksbGFiZWw9J1RlbXBlcmF0dXJlIChLKScsc2hyaW5rPS44KTtmaWcuc2F2ZWZpZyhtLydyZWNvbnN0cnVjdGVkX3Nsb3dfbWFwcy5wZGYnKTtwbHQuY2xvc2UoZmlnKQpmaWcsYXg9cGx0LnN1YnBsb3RzKDIsMixmaWdzaXplPSg4LjUsNi41KSxsYXlvdXQ9J2NvbnN0cmFpbmVkJyk7Ymluc3g9bnAuYXJhbmdlKC01MDAsNTA1LDUpO2JpbnNUPW5wLmdlb21zcGFjZSgxMCwzZTQsMTIxKQpmb3IgYSwoZG0sdCkgaW4gemlwKGF4LmZsYXQsWygwLDApLCgxLDApLCgwLDUwKSwoMSw1MCldKToKIFUseCx5LFQsVyxfLF89ZmllbGRzKGRtLHQpO1g9bnAuYnJvYWRjYXN0X3RvKCgoeFs6LTFdK3hbMTpdKS8yKVs6LE5vbmVdLFcuc2hhcGUpO0gsKl89bnAuaGlzdG9ncmFtMmQoWC5yYXZlbCgpLFQucmF2ZWwoKSxiaW5zPShiaW5zeCxiaW5zVCksd2VpZ2h0cz1XLnJhdmVsKCkvVy5zdW0oKSk7YXNzZXJ0IEguc3VtKCk+Ljk5OTkKIGltPWEucGNvbG9ybWVzaChiaW5zeCxiaW5zVCxILlQsbm9ybT1Mb2dOb3JtKDFlLTUsLjEpLGNtYXA9J3ZpcmlkaXMnLHNoYWRpbmc9J2ZsYXQnLHJhc3Rlcml6ZWQ9VHJ1ZSk7YS5heHZsaW5lKC00MCxjPSd3aGl0ZScsbHM9Jy0tJyxsdz0uOCk7YS5zZXQoeXNjYWxlPSdsb2cnLHhsaW09KC0zNTAsMTUwKSx4bGFiZWw9J3ggKHBjKScseWxhYmVsPSdUZW1wZXJhdHVyZSAoSyknLHRpdGxlPWYneyJFbmNvdW50ZXIiIGlmIGRtIGVsc2UgIkNvbnRyb2wifSwge3R9IE15cicpCmZpZy5jb2xvcmJhcihpbSxheD1heC5yYXZlbCgpLnRvbGlzdCgpLGxhYmVsPSdUYWdnZWQgbWFzcyBmcmFjdGlvbiBwZXIgYmluJyxzaHJpbms9LjgpO2ZpZy5zYXZlZmlnKG0vJ3Nsb3dfam9pbnRfZGVuc2l0eS5wZGYnKTtwbHQuY2xvc2UoZmlnKQo=', 'GRID.npz': 'UEsDBC0AAAAAAAAAIQDTPjhj//////////8FABQAeC5ucHkBABAAmBAAAAAAAACYEAAAAAAAAJNOVU1QWQEAdgB7J2Rlc2NyJzogJzxmOCcsICdmb3J0cmFuX29yZGVyJzogRmFsc2UsICdzaGFwZSc6ICg1MTUsKSwgfSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAKAAAAAABAf8AAAAAAAKB+wAAAAAAAAH7AAAAAAABgfcAAAAAAAMB8wAAAAAAAIHzAAAAAAACAe8AAAAAAAOB6wAAAAAAAQHrAAAAAAACgecAAAAAAAAB5wAAAAAAAYHjAAAAAAADAd8AAAAAAACB3wAAAAAAAgHbAAAAAAADgdcAAAAAAAEB1wAAAAAAAoHTAAAAAAAAAdMAAAAAAAGBzwAAAAAAAwHLAAAAAAACwcsAAAAAAAKBywAAAAAAAkHLAAAAAAACAcsAAAAAAAHBywAAAAAAAYHLAAAAAAABQcsAAAAAAAEBywAAAAAAAMHLAAAAAAAAgcsAAAAAAABBywAAAAAAAAHLAAAAAAADwccAAAAAAAOBxwAAAAAAA0HHAAAAAAADAccAAAAAAALBxwAAAAAAAoHHAAAAAAACQccAAAAAAAIBxwAAAAAAAcHHAAAAAAABgccAAAAAAAFBxwAAAAAAAQHHAAAAAAAAwccAAAAAAACBxwAAAAAAAEHHAAAAAAAAAccAAAAAAAPBwwAAAAAAA4HDAAAAAAADQcMAAAAAAAMBwwAAAAAAAsHDAAAAAAACgcMAAAAAAAJBwwAAAAAAAgHDAAAAAAABwcMAAAAAAAGBwwAAAAAAAUHDAAAAAAABAcMAAAAAAADBwwAAAAAAAIHDAAAAAAAAQcMAAAAAAAABwwAAAAAAA4G/AAAAAAADAb8AAAAAAAKBvwAAAAAAAgG/AAAAAAABgb8AAAAAAAEBvwAAAAAAAIG/AAAAAAAAAb8AAAAAAAOBuwAAAAAAAwG7AAAAAAACgbsAAAAAAAIBuwAAAAAAAYG7AAAAAAABAbsAAAAAAACBuwAAAAAAAAG7AAAAAAADgbcAAAAAAAMBtwAAAAAAAoG3AAAAAAACAbcAAAAAAAGBtwAAAAAAAQG3AAAAAAAAgbcAAAAAAAABtwAAAAAAA4GzAAAAAAADAbMAAAAAAAKBswAAAAAAAgGzAAAAAAABgbMAAAAAAAEBswAAAAAAAIGzAAAAAAAAAbMAAAAAAAOBrwAAAAAAAwGvAAAAAAACga8AAAAAAAIBrwAAAAAAAYGvAAAAAAABAa8AAAAAAACBrwAAAAAAAAGvAAAAAAADgasAAAAAAAMBqwAAAAAAAoGrAAAAAAACAasAAAAAAAGBqwAAAAAAAQGrAAAAAAAAgasAAAAAAAABqwAAAAAAA4GnAAAAAAADAacAAAAAAAKBpwAAAAAAAgGnAAAAAAABgacAAAAAAAEBpwAAAAAAAIGnAAAAAAAAAacAAAAAAAOBowAAAAAAAwGjAAAAAAACgaMAAAAAAAIBowAAAAAAAYGjAAAAAAABAaMAAAAAAACBowAAAAAAAAGjAAAAAAADgZ8AAAAAAAMBnwAAAAAAAoGfAAAAAAACAZ8AAAAAAAGBnwAAAAAAAQGfAAAAAAAAgZ8AAAAAAAABnwAAAAAAA4GbAAAAAAADAZsAAAAAAAKBmwAAAAAAAgGbAAAAAAABgZsAAAAAAAEBmwAAAAAAAIGbAAAAAAAAAZsAAAAAAAOBlwAAAAAAAwGXAAAAAAACgZcAAAAAAAIBlwAAAAAAAYGXAAAAAAABAZcAAAAAAACBlwAAAAAAAAGXAAAAAAADgZMAAAAAAAMBkwAAAAAAAoGTAAAAAAACAZMAAAAAAAGBkwAAAAAAAQGTAAAAAAAAgZMAAAAAAAABkwAAAAAAA4GPAAAAAAADAY8AAAAAAAKBjwAAAAAAAgGPAAAAAAABgY8AAAAAAAEBjwAAAAAAAIGPAAAAAAAAAY8AAAAAAAOBiwAAAAAAAwGLAAAAAAACgYsAAAAAAAIBiwAAAAAAAYGLAAAAAAABAYsAAAAAAACBiwAAAAAAAAGLAAAAAAADgYcAAAAAAAMBhwAAAAAAAoGHAAAAAAACAYcAAAAAAAGBhwAAAAAAAQGHAAAAAAAAgYcAAAAAAAABhwAAAAAAA4GDAAAAAAADAYMAAAAAAAKBgwAAAAAAAgGDAAAAAAABgYMAAAAAAAEBgwAAAAAAAIGDAAAAAAAAAYMAAAAAAAMBfwAAAAAAAgF/AAAAAAABAX8AAAAAAAABfwAAAAAAAwF7AAAAAAACAXsAAAAAAAEBewAAAAAAAAF7AAAAAAADAXcAAAAAAAIBdwAAAAAAAQF3AAAAAAAAAXcAAAAAAAMBcwAAAAAAAgFzAAAAAAABAXMAAAAAAAABcwAAAAAAAwFvAAAAAAACAW8AAAAAAAEBbwAAAAAAAAFvAAAAAAADAWsAAAAAAAIBawAAAAAAAQFrAAAAAAAAAWsAAAAAAAMBZwAAAAAAAgFnAAAAAAABAWcAAAAAAAABZwAAAAAAAwFjAAAAAAACAWMAAAAAAAEBYwAAAAAAAAFjAAAAAAADAV8AAAAAAAIBXwAAAAAAAQFfAAAAAAAAAV8AAAAAAAMBWwAAAAAAAgFbAAAAAAABAVsAAAAAAAABWwAAAAAAAwFXAAAAAAACAVcAAAAAAAEBVwAAAAAAAAFXAAAAAAADAVMAAAAAAAIBUwAAAAAAAQFTAAAAAAAAAVMAAAAAAAMBTwAAAAAAAgFPAAAAAAABAU8AAAAAAAABTwAAAAAAAwFLAAAAAAACAUsAAAAAAAEBSwAAAAAAAAFLAAAAAAADAUcAAAAAAAIBRwAAAAAAAQFHAAAAAAAAAUcAAAAAAAMBQwAAAAAAAgFDAAAAAAABAUMAAAAAAAABQwAAAAAAAgE/AAAAAAAAAT8AAAAAAAIBOwAAAAAAAAE7AAAAAAACATcAAAAAAAABNwAAAAAAAgEzAAAAAAAAATMAAAAAAAIBLwAAAAAAAAEvAAAAAAACASsAAAAAAAABKwAAAAAAAgEnAAAAAAAAAScAAAAAAAIBIwAAAAAAAAEjAAAAAAACAR8AAAAAAAABHwAAAAAAAgEbAAAAAAAAARsAAAAAAAIBFwAAAAAAAAEXAAAAAAACARMAAAAAAAABEwAAAAAAAgEPAAAAAAAAAQ8AAAAAAAIBCwAAAAAAAAELAAAAAAACAQcAAAAAAAABBwAAAAAAAgEDAAAAAAAAAQMAAAAAAAAA/wAAAAAAAAD7AAAAAAAAAPcAAAAAAAAA8wAAAAAAAADvAAAAAAAAAOsAAAAAAAAA5wAAAAAAAADjAAAAAAAAAN8AAAAAAAAA2wAAAAAAAADXAAAAAAAAANMAAAAAAAAAzwAAAAAAAADLAAAAAAAAAMcAAAAAAAAAwwAAAAAAAAC7AAAAAAAAALMAAAAAAAAAqwAAAAAAAACjAAAAAAAAAJsAAAAAAAAAkwAAAAAAAACLAAAAAAAAAIMAAAAAAAAAcwAAAAAAAABjAAAAAAAAAFMAAAAAAAAAQwAAAAAAAAAjAAAAAAAAAAMAAAAAAAADwvwAAAAAAAAAAAAAAAAAA8D8AAAAAAAAAQAAAAAAAAAhAAAAAAAAAEEAAAAAAAAAUQAAAAAAAABhAAAAAAAAAHEAAAAAAAAAgQAAAAAAAACJAAAAAAAAAJEAAAAAAAAAmQAAAAAAAAChAAAAAAAAAKkAAAAAAAAAsQAAAAAAAAC5AAAAAAAAAMEAAAAAAAAAxQAAAAAAAADJAAAAAAAAAM0AAAAAAAAA0QAAAAAAAADVAAAAAAAAANkAAAAAAAAA3QAAAAAAAADhAAAAAAAAAOUAAAAAAAAA6QAAAAAAAADtAAAAAAAAAPEAAAAAAAAA9QAAAAAAAAD5AAAAAAAAAP0AAAAAAAABAQAAAAAAAgEBAAAAAAAAAQUAAAAAAAIBBQAAAAAAAAEJAAAAAAACAQkAAAAAAAABDQAAAAAAAgENAAAAAAAAAREAAAAAAAIBEQAAAAAAAAEVAAAAAAACARUAAAAAAAABGQAAAAAAAgEZAAAAAAAAAR0AAAAAAAIBHQAAAAAAAAEhAAAAAAACASEAAAAAAAABJQAAAAAAAgElAAAAAAAAASkAAAAAAAIBKQAAAAAAAAEtAAAAAAACAS0AAAAAAAABMQAAAAAAAgExAAAAAAAAATUAAAAAAAIBNQAAAAAAAAE5AAAAAAACATkAAAAAAAABPQAAAAAAAgE9AAAAAAAAAUEAAAAAAAEBQQAAAAAAAgFBAAAAAAADAUEAAAAAAAABRQAAAAAAAQFFAAAAAAACAUUAAAAAAAMBRQAAAAAAAAFJAAAAAAABAUkAAAAAAAIBSQAAAAAAAwFJAAAAAAAAAU0AAAAAAAEBTQAAAAAAAgFNAAAAAAADAU0AAAAAAAABUQAAAAAAAQFRAAAAAAACAVEAAAAAAAMBUQAAAAAAAAFVAAAAAAABAVUAAAAAAAIBVQAAAAAAAwFVAAAAAAAAAVkAAAAAAAEBWQAAAAAAAgFZAAAAAAADAVkAAAAAAAABXQAAAAAAAQFdAAAAAAACAV0AAAAAAAMBXQAAAAAAAAFhAAAAAAABAWEAAAAAAAIBYQAAAAAAAwFhAAAAAAAAAWUAAAAAAAEBZQAAAAAAAgFlAAAAAAADAWUAAAAAAAABaQAAAAAAAQFpAAAAAAACAWkAAAAAAAMBaQAAAAAAAAFtAAAAAAABAW0AAAAAAAIBbQAAAAAAAwFtAAAAAAAAAXEAAAAAAAEBcQAAAAAAAgFxAAAAAAADAXEAAAAAAAABdQAAAAAAAQF1AAAAAAACAXUAAAAAAAMBdQAAAAAAAAF5AAAAAAABAXkAAAAAAAIBeQAAAAAAAwF5AAAAAAAAAX0AAAAAAAEBfQAAAAAAAgF9AAAAAAADAX0AAAAAAAABgQAAAAAAAIGBAAAAAAABAYEAAAAAAAGBgQAAAAAAAgGBAAAAAAACgYEAAAAAAAMBgQAAAAAAA4GBAAAAAAAAAYUAAAAAAACBhQAAAAAAAQGFAAAAAAABgYUAAAAAAAIBhQAAAAAAAoGFAAAAAAADAYUAAAAAAAOBhQAAAAAAAAGJAAAAAAAAgYkAAAAAAAEBiQAAAAAAAYGJAAAAAAACAYkAAAAAAAKBiQAAAAAAAwGJAAAAAAADgYkAAAAAAAABjQAAAAAAAIGNAAAAAAABAY0AAAAAAAGBjQAAAAAAAgGNAAAAAAACgY0AAAAAAAMBjQAAAAAAA4GNAAAAAAAAAZEAAAAAAAEBlQAAAAAAAgGZAAAAAAADAZ0AAAAAAAABpQAAAAAAAQGpAAAAAAACAa0AAAAAAAMBsQAAAAAAAAG5AAAAAAABAb0AAAAAAAEBwQAAAAAAA4HBAAAAAAACAcUAAAAAAACByQAAAAAAAwHJAAAAAAABgc0AAAAAAAAB0QAAAAAAAoHRAAAAAAABAdUAAAAAAAOB1QAAAAAAAgHZAAAAAAAAgd0AAAAAAAMB3QAAAAAAAYHhAAAAAAAAAeUAAAAAAAKB5QAAAAAAAQHpAAAAAAADgekAAAAAAAIB7QAAAAAAAIHxAAAAAAADAfEAAAAAAAGB9QAAAAAAAAH5AAAAAAACgfkAAAAAAAEB/QFBLAwQtAAAAAAAAACEAGTHsEP//////////BQAUAHkubnB5AQAQAAgHAAAAAAAACAcAAAAAAACTTlVNUFkBAHYAeydkZXNjcic6ICc8ZjgnLCAnZm9ydHJhbl9vcmRlcic6IEZhbHNlLCAnc2hhcGUnOiAoMjA5LCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgCgAAAAAAQH/AAAAAAACgfsAAAAAAAAB+wAAAAAAAYH3AAAAAAADAfMAAAAAAACB8wAAAAAAAgHvAAAAAAADgesAAAAAAAEB6wAAAAAAAoHnAAAAAAAAAecAAAAAAAGB4wAAAAAAAwHfAAAAAAAAgd8AAAAAAAIB2wAAAAAAA4HXAAAAAAABAdcAAAAAAAKB0wAAAAAAAAHTAAAAAAABgc8AAAAAAAMBywAAAAAAAIHLAAAAAAACAccAAAAAAAOBwwAAAAAAAQHDAAAAAAABAb8AAAAAAAABuwAAAAAAAwGzAAAAAAACAa8AAAAAAAEBqwAAAAAAAAGnAAAAAAADAZ8AAAAAAAIBmwAAAAAAAMGbAAAAAAADgZcAAAAAAAJBlwAAAAAAAQGXAAAAAAADwZMAAAAAAAKBkwAAAAAAAUGTAAAAAAAAAZMAAAAAAALBjwAAAAAAAYGPAAAAAAAAQY8AAAAAAAMBiwAAAAAAAcGLAAAAAAAAgYsAAAAAAANBhwAAAAAAAgGHAAAAAAAAwYcAAAAAAAOBgwAAAAAAAkGDAAAAAAABAYMAAAAAAAOBfwAAAAAAAQF/AAAAAAACgXsAAAAAAAABewAAAAAAAYF3AAAAAAADAXMAAAAAAACBcwAAAAAAAgFvAAAAAAADgWsAAAAAAAEBawAAAAAAAoFnAAAAAAAAAWcAAAAAAAGBYwAAAAAAAwFfAAAAAAAAgV8AAAAAAAIBWwAAAAAAA4FXAAAAAAABAVcAAAAAAAKBUwAAAAAAAAFTAAAAAAABgU8AAAAAAAMBSwAAAAAAAIFLAAAAAAACAUcAAAAAAAOBQwAAAAAAAQFDAAAAAAABAT8AAAAAAAABOwAAAAAAAwEzAAAAAAACAS8AAAAAAAEBKwAAAAAAAAEnAAAAAAADAR8AAAAAAAIBGwAAAAAAAQEXAAAAAAAAARMAAAAAAAMBCwAAAAAAAgEHAAAAAAABAQMAAAAAAAAA+wAAAAAAAgDvAAAAAAAAAOcAAAAAAAIA2wAAAAAAAADTAAAAAAACAMcAAAAAAAAAuwAAAAAAAACnAAAAAAAAAJMAAAAAAAAAewAAAAAAAABTAAAAAAAAABMAAAAAAAAAAAAAAAAAAAARAAAAAAAAAFEAAAAAAAAAeQAAAAAAAACRAAAAAAAAAKUAAAAAAAAAuQAAAAAAAgDFAAAAAAAAANEAAAAAAAIA2QAAAAAAAADlAAAAAAACAO0AAAAAAAAA+QAAAAAAAQEBAAAAAAACAQUAAAAAAAMBCQAAAAAAAAERAAAAAAABARUAAAAAAAIBGQAAAAAAAwEdAAAAAAAAASUAAAAAAAEBKQAAAAAAAgEtAAAAAAADATEAAAAAAAABOQAAAAAAAQE9AAAAAAABAUEAAAAAAAOBQQAAAAAAAgFFAAAAAAAAgUkAAAAAAAMBSQAAAAAAAYFNAAAAAAAAAVEAAAAAAAKBUQAAAAAAAQFVAAAAAAADgVUAAAAAAAIBWQAAAAAAAIFdAAAAAAADAV0AAAAAAAGBYQAAAAAAAAFlAAAAAAACgWUAAAAAAAEBaQAAAAAAA4FpAAAAAAACAW0AAAAAAACBcQAAAAAAAwFxAAAAAAABgXUAAAAAAAABeQAAAAAAAoF5AAAAAAABAX0AAAAAAAOBfQAAAAAAAQGBAAAAAAACQYEAAAAAAAOBgQAAAAAAAMGFAAAAAAACAYUAAAAAAANBhQAAAAAAAIGJAAAAAAABwYkAAAAAAAMBiQAAAAAAAEGNAAAAAAABgY0AAAAAAALBjQAAAAAAAAGRAAAAAAABQZEAAAAAAAKBkQAAAAAAA8GRAAAAAAABAZUAAAAAAAJBlQAAAAAAA4GVAAAAAAAAwZkAAAAAAAIBmQAAAAAAAwGdAAAAAAAAAaUAAAAAAAEBqQAAAAAAAgGtAAAAAAADAbEAAAAAAAABuQAAAAAAAQG9AAAAAAABAcEAAAAAAAOBwQAAAAAAAgHFAAAAAAAAgckAAAAAAAMByQAAAAAAAYHNAAAAAAAAAdEAAAAAAAKB0QAAAAAAAQHVAAAAAAADgdUAAAAAAAIB2QAAAAAAAIHdAAAAAAADAd0AAAAAAAGB4QAAAAAAAAHlAAAAAAACgeUAAAAAAAEB6QAAAAAAA4HpAAAAAAACAe0AAAAAAACB8QAAAAAAAwHxAAAAAAABgfUAAAAAAAAB+QAAAAAAAoH5AAAAAAABAf0BQSwECLQMtAAAAAAAAACEA0z44Y5gQAACYEAAABQAAAAAAAAAAAAAAgAEAAAAAeC5ucHlQSwECLQMtAAAAAAAAACEAGTHsEAgHAAAIBwAABQAAAAAAAAAAAAAAgAHPEAAAeS5ucHlQSwUGAAAAAAIAAgBmAAAADhgAAAAA', 'original_DM0.csv': 'dF9NeXIscGhhc2VfcGMsRE0sbngsbnksZGVsdGFfSix0YWdfbWVhbl92eF9rbXMsdGFnX2NlbnRlcl94X3BjLHRhZ19tYXNzX3JlbF9jaGFuZ2UscmhvX21pbixwX21pbix0YWdfbWluLHRhZ19leGNlc3NfbWF4LGZhbGxiYWNrX2ZhY2VzLHN0ZXBfY291bnQsdGFnX21lYW5fVF9LLHRhZ19UX2x0MTAwLHRhZ19UXzEwMF8xMDAwLHRhZ19UX2dlMTAwMCx0YWdfY2VudGVyX3lfcGMsdGFnX3Jtc19tYWpvcl9wYyx0YWdfcm1zX21pbm9yX3BjLHRhZ19hc3BlY3QsdGFnX2luc2lkZV8yMHBjLHRhZ19pbnNpZGVfNDBwYyx0YWdfaW5zaWRlX3J0LGRtX3hfcGMsZG1feV9wYyxUX21pbl9LLFRfbWF4X0ssdGFnX1RfbmVhcl9LLHRhZ19UX2Zhcl9LLGN1bV9pbl94LTQwLG1hc3NfaW5jcl94LTQwLGJ1ZGdldF9yZXNfeC00MCxsZWZ0X2ZsdXhfeC00MCx5X2xvc3NfeC00MCxjdW1faW5feDAsbWFzc19pbmNyX3gwLGJ1ZGdldF9yZXNfeDAsbGVmdF9mbHV4X3gwLHlfbG9zc194MCxjdW1faW5feDQwLG1hc3NfaW5jcl94NDAsYnVkZ2V0X3Jlc194NDAsbGVmdF9mbHV4X3g0MCx5X2xvc3NfeDQwDQotNTAuMCw3MC4wLDAuMCwyNzYuMCwyMDguMCwtNS45MTY2ODk1MDEwNDUwNTZlLTE4LDAuMCwwLjAsMC4wLDEuMCw0Mi4xOTgwNzUzMTAxMzUxNiwwLjAsLTEuMCwwLjAsMC4wLDE5NC41ODM0NTUxMjY3NjE2NiwwLjQ4NDQyMDY3NTM0NTIyNzM3LDAuNDkxMjY3MzA4ODQ0MzI4MiwwLjAyNDMxMjAxNTgxMDQ0NDQ0Nyw3MC4wMDA2Mjk3ODMxNzE3OSwzNS4wMDk1MjAzNTg4NzEwNCwzLjAxMzg1Njg4NjY3MDg1NDYsMTEuNjE2MTg1Mzk4MDg3MzcsMC4wLDAuMCwwLjAsMjA4LjU5NTMzNTMyNTI2Nzg4LDI0OC41NTcwNjA4OTMwMDk0NCw1My4yNzA1MTQ3MzI1MjM3OSw2MzUzLjk0Nzk2MzA4NDQzNixuYW4sMTk0LjU4MzQ1NTEyNjc2MTY2LC0wLjAsMC4wLDAuMCwwLjAsMC4wLC0wLjAsMC4wLDAuMCwwLjAsMC4wLC0wLjAsMC4wLDAuMCwwLjAsMC4wDQotNDAuMCw3MC4wLDAuMCwyNzYuMCwyMDguMCw4LjM4Njc1OTg4NTQ0Njg0NGUtMDUsMi44NzA5NTU4NTczODYyNThlLTA2LDQuNzYzOTQwOTgwMzc0OTQ3ZS0wNSwwLjAsMC45NzQ1NjkwNTMxNjIzMTU3LDI1LjA4NDk1MTk0ODE0NzY1NSwwLjAsLTAuODUzMzk1MzkwNDEwNjI1MywwLjAsNDI4LjAsNzEuMzA0NzI2MTAwMDQsMC45NzU3ODM4MTQxNzk1NTU3LDAuMDIwNzU2Mjc2OTU3ODgxODY4LDAuMDAzNDU5OTA4ODYyNTYyNDMsNzAuMDAxMTgwNTgyMTMwNDQsMzQuNjI2OTA2NDIwOTgxNzMsMi4xMzE3NDk5ODEyODU4NTI3LDE2LjI0MzQxODE4OTI2Mzk2LDAuMCwwLjAsMy40NTM1OTM4NTkxMzQ0Nzg0ZS0wOSwxOTAuNjgxNTY1MjI1MzU0MTgsMTczLjY0NTk1MzcyMjc4NTcsNDEuMjc1OTAzMjk4MDgyMTg0LDYzNTcuODQxNDI4MzU3MjQ5LG5hbiw3MS4zMDQ3MjYxMDAwNCw1Ljg2NzQ4Nzg2MzE3MTM4ZS0yMCw1Ljg2NzQ4Nzg2MzE3MTM4MmUtMjAsMS4wMjYzODIwMTc3NTI4MzZlLTM1LDAuMCwyLjMwMDQ3OTc2NzY4NDgxNDZlLTgxLC0xLjQyOTQ3MDY2Mzc3MDYyNTZlLTA1LC0xLjQyOTQ3MDY2Mzc4MTU1MzllLTA1LC0xLjA5MjgyNTI2MjQ4MDYyMTdlLTE2LDAuMCw3LjAwNTc4NjgyMjc5OTE5MzVlLTQ2LC02LjAyMjExMTA2MjAzNzEwNmUtMjAsMC4wLDYuMDIyMTExMDYyMDM3MTA2ZS0yMCwwLjAsNy4wMDU3ODY4MjI3OTkxOTM1ZS00Ng0KLTMwLjAsNzAuMCwwLjAsMjc2LjAsMjA4LjAsNi41NDQ4NzYzMzMzODc4NjZlLTA1LC0xLjAzNDA5OTUxODE2MjIyNTZlLTA1LDAuMDAwMTQ2MzcwODIxMTYyNDQyNjIsMC4wLDAuODQ4NDMwNDU0NDQzNjY4MiwyNC41MzMyNDE3MTk2Mzc0OSwwLjAsLTAuNjkxMDQ0MDg3ODQ3ODYzOCwwLjAsODgwLjAsNzAuMzc5NjQwNzI4NTM4NzYsMC45ODUwNTUwNDM2MzcyOTQ1LDAuMDEwNjMzODc1NTI2MDY2MjYxLDAuMDA0MzExMDgwODM2NjM5NDMzLDcwLjAwMjA2NTU5NjQ2MTc2LDMzLjkyMDU1NDM0MTQxNjU2LDEuNDAwOTE0MjAxNzc3NTQwNCwyNC4yMTMxNTYxNjUwMDc2MDcsNy4xMzI5Nzg4NjcyMjc1MTJlLTcyLDMuNDAzNzYwODg4NDkyOTYwNWUtNjcsMC43NzQwNDM1NjA5NDM0NDI0LDE1OC40OTc5NTI5MjE1MjQxMywxMDcuNDQwNTQ4NjczMTcxNzIsNDEuMjE1NjQ5OTkwMzk0NDMsNjYzNi4yMTcwODMxNzE2NjksbmFuLDcwLjM3OTY0MDcyODUzODc2LDMuODE4MTI4MjgwNjE3MDAyZS0xNywzLjgxODEyODI4MDYxNzAwNGUtMTcsMi4xMDIwMzAzNzIzNTc4MDhlLTMyLDAuMCwzLjgzMTQ0NzUyNzA4OTEwMzZlLTYyLDMuNDYxNTYyMTEwNjc2Njk1N2UtMDUsMy40NjE1NjIxMTA2NjkyNzE2ZS0wNSwtNy40MjQxNzQxNjAwNTE1NTVlLTE3LDAuMCwxLjI5ODk3NTEzNzc2MTg0NjNlLTQxLC03LjA3NzI1MzE0MDcyMTE4NWUtMTcsLTEuODkzMzQwNjQwMzM0NDE3OWUtMTYsLTEuMTg1NjE1MzI2MjYyMjk5NGUtMTYsMC4wLDEuMjk4OTc1MTM3NzYxODQ2M2UtNDENCi0yMC4wLDcwLjAsMC4wLDI3Ni4wLDIwOC4wLDguNzA5ODU5NTgyMTk3NDQ3ZS0wNSwtNy4wMzM2NTEwNTY1MzcxMzVlLTA2LDMuMjE1ODQ4OTM4NzMxOTMyNGUtMDUsMC4wLDAuODY4MTc3OTEyMDE5NDY3NCwyNC43NjczNTQxNDgzODA5NCwwLjAsLTAuNjM2Nzg3OTg2NTkzMjg2NCwwLjAsMTMzNC4wLDc5Ljk1NTYwMzIwNDE3MjYyLDAuOTc2ODYzNDQ2NzQyNzMzNiwwLjAxOTExOTEyMTAxMjgzMDA2NiwwLjAwNDAxNzQzMjI0NDQzNjA1MzUsNzAuMDAyNzc2MzQxMzkwOSwzMy4zODQ4MzU2NDI3NTU4MjQsMi41NzAzMzU5MjQ2MjM3NDEsMTIuOTg4NTEwNjkzNDU4NDI4LDEuMzM1OTEwMjk2NjMzMzllLTYzLDIuMDIyMjA0NzM2OTI1OTg0N2UtNTQsMC45OTkzNzgxOTg3MDc0Mzc4LDExNC4wNTE5OTkwNDU2NTgxNSw1NC4yMzIxMTI4Njg4NDU2Myw1MS4xNTg5MzkwMDMxMTMzNiw2NDk3LjM0ODU2OTk5OTE0NixuYW4sNzkuOTU1NjAzMjA0MTcyNjMsMi4yNTE5ODUyODk3MDU2MDgzZS0xNywyLjI1MTk4NTI4OTcwNTYwOWUtMTcsNy44ODI2MTM4OTYzNDE3OGUtMzMsMC4wLDkuMDUwOTI1MzQ4MDA0MDU0ZS01Myw2LjIwODAxNzYyNzY1OTk4M2UtMDUsNi4yMDgwMTc2Mjc2NjEyMTZlLTA1LDEuMjMzMDI4ODQ3MTkwNDQ0M2UtMTcsMC4wLDYuMjgyODgwMzgzNTEyODM1ZS0zOSwtNS40ODgyMjY2NTYzOTEyNGUtMTcsLTEuODkzMzQwNjQwMzM0NDE3OWUtMTYsLTEuMzQ0NTE3OTc0Njk1MjkzOGUtMTYsMC4wLDYuMjgyODgwMzgzNTE0MDgzZS0zOQ0KLTEwLjAsNzAuMCwwLjAsMjc2LjAsMjA4LjAsMC4wMDAxMzQ3ODQyNzI4MjM2NTU1NiwtOC43NjY4NTc5NDIwMTIyNjhlLTA1LC0wLjAwMDI2Nzc0MDc2ODEyODY2NDIsLTIuMjIwNDQ2MDQ5MjUwMzEzZS0xNiwwLjg1MDMyNDIwNjM1OTU5NDksMjQuMTU1ODYyMjYzNzg2OTY1LDAuMCwtMC40NzMxNTY4NzUxMjMxNTg4NSwwLjAsMTgwMi4wLDgyLjAwNzE5NDc2ODEzOTM0LDAuOTY5OTQ5MjEzMjc1OTg0NSwwLjAyNjIyMjY4OTc4NzAzNTI1MywwLjAwMzgyODA5NjkzNjk4MDI0MjUsNzAuMDA0MDMwOTcxOTQ5MjUsMzIuOTUxNTEyNTM3MDIwODA0LDIuOTYyMzEwNzYzNTg2MTUxNiwxMS4xMjM1ODM5NzQ0MDA0MjUsMS42NTcyNTQxNDM4NzcyODZlLTMwLDcuODg3MzUxMjc2OTY5NDcyZS0xNiwwLjk5OTQ0MjkwOTc5Mzc3MTYsNjAuMTA2NzIxNTUxNjEwMzM1LDE3LjU2MTUwODAyMDMzMTI3NCw1MS4zMjYyODk5MzIwNjIyOSw2NTA0Ljg1MDQ1NzU5NTYzOSxuYW4sODIuMDA3MTk0NzY4MTM0NCwxLjIxODI5NDE2NzQ3Mzk1MTVlLTE3LDEuMjE4Mjk0MTY3NDczOTU1MmUtMTcsMy42Nzg1NTMxNTE2MjYxNjRlLTMyLDAuMCw2LjY4NTQ0NzcyMTcxNTQyN2UtNDcsNC44ODA0MDA5ODk3MzEyMDZlLTA1LDQuODgwNDAwOTg5NzM0NjAxM2UtMDUsMy4zOTU3NDA2NDQzMDA5NTZlLTE3LDAuMCwxLjU0NjYxNzM1NDU0MjI1MWUtMzUsLTMuNzM1NjcwNTY2NzQ0OTIzZS0xNywtMS44OTMzNDA2NDAzMzQ0MTc5ZS0xNiwtMS41MTk3NzM1ODM2NTk5MjU2ZS0xNiwwLjAsMS41NDcxODM2NzAzMTQ1NTZlLTM1DQotNS4wLDcwLjAsMC4wLDI3Ni4wLDIwOC4wLDAuMDAwMTQwOTc3MTQ3OTEzODAyMDYsLTkuNzAzODAzMTI5MzIzOTAxZS0wNSwtMC4wMDA3OTE5NzY0MzI4MzY3ODksMC4wLDAuODY2MDI2NjEzMzQ5MDUzMywyNC40NjQzMzczOTY5NzY5NSwwLjAsLTAuNDQ5Mjc2MzUxMTE0MTMwNjcsMC4wLDIwMzkuMCw4MC40NDQ4NTM4ODg0NTIxNCwwLjk3Mzc3ODU0MzAxOTE0MjQsMC4wMjMwNzA2OTA5ODEyMzk5MSwwLjAwMzE1MDc2NTk5OTYxNzQyNDIsNzAuMDA0NzY0NTQ3NTkzNTQsMzIuNzQzMzQxNDExMzY3NTYsMi45OTc4MzY3MjI5NTA1MTMsMTAuOTIyMzIzMTQwNzgxNjk2LDEuNzMxMDgzNTM5MjQ1MjE5NmUtMDYsMC4xMzI5NTk5MTE0OTE3MzQzMywwLjk5OTQwOTIxNzI1OTgzNDcsMzAuNTk4MTg3ODMzNTY2MTYsNi4yNzQzNDczMDY3MTE5MDgsNTEuMzYzOTc0NTcxODMwNyw2NTE2LjM2OTA4MjQwMzUzLDc4LjM4MjQ1NDM1NTA4NzM2LDgwLjc2MTEyMTIyNDkzNTUxLDkuMjk3MTE4OTMzNjQ5MDMyZS0xOCw5LjI5NzExODkzMzY0OTA4M2UtMTgsNC45OTIzMjIxMzQzNDk3OTRlLTMyLDAuMCw0Ljg1MzU4ODE5Mzk1NzY5NTVlLTQ1LDAuMDAwMTcxNTA1Mzg5ODI1NzE2MDIsMC4wMDAxNzE1MDUzODk4MjU3NDg5OCwzLjI5NTc4MDk0ODYyOTAwMzVlLTE3LDAuMCwxLjY3NzQ3NzA2NzQ2NTA0NDVlLTMzLC0zLjE0Nzc2OTc0NjY5NDE5MmUtMTcsLTEuODkzMzQwNjQwMzM0NDE3OWUtMTYsLTEuNTc4NTYzNjY1NjY0OTk4N2UtMTYsMC4wLDEuNjg2MDkwNjI5MjA2NjkxMmUtMzMNCjAuMCw3MC4wLDAuMCwyNzYuMCwyMDguMCwwLjAwMDIxMTc3NTQzMDQyOTg0NSw2LjM3MDMyOTU2MDQ0ODAyM2UtMDUsLTAuMDAwODU2MDkzMTM1ODQ0NzQwMywtMy4zMzA2NjkwNzM4NzU0Njk2ZS0xNiwwLjg0NzQzMzUxMzA4NzQyLDI0LjI2NDI0MzYwMjM4MDE2NiwwLjAsLTAuNzc0MjQ5NjE5MDY2MzcwMSwwLjAsMjI4MC4wLDc5Ljk3Mjc3NDE0NTY3NzI1LDAuOTc3NDMwMTI5MDMzMzM0NywwLjAxODcyNjUyNjc5MTE2OTA3NSwwLjAwMzg0MzM0NDE3NTQ5NjE0Myw3MC4wMDU2NzkwMjc0MTQxNSwzMi41NDc2ODA1OTA2OTA2MDQsMi45NzcwNzc5OTE3NDQ0Mjk3LDEwLjkzMjc2MDQ3MjE2MzIzMywwLjA2MTY3NzYyNjUzOTU0MzIsMC4xOTM1NzYxODg5OTI4NDgzNCwwLjk5OTM2MTU2NDIzMjA4NjMsMC4wLDAuMCw0OS44MTc2NzMyNTYwNDMwNyw2NTE2LjUwMTQ2NTA0MjIxMSw4OS42MzMxMjQxOTEwMzY3LDc3LjY1Mzg3NzE1ODAyMjQ1LDcuNTQ5NTMxNjc3NzkwMDI2ZS0xOCw3LjU0OTUzMTY3Nzc5MDA3M2UtMTgsNC41OTgxOTE0Mzk1MzI3MDVlLTMyLDAuMCw4LjMxNDU4NzQ0MTc4NzE5MmUtNDQsMC4wMDAyOTA4NjQ0NDQzOTA1MTIsMC4wMDAyOTA4NjQ0NDQzOTA1MzYsMi4zOTQ0MTAyODI0NTQxNzFlLTE3LDAuMCwzLjQzMTU5NjkzODk5MTE0ZS0zMiwtMi43OTE0OTYzMDEzMjI4Mjk1ZS0xNywtMy43ODY2ODEyODA2Njg4MzU3ZS0xNiwtMy41MDc1MzE2NTA1MzY1NTI2ZS0xNiwwLjAsMy40ODQ3NjIwNzg3ODkwNjFlLTMyDQo1LjAsNzAuMCwwLjAsMjc2LjAsMjA4LjAsMC4wMDAyMzI2MTc0NzU0OTIwMTc4NCwwLjAwMDI2MjU1NzE1MjYwOTE4NzcsMC4wMDAyNTY3MTA0MzgwMzY2Nzk1LC0zLjMzMDY2OTA3Mzg3NTQ2OTZlLTE2LDAuODY0NDY0NzQ1MTE5ODM0MSwyMy41Mzk3NTUwOTIzNTQxMDYsMC4wLC0wLjc0ODI3ODAzOTk5ODE1NiwwLjAsMjUxMy4wLDc5LjM1NzYzMzM0NTM4NzM4LDAuOTcyMzM0NTc2NDU2MTM4OSwwLjAyNDAxNTU0NTk3MDM2MDMxLDAuMDAzNjQ5ODc3NTczNTAwOTI0LDcwLjAwNzE4ODY4ODUwOTIyLDMyLjM3NDM4NzI2MjMzMDYxNSwyLjk0OTA1NzAyOTMzNDczMywxMC45Nzc4Nzc2NTM4NzI5MTYsMS43MjI3OTc1MzIxMDI0NTQxZS0wOSwwLjA1NjI0ODQ3NDkzNjY4MjkwNSwwLjk5OTM2NDg2MTA0ODY1NjIsLTMxLjIxNjE3NzgzNTg0NDQ5LC0xLjExOTcxNzc0MzEyODA1MjUsNTAuMDQ0OTUxODE0NTc2MDI0LDY0OTkuNjYxNDU2MDMzOTg1LDExNS42MTk0OTc5OTE0NDc3Myw3Ny4xOTYzOTIyNDQ4MDgwMSw0LjU4MDY0MjI4MTcxNzYyZS0xOCw0LjU4MDY0MjI4MTcxNzY1OTRlLTE4LDMuOTQxMzA2OTQ4MTcwODllLTMyLDAuMCw1LjY0MTY5NDk0MTY1MDg5NWUtNDMsMC4wMDAyODI2MDY2ODQzNjMwNzU1NywwLjAwMDI4MjYwNjY4NDM2MzAxNDQsLTYuMTE1NDUzMjg4OTcwNzg4ZS0xNywwLjAsMi4yMzQ3MDA2NTc1NDYzNDhlLTMxLC0xLjc5MjYxMjgxNTAxODUwNjVlLTE3LC0zLjc4NjY4MTI4MDY2ODgzNTdlLTE2LC0zLjYwNzQxOTk5OTE2Njk4MjdlLTE2LDAuMCwyLjI5ODY3OTMyMTc5OTk2OWUtMzENCjEwLjAsNzAuMCwwLjAsMjc2LjAsMjA4LjAsMC4wMDAyMDY5OTkxOTg3MjUzMDA5MiwwLjAwMDE0MjgxMzI2NDA0ODE1NDc4LDAuMDAxMTIxNjMwODU2NDY0OTM2MywtNS41NTExMTUxMjMxMjU3ODNlLTE2LDAuODQ3NjY2Mjg1MjQyODY0NSwyNC4xMDU3MzU3MzE5MTA3ODcsMC4wLC0wLjcwNTA2NTg0NzE4NzU1MzgsMC4wLDI3NDQuMCw3Ny45OTkyNTc1NTIwOTU2NCwwLjk3MTYxMjA3OTQwMTE1ODcsMC4wMjQ2MjEzNTY1NzI1MDA2MDMsMC4wMDM3NjY1NjQwMjYzNDA4NTgsNzAuMDA4NTg5OTk1MzA1MjksMzIuMjM3NDEyNjc2MjcyMzEsMi45OTk0NTU4NTQ5ODA0MDM2LDEwLjc0Nzc1MzY3MDk2ODA3Miw0LjA2MzY4OTc0MDcxMzYyNGUtMjMsNC41OTc0MzkwMDQzMDI4OTFlLTEzLDAuOTk5MzM2NTM1NTE2NDU1MywtNjIuNTcwODU0OTQ4MjA0NCwyLjk3ODIzODQ1NjYwNTAxMjYsNDkuNDY3MjMzNTA3NDI1ODgsNjUwNS4yNTA4OTE2MzI3OTEsbmFuLDc3Ljk5OTI1NzU0OTE4NzE5LDEuODA0Nzk4MjYxMzkxNzY3ZS0xOCwxLjgwNDc5ODI2MTM5MTgwMzVlLTE4LDMuNjQ1NzA4OTI3MDU4MDczZS0zMiwtNy4zNzcxOTIwODA2NjA0NDhlLTEyMywyLjEwMzM2MTU4NjE0OTcyMjdlLTQyLDAuMDAwMjU1OTI1OTcyODM5NzYwMzYsMC4wMDAyNTU5MjU5NzI4Mzk1ODc4LC0xLjcyNTU0NzAyNDAxNTcxODNlLTE2LC03LjM3NzE5MjA4MDY2MDQ0OGUtMTIzLDYuMzAxMDczNTU0NzEwMzAxNWUtMzEsLTcuNDU0MTkxNzgyNTM5OTdlLTE4LC0zLjc4NjY4MTI4MDY2ODgzNTdlLTE2LC0zLjcxMjEzOTM2Mjg0MzQyOWUtMTYsLTcuMzc3MTkyMDgwNjYwNDQ4ZS0xMjMsNi41MzE2NTI4OTg1NzgzNzhlLTMxDQoyMC4wLDcwLjAsMC4wLDI3Ni4wLDIwOC4wLDguMzc1NjcwODk4ODM2Nzc3ZS0wNiwwLjAwMDIxODI1NjQ5NzMxMTU3MjI3LDAuMDAzNzg1NTgzOTMzNDkzMTMxLC0zLjMzMDY2OTA3Mzg3NTQ2OTZlLTE2LDAuODgwNzI4NTAwNjc5NjUxMSwyMy42MjE4MDc2MDUxNTEwNywwLjAsLTAuNjM1OTAxNDU4NzY0NTM2NiwwLjAsMzIxOC4wLDc2LjM1NzQxMDkxNDI4MzE0LDAuOTc2NDIwNzc4MzEzMjk4OSwwLjAxOTEwNDIyOTA2OTU4OTk4NywwLjAwNDQ3NDk5MjYxNzExMTE0MSw3MC4wMDk1NDI0NTk3NDM4NywzMi4xOTMyNjYwNjI3OTczNTYsMy40MzA2NjY5NDYzNDMzMTg0LDkuMzgzOTY3MTg5NTYxMDY2LDEuMjk2MjA1OTU1MTczNjMyMmUtNTYsMy4wMzMxMjU0ODY3ODg5MDA3ZS00NywwLjk5OTIyMTk2Mzg2MDYyNTUsLTEyMy43ODM1NDAxMzM4OTgyMSwyNi42NzU0NzUyMTExODUzNzIsNDQuOTI2OTU2NzQ1MDc5NjIsNjQ2NC4xODU5MTcwMzkxMzIsbmFuLDc2LjM1NzQxMDkxNDI4MzEyLDguOTAyMDQ3ODk4NTIyOTVlLTE1LDguOTAyMDQ3ODk4NTIyOTUzZS0xNSw0LjAzNTg5ODMxNDkyNjk5MTVlLTMwLC00LjU1NDc4NzQwNjA3MzI1MmUtMTA0LDEuMjQ4NTI1NDM0OTQwODM1NGUtNDEsMC4wMDAyNDY4MDA0Nzc5Njk1OTI3LDAuMDAwMjQ2ODAwNDc3OTY5NDQxNCwtMS41MTMzNzgyMzY0MzkxODA2ZS0xNiwtNC41NTQ3ODc0MDYwNzMyNTJlLTEwNCwxLjI3MjgwNTkwNzE4NDI5MjZlLTMwLC0yLjAwMzQwMjg0MzMzMDQ1MDdlLTEzLC0yLjAwODgzNDQxOTM5NDgxNzRlLTEzLC01LjQzMTU3NjA2NDM2Njc5OWUtMTYsLTQuNTU0Nzg3NDA2MDczMjUyZS0xMDQsMS4yOTM0NTg0OTk1MDkwNzk2ZS0zMA0KMzAuMCw3MC4wLDAuMCwyNzYuMCwyMDguMCwwLjAwMDUxMTQwMDkzMzg1MDg0MzQsMC4wMDA0NTAzNjAzMDg1Njk2MTU3MywwLjAwOTI4NjcyNjczNTMyMTcxMiwtNy43NzE1NjExNzIzNzYwOTZlLTE2LDAuODIzMTUyODMyMjA5NTQyLDI0LjIwNjYwMzY5MDI1NzEsMC4wLC0wLjcwODIwODY0NDc0Mzg4MywwLjAsMzY4MS4wLDc3LjY3NjYwMTU3NTY4ODQ0LDAuOTc2ODA3MTk4NTAzMDA0LDAuMDE5MTgzNzk0MjExNDU2MzE1LDAuMDA0MDA5MDA3Mjg1NTM5NzcxLDcwLjAxMDk1MzYwMDI5Mzk5LDMyLjU4MTA2NjYyMjUwMDA2LDQuMTg3NTI0NzExMzE0MjQyLDcuNzgwNTA3MzA4ODgwOTUyLDkuNDA2NjkwOTIzNDQ2NjgyZS02Nyw1LjU4NTM4MjA0OTYwNDQ2MmUtNjMsMC4wMTEwODI2NjE3MTUyMjY3NTcsLTE3OS45MjcxNjcyOTA2OTE3OCw2OS45ODAxMTA0Njg4NDM2Miw0OS4zNzA5NzkzMzkyMTkyNzUsNjQ5NC41NTQ5NjQwNTk2NjUsbmFuLDc3LjY3NjYwMTU3NTY4ODQyLDYuNDUzODQxNjAwNjcwMDEyZS0wNiw2LjQ1Mzg0MTYwMDY3MDAxMmUtMDYsMC4wLC0xLjMyOTI0OTA2MTcwODg1ZS05OSw1LjE3ODk4NzE5MjYyMDc2NGUtNDEsMC4wMDAyNzQ3NDMxNzE2OTczMDc4MywwLjAwMDI3NDc0MzE3MTY5NzA3NTYsLTIuMzIyMzAwNjI5MTYwMTg0NGUtMTYsLTEuMzI5MjQ5MDYxNzA4ODVlLTk5LDEuMjg3NTcyMzQ5NTI5MzkzZS0zMCwtNi4xMzE1Nzc4NzM1NjQ5MzE2ZS0wNiwtNi4xMzE1Nzc4NzM4ODczMzRlLTA2LC0zLjIyNDAyNTE5ODg4MTgzMjdlLTE2LC0xLjMyOTI0OTA2MTcwODg1ZS05OSwxLjE5MDAyNDU1NzIxNTY4NzVlLTMwDQo0MC4wLDcwLjAsMC4wLDI3Ni4wLDIwOC4wLDAuMDAwNTY3NTUxMTQyMDEyMTU4MywwLjAwMDYxMjY0MDg1Njk3ODU4NzcsMC4wMTM0NTM4ODU2NTYxMTY1OSwtNy43NzE1NjExNzIzNzYwOTZlLTE2LDAuODc2NzQzNTczODYyODQ4NiwyMy43ODEyODA2MDU5MTIyOSwwLjAsLTAuNzg1NTMyNjkwMjE1NTcxOCwwLjAsNDE0NC4wLDc5LjE3MzQ1ODI4NTc2ODMsMC45NzI0ODQ1MTQ5NjA5MzQxLDAuMDIzNTAzMzg3ODc2Nzg0NTEsMC4wMDQwMTIwOTcxNjIyODE1NzgsNzAuMDA5MDY1NzIwOTAxOTcsMzMuNDM5ODM3OTMyOTg1OTk1LDUuMDk2Nzg3ODc4OTU1NzA1NSw2LjU2MDk2MzI0MzM1MTEzOCw4LjkyMjE4OTYyMDI2NDAyNGUtNzIsMy4yMjI3MDAyMDYzNzA0NDVlLTY4LDQuMDM4NTQ1MjYwMjcyNzE5ZS0yMSwtMjI3LjYyNDMwNDc0MTY2NTc4LDEzMC41MzA3MTQwNjQ2ODMzLDUxLjM3NDc2MzE1MjMwMDEsNjQ2Ny4zMDc3MjU3NjkxNDksbmFuLDc5LjE3MzQ1ODI4NTc2ODI5LDYuODUzMDE1MDkwMDA1MjU1ZS0wNSw2Ljg1MzAxNTA5MDAwNTI2ZS0wNSwzLjQ2NjgxMDI1NDUxODU4NzRlLTIwLC04LjEwNTE1NzUwMDg2OTU4M2UtOTksMS45ODEzMTcwNzQ4ODQyODY2ZS00MCw5LjMyMTIzMTA1NTQyMDI5M2UtMDUsOS4zMjEyMzEwNTU0MTMyNzdlLTA1LC03LjAxNTY2ODM1MTcyNzQ0OGUtMTcsLTguMTA1MTU3NTAwODY5NTgzZS05OSwxLjExNzAwMjk0NjI3MDIwODRlLTMwLC02LjA2ODUwODg4OTc3NjUwNDRlLTA1LC02LjA2ODUwODg4OTg1MDU1M2UtMDUsLTcuNDA0ODc1NTgyOTY4MDY4ZS0xNiwtOC4xMDUxNTc1MDA4Njk1ODNlLTk5LDguMTgxNjQ4NTE4ODYxMzY3ZS0zMQ0KNTAuMCw3MC4wLDAsMjc2LDIwOCwtNC44OTUyNzI2Njk5NDc2MTFlLTA1LDAuMDAwMzYxNjY0Mjg1NTg0MDgxNSwwLjAxOTc1Nzg1Nzc2NzQ3ODMxLC01LjU1MTExNTEyMzEyNTc4M2UtMTYsMC44Mzg0NTQxMjA0ODA0OTUsMjMuMjE2MTEyMDYxMjM5NzYsMS41MzMyODcwMjk2ODM5MTg4ZS0yOTYsLTAuNjY5ODA1MTI4NDIxMTQ1NSwwLDQ2MDksNzguODE2ODIzMDI1NzY5NDUsMC45NTkzMTgwNTE5NzE3MzA2LDAuMDM3MTg3Mzg5Njg3NTY2NTgsMC4wMDM0OTQ1NTgzNDA3MDI4NTIsNzAuMDA0ODc1Mzg3Mzk1NCwzNC43Nzc3OTUyMzQ0NjcwNjQsNS44NDEyMTc3NTI1MzE0NzEsNS45NTM4NjA0MjkwODUxOTEsOS43OTc4NjU2MTA2MDI2OGUtNzQsNC44MDg5NDc1NzY2MTI0OTllLTcxLDQuNzMzMDE2NDg5ODI0MzAzZS01NywtMjY0LjAyNTE5NzEwNjg4NzQsMjA0Ljg0Mjg2Nzg3MTA2NzQsNTIuOTA3MzE1NDgyNjk0NjQsNjUxOS44OTk5NDg5NzIwMjMsbmFuLDc4LjgxNjgyMzAyNTc2OTQ0LDAuMDAxMjE4Njc0MjEwMDMzMjAxMiwwLjAwMTIxODY3NDIxMDAzMzIwMTksNS41NDY4OTY0MDcyMjk3NGUtMTksOC42NTY2MzM3NTExMDU4MTllLTk5LDguMDkzNzEwMDE1MTYyNzY4ZS00MCwtMy40NTAzMTU3OTgxNzUzMTVlLTA1LC0zLjQ1MDMxNTc5ODE5MDE5MDVlLTA1LC0xLjQ4NzU1MDUwMDA0MzAxN2UtMTYsOC42NTY2MzM3NTExMDU4MTllLTk5LDUuMTQ4Mzg5MzE5MDg1MjhlLTMxLC04Ljc3OTM4MDE1OTkwNjYzN2UtMDUsLTguNzc5MzgwMTU5OTU4MDJlLTA1LC01LjEzODI3NTAzODU2MzgxNmUtMTYsOC42NTY2MzM3NTExMDU4MTllLTk5LC0xLjAwNDkxNzY4MzI4MDc0MzVlLTMxDQo=', 'original_DM1.csv': 'dF9NeXIscGhhc2VfcGMsRE0sbngsbnksZGVsdGFfSix0YWdfbWVhbl92eF9rbXMsdGFnX2NlbnRlcl94X3BjLHRhZ19tYXNzX3JlbF9jaGFuZ2UscmhvX21pbixwX21pbix0YWdfbWluLHRhZ19leGNlc3NfbWF4LGZhbGxiYWNrX2ZhY2VzLHN0ZXBfY291bnQsdGFnX21lYW5fVF9LLHRhZ19UX2x0MTAwLHRhZ19UXzEwMF8xMDAwLHRhZ19UX2dlMTAwMCx0YWdfY2VudGVyX3lfcGMsdGFnX3Jtc19tYWpvcl9wYyx0YWdfcm1zX21pbm9yX3BjLHRhZ19hc3BlY3QsdGFnX2luc2lkZV8yMHBjLHRhZ19pbnNpZGVfNDBwYyx0YWdfaW5zaWRlX3J0LGRtX3hfcGMsZG1feV9wYyxUX21pbl9LLFRfbWF4X0ssdGFnX1RfbmVhcl9LLHRhZ19UX2Zhcl9LLGN1bV9pbl94LTQwLG1hc3NfaW5jcl94LTQwLGJ1ZGdldF9yZXNfeC00MCxsZWZ0X2ZsdXhfeC00MCx5X2xvc3NfeC00MCxjdW1faW5feDAsbWFzc19pbmNyX3gwLGJ1ZGdldF9yZXNfeDAsbGVmdF9mbHV4X3gwLHlfbG9zc194MCxjdW1faW5feDQwLG1hc3NfaW5jcl94NDAsYnVkZ2V0X3Jlc194NDAsbGVmdF9mbHV4X3g0MCx5X2xvc3NfeDQwDQotNTAuMCw3MC4wLDEuMCwyNzYuMCwyMDguMCwtNS45MTY2ODk1MDEwNDUwNTZlLTE4LDAuMCwwLjAsMC4wLDEuMCw0Mi4xOTgwNzUzMTAxMzUxNiwwLjAsLTEuMCwwLjAsMC4wLDE5NC41ODM0NTUxMjY3NjE2NiwwLjQ4NDQyMDY3NTM0NTIyNzM3LDAuNDkxMjY3MzA4ODQ0MzI4MiwwLjAyNDMxMjAxNTgxMDQ0NDQ0Nyw3MC4wMDA2Mjk3ODMxNzE3OSwzNS4wMDk1MjAzNTg4NzEwNCwzLjAxMzg1Njg4NjY3MDg1NDYsMTEuNjE2MTg1Mzk4MDg3MzcsMC4wLDAuMCwwLjAsMjA4LjU5NTMzNTMyNTI2Nzg4LDI0OC41NTcwNjA4OTMwMDk0NCw1My4yNzA1MTQ3MzI1MjM3OSw2MzUzLjk0Nzk2MzA4NDQzNixuYW4sMTk0LjU4MzQ1NTEyNjc2MTY2LC0wLjAsMC4wLDAuMCwwLjAsMC4wLC0wLjAsMC4wLDAuMCwwLjAsMC4wLC0wLjAsMC4wLDAuMCwwLjAsMC4wDQotNDAuMCw3MC4wLDEuMCwyNzYuMCwyMDguMCwwLjMwMTM0OTA2MDU2MDc0NTcsMC40OTUzOTg2ODU0OTMwNDExNywyLjE2MzQ0MjQyMTc1NzE3NywwLjAsMC45NjgzOTYxOTc4MTQwMTcsMjQuMzQ0MjkyNjk4MjI1NDM4LDAuMCwtMC4yNjQ4Mjg3NzUyMzI0ODc2LDAuMCw0NDcuMCw3My43NzIyMTI4MDM3MTExNywwLjk3MzI0NDU1MTQ0Nzg0NDksMC4wMjI2ODE5MDk2MDE4NzEzMzgsMC4wMDQwNzM1Mzg5NTAyODM3MzQsNzEuMjU1NzI4Mjk4Nzc1OCwzNC41NjQ1MjgwODcxOTk4NTQsMi4xNDYxNzg3Njc3ODQ5NzM3LDE2LjEwNTE0ODY1MTE4NzcwNCwxLjQ3ODMxOTcxMTk0ODQzNzhlLTgwLDIuNjIxMTg4NDk0MDk0MDkxN2UtNzYsMi42NTg2Mjg0OTAwMDQ5NDdlLTA3LDE5MC42ODE1NjUyMjUzNTQxOCwxNzMuNjQ1OTUzNzIyNzg1NywzOC4xMDUxMjk5MTg0NzY4Nyw2MzczLjU0ODUxMTk1MTQ3OCxuYW4sNzMuNzcyMjEyODAzNzExMTcsMy44NDYyNjU3MzcwNjc3NzZlLTIxLDMuODQ2MjY1NzM3MDY3Nzc2ZS0yMSw2LjQxNDg4NzYxMDk1NTIyNWUtMzcsMC4wLDEuMjgyODI2ODQwNzA3MjEwNWUtODksLTAuMzE5ODkyODk4OTM0MDkwNiwtMC4zMTk4OTI4OTg5MzQwOTA0NSwxLjQyMDAwNTQ4MDI1MDgxMzRlLTE2LDAuMCwxLjU5NzI2OTc4NjI2NjUwNjZlLTUxLC0xLjc2NDAxODM2MjU2NjU4MjFlLTE1LC0xLjg5MzM0MDY0MDMzNDQxOGUtMTUsLTEuMjkzMjIyNzc3Njc4MzU2NGUtMTYsMC4wLDEuNTk3MjY5Nzg2MjY1MDIyNmUtNTENCi0zMC4wLDcwLjAsMS4wLDI3Ni4wLDIwOC4wLDAuNjE1MjM2NTMzODIxMDg3OSwxLjQ5NzM5NjQ1NTY1NTgwNCwxMS41NjI4ODY5NjM4MjI4NTUsMC4wLDAuODExMTgzNzA2Mjc1NTA2NCwyMy42OTcxNDI4MTQ5NDIxNiwwLjAsLTAuNjg2Nzk5MjE1NTgwMjQwNiwwLjAsOTc0LjAsOTMuOTY3Mjk3OTQ5MTY4MzcsMC45MzI0ODM4MDcwNjQ2MDA3LDAuMDU5MTUyMTYyMDc5MDc4NDYsMC4wMDgzNjQwMzA4NTYzMjEwNDMsNzMuODc5MTc2OTUxMDAyMjEsMzIuOTc0OTYzMzEwMjU0NTIsMS45OTY5NDI3MDUwNTA1NTU3LDE2LjUxMjcyMzc4ODY0NzU2MywxLjI1NDI1MTYxMzU1NDI4NDNlLTQxLDUuNDQ1MzA4MDA0ODQ1ODYyZS0zNywwLjkyNTAxMDk3NzY2MjUxMDYsMTU4LjQ5Nzk1MjkyMTUyNDEzLDEwNy40NDA1NDg2NzMxNzE3Miw0My4yMTE1MTMzMzQ5MjkwNjYsNjYxNi4wMjMwMDU4MzU3OTUsbmFuLDkzLjk2NzI5Nzk0OTE2ODM1LDguNTM2NTcyMTkyNjI5OTA1ZS0yMSw4LjUzNjU3MjE5MjYyOTg3NWUtMjEsLTMuMDc5MTQ2MDUzMjU4NTA4ZS0zNSwwLjAsOS4yMDU2NzQyNjY2Mjg1N2UtNzgsLTAuNDk5ODI3MjAxOTEzMjgzMzQsLTAuNDk5ODI3MjAxOTEzMjgyOTUsMy43ODY2ODEyODA2Njg4MzU3ZS0xNiwwLjAsLTMuMjkyMzMwNjMzMzM1NjI3NGUtNTAsLTEuNzc2NDAyMzE2MzM4NzQ0MmUtMDcsLTEuNzc2NDAyMzE2MjA1NzI4MmUtMDcsMS4zMzAxNjA0NzQzNDA5N2UtMTcsMC4wLC0zLjI5MjMzMDYzNTIzMzM3NzRlLTUwDQotMjAuMCw3MC4wLDEuMCwyNzYuMCwyMDguMCwwLjU4MDU1ODI3NjkxMjU5MDcsMy40NTY2MTQ3Mjk0NDg4Nzg3LDM1Ljg5NzI4MTQxNTI0MDExNiwyLjIyMDQ0NjA0OTI1MDMxM2UtMTYsMC43Nzk1NDMwNzkzNTM4MzE1LDIyLjk2MDkxNTIwNjQzNzY4LDAuMCwtMC42NDYzMzkzNjY3Njc5MDA5LDcyLjAsMTU0MS4wLDExNi4xMzg1MzYyNjQ3Mjk2NCwwLjkwNzY3MTI0MTMxMzU3NDgsMC4wODEzOTEzNjI4NzI0MDE4LDAuMDEwOTM3Mzk1ODE0MDIzMTY0LDczLjIwMzI2ODgwMTE2NDU4LDI4LjQ2NzYxNDMzNTY2OTE5Nyw0LjI2MTI2NjQyNjM4OTM5Miw2LjY4MDU1MjU2MDQ3MjA0MSwxLjk1MDM0ODk0MjMyNzg5NTNlLTIyLDguMzgxMDgyMzExNDYxMjg1ZS0xNywwLjk5OTk3ODAwMDU1NjIyOTQsMTE0LjA1MTk5OTA0NTY1ODE1LDU0LjIzMjExMjg2ODg0NTYzLDUzLjc3Mzc5OTkwMDQ2NzAyNiw2OTA1Ljc1MjUzNTM1NzU1NCxuYW4sMTE2LjEzODUzNjI2NDcyOTExLDguODI4NzA4MjczNzQyNjY2ZS0yNiw4LjgyODcwODI3MDkwMzc5OWUtMjYsLTIuODM4ODY3NzE0NDIwMDY0NmUtMzUsMC4wLDIuMzEzMjUyMDk5NTc1NTg5ZS03NiwtMC40OTk5OTgyNjk3NjE4ODEyLC0wLjQ5OTk5ODI2OTc2MTg4MTQ0LC0xLjg5MzM0MDY0MDMzNDQxNzllLTE2LDAuMCwtMS4yNzQ0NDgzNzMxMzQ3MzA3ZS00OSwtMC4yNTg0NTA5NzE4NzEyMzIyLC0wLjI1ODQ1MDk3MTg3MTIzMjEsOS40NjY3MDMyMDE2NzIwODllLTE3LDAuMCwtMS4yNzQ0NDgzNzU4ODM5MTk2ZS00OQ0KLTEwLjAsNzAuMCwxLjAsMjc2LjAsMjA4LjAsLTIuMjk0MjIwMTUyMDUxOTM2LDQuMzU3NzY3NDg3NjY0MzIxLDgyLjgwMDQ5OTAzMjIyNzc4LDIuMjIwNDQ2MDQ5MjUwMzEzZS0xNiwwLjc4NzExNTA1MTc2OTk2MjQsMjEuMjAwMjU5MTI1MTMxODA0LDAuMCwtMC43MzAzODUyMjUyMzEyOTksNjk2LjAsMjIwMS4wLDk2Ljg4MDc3OTQ5NjkyNTMxLDAuOTE4OTYzNTE1NDY1MDUzNSwwLjA3MzEwOTM2OTY4NDQ4NTY2LDAuMDA3OTI3MTE0ODUwNDYxMDI3LDQ5LjY1NjkwNzUyNzk4MjEsMTcuMzY0MjI1MjQwNDUzMTQ1LDYuNzU5OTk1NDUzOTQ4NzAyLDIuNTY4Njc0MTAwMjYyOTI1LDAuMDYxNjk0Nzc0Mjg2NDQwNzEsMC40ODA5NTU2OTY4Mzk3MTYyLDAuOTk5OTk2MDQ0MzI3ODQsNjAuMTA2NzIxNTUxNjEwMzM1LDE3LjU2MTUwODAyMDMzMTI3NCw0Mi45NTQyOTI1ODczOTgwODYsNzY4My42MzQ2MzQ4MjE2Myw4NS42OTUzNTUyNjAwNTQsMTA3LjI0NTM5MjgzNjM1OTMsMy4wOTc1NzU0NTkyNDQyNTQ1ZS0zNSwyLjU4NzAyMzI5NTgyMTA5MzRlLTM2LC0yLjgzODg3MzEyOTY2MjE0NWUtMzUsMC4wLC03LjM2NTk5NjUyMjc5NDMxN2UtNzAsLTAuNDk5OTk5OTk5OTk5NDExODYsLTAuNDk5OTk5OTk5OTk5NDExNjQsMS44OTMzNDA2NDAzMzQ0MTc5ZS0xNiwwLjAsLTIuNjYzNDg5NzIzMDM5ODQ4NGUtNDksLTAuOTk5NzY3NTEyNzgyODc5MSwtMC45OTk3Njc1MTI3ODI4Nzg5LDEuODkzMzQwNjQwMzM0NDE3OWUtMTYsMC4wLC0yLjY2MzQ4OTcyNjcwODc4NjZlLTQ5DQotNS4wLDcwLjAsMS4wLDI3Ni4wLDIwOC4wLC0zLjExNzI5NzQ2ODM2NjcwNzQsMC4zNDEwMTkzNzk5NjQ0NzIzLDkzLjkyMjQzMzkwMDc2ODc4LDIuMjIwNDQ2MDQ5MjUwMzEzZS0xNiwwLjc4OTEyMDk2NDc4NDg1NjYsMjIuNTI3NjY5ODk5MjMwOTY1LDAuMCwtMC43ODkxMjA5NjQ3ODQ4NTY2LDYwNDYuMCwyNTUyLjAsNzcuNDY5MDk4NTQ2Mzg5MjYsMC45MTM1OTkzNjY2MjUyMTk2LDAuMDc5MDMwMDIzNjE4NTA3NDIsMC4wMDczNzA2MDk3NTYyNzI4ODA0LDIwLjE0MzAwMzEzMjU2NTk1LDEzLjg2NjM0MjEwMjMyMDc5Nyw3LjAwNDk1NTQ5NzczNjE4OSwxLjk3OTUwNDY2NzM0NjE0MTQsMC4wMDMxMjA2Mzc3OTI0NDQ1NzgsMC4wMTMwMjg1OTc5OTMyMzM0MiwwLjk5OTk5Njk1MjE4OTQ1MTEsMzAuNTk4MTg3ODMzNTY2MTYsNi4yNzQzNDczMDY3MTE5MDgsMjYuMDk4NzIyMDI2MDkxMDU3LDgxNTYuNDkxMDExNzI0ODQ4LDMwOC44NzE4MjEzNTYzMzcyLDc0LjQxNDQ0NzY3ODE4NzM0LDIuODM4ODczMTMyOTA4Nzk3M2UtMzUsMy4yNDY2NDc5NDMzNzk1Njg1ZS00NCwtMi44Mzg4NzMxMjk2NjIxNDk0ZS0zNSwwLjAsLTEuNDYzNTM0NzAxNDYzNTk5MmUtNjQsLTAuNDk5OTk5OTk5OTk5OTk5NiwtMC41LC0zLjc4NjY4MTI4MDY2ODgzNTdlLTE2LDAuMCwtMy40MDg2MDcxOTE4ODQ3MDZlLTQ5LC0wLjk5ODM4Njc5MDIwNTM3NTUsLTAuOTk4Mzg2NzkwMjA1Mzc0NSw5LjQ2NjcwMzIwMTY3MjA5ZS0xNiwwLjAsLTMuNDA4NjA3MTk1NTY0NTgxNmUtNDkNCjAuMCw3MC4wLDEsMjc2LDIwOCwtMi45Nzk0ODA3MTI5Njg3OTQ0LC0xLjkwNTczNDQ2NDM1NjE5Niw4OS42ODU1NjM2MzQ3NzMsMi4yMjA0NDYwNDkyNTAzMTNlLTE2LDAuNzc3MjkyMTkxNDg3MzU0OCwyMC4zMjI3MzQ1NDc4OTM0MSwwLjAsLTAuNzEyMDc0MDk2ODA0MDQzOCw2MDkyLDI5MDgsNzkuMDE0NDI0OTMyNzEwNTcsMC45MTQyNTAzNzI3ODkyNzc3LDAuMDc4MDM1OTM3NjM4MDgzMTQsMC4wMDc3MTM2ODk1NzI2Mzg5MDQsLTEwLjU3MTk5Njg0MTk0MjU0MywxNC4zNzQyMzI3MTkwOTM2NjIsNy4wNzI1MTg1MjgxOTMzMiwyLjAzMjQwNjUxMjg2OTk3MzUsMC4wMDY5NTUyNTYwNDMyMjMzNDgsMC4wMTE4MjcyMjk2MTczMjg3NjYsMC45OTk5OTgzNTc2MDE2OTY5LDAuMCwwLjAsMjcuOTY5NzQ2MzkxNzkyOTk1LDc4NDkuODE2NjI5MzA5NjM3LDE5My45NjY1NzY5MzU1NjQ1Miw3Ny42Mzg1ODcwODU4MzU1NSwyLjgzODg3MzEzMDM4NDQ2NDVlLTM1LDcuMjIzMTQxMzQxOTgzNDAzZS00NSwtMi44Mzg4NzMxMjk2NjIxNTFlLTM1LDAuMCwtMS44OTQzNzAzODUxNDQ4NDQzZS02MCwtMC40OTYyODEzMTI0MDE2ODc1NSwtMC40OTYyODEzMTI0MDE2ODgyLC02LjYyNjY5MjI0MTE3MDQ2MjVlLTE2LDAuMCwtNC4xMTgzMzgyNDQwMzUzMDdlLTQ5LC0wLjk4NzE3NjM3NTgxMjY2MzYsLTAuOTg3MTc2Mzc1ODEyNjYyMywxLjMyNTMzODQ0ODIzNDA5MjVlLTE1LDAuMCwtNC4xMTgzMzgyNDc3MTU0MTg2ZS00OQ0KNS4wLDcwLjAsMSwyNzYsMjA4LC0yLjUxNjEzODc0ODE3NDgyNjMsLTMuNTM1NDMzMTAxMjI0NTcxLDc1LjcyNjA2ODMyNjk2NTA4LC0yLjIyMDQ0NjA0OTI1MDMxM2UtMTYsMC40NzEyNzk5NzA3NDc4NDQ5NSwyMS4yODQxMjY0MzAxNTkwMjcsMC4wLC0wLjI3MzUzMzUwNjEwMzg4NjA1LDYyMzAsMzMyNSw4OS45MzQ1ODAwNzEyMDI3MiwwLjkyNTUzNjg3NzUxMDU2MDksMC4wNjIyNzU0NjMxNTE4OTMwNiwwLjAxMjE4NzY1OTMzNzU0NjA4NSwtMzguMzQzMjk1NTU5MjMyNDE2LDE4LjY1NDk5NjQzMDUwMTgxLDUuMjUzNTIwODMxMzk5OTQ2LDMuNTUwOTUxMjYyODE4Mjg1OCwwLjAwODk0MzI5MTMwNDQ2ODc2MywwLjAxMDk1MzU3NTQxOTg4OTAwOSwwLjk5OTk5OTIwODg1OTMyNDMsLTMxLjIxNjE3NzgzNTg0NDQ5LC0xLjExOTcxNzc0MzEyODA1MjUsMzAuMzI0MTEwNDkyNDAxNjk0LDc0MTguMzE4MDc1OTk2NjA1LDE5Ni44MTEyNjg0OTUwMjczLDg4Ljc1MDkzMzAzNjg2MzE5LDAuMDAwODcwNzE4MjYyMjExNjQ3NSwwLjAwMDg3MDcxODI2MjIxMTY0NzUsMC4wLDAuMCwtNS4yMjczNzM4NTQ1MzQ4ODRlLTU3LC0wLjQ5MDM5NTkzMzk4OTQ3OTIsLTAuNDkwMzk1OTMzOTg5NDc4NTYsNi42MjY2OTIyNDExNzA0NjI1ZS0xNiwwLjAsLTQuNzgxNzMyNDkyMzY4MDc1NmUtNDksLTAuOTY5MDEzMjY4NDA0MzEyMywtMC45NjkwMTMyNjg0MDQzMTEsMS4zMjUzMzg0NDgyMzQwOTI1ZS0xNSwwLjAsLTQuNzgxNzMyNDk2MDQ4MTg5ZS00OQ0KMTAuMCw3MC4wLDEsMjc2LDIwOCwtMS45NTU2NjIzMDcwOTc0OTIxLC00LjcxOTA4MDMzODUyMTA2Myw1NC41MTM5Mzk5NDQ5ODY4MSwwLjAsMC40MjY5ODEwODYxMTkyMjQyLDE5LjYxMzgxODE2NDg5MjY4OCwwLjAsLTAuMTYzNTcwMzA0MjY4ODI0NCw2NDIzLDM3NjYsMTA3LjA0MzQ1NjcyMTYxOTMsMC45MjUxNTA2MTk5MTg3NzkxLDAuMDYxMjEwODE2MzIxMTQ5MDcsMC4wMTM2Mzg1NjM3NjAwNzE5MDYsLTYwLjU2NjkwNTU2NDgwNjU2LDIyLjEzNzkzODAyNDgwNDA1LDUuMDk4MDQ1MTUyNTg4MDU3LDQuMzQyNDM2NjMyNTEzMDY4LDAuMDA4OTg1ODQ4MDM3NTcxNDcxLDAuMDA5NDQwNTYxNjIzMjE5MjA2LDAuOTk5OTk5NjM3Njk1MDQzNCwtNjIuNTcwODU0OTQ4MjA0NCwyLjk3ODIzODQ1NjYwNTAxMjYsMzAuMDI3MzQzMzcyMjU3NzE4LDc1MDEuNTk4MDcwMDcyMjY5LDg5LjA2MDQ1NTQ3NjEyMzk1LDEwNy4yMTQ4NDQzNDg1MDA4LDAuMDA4OTkwMDc3Nzc4MTAwNDE3LDAuMDA4OTkwMDc3Nzc4MTAwNDI5LDEuMTgzMzM3OTAwMjA5MDExMmUtMTcsMC4wLC00LjM4ODY0ODYxMTE0MTM0MzNlLTU0LC0wLjQ4MDcyNjY0ODU0NzgxMTMzLC0wLjQ4MDcyNjY0ODU0NzgxMSwzLjc4NjY4MTI4MDY2ODgzNTdlLTE2LDAuMCwtNS40MTcwNjA2NzY1MDQzNjNlLTQ5LC0wLjg1NzEwMDQ5NTc5NjAzNTIsLTAuODU3MTAwNDk1Nzk2MDMxNCwzLjc4NjY4MTI4MDY2ODgzNmUtMTUsMC4wLC01LjQxNzA2MDY4MDE4NDQ3NmUtNDkNCjIwLjAsNzAuMCwxLDI3NiwyMDgsLTAuODA0MzkzNjQ2MzM1NzQ2MSwtNi4wMjkyODAzMTQ4MTQyMDMsLTEuOTgwOTg3NTcxMDc1NDMyLDQuNDQwODkyMDk4NTAwNjI2ZS0xNiwwLjM4NzI1MzUyMTM0NDExOTMsMTguMTAxNzIzMTM1NDcyNDE1LDAuMCwtMC4xNDEyMTc5NTE3NzI5Njk5Nyw2NDQwLDQ2MTgsMTY2LjEzMTMxODU1ODI3NzEsMC45MTQyODQwOTg5NTQ3NywwLjA2Mzc3NDIxMDM5NDgzNjE4LDAuMDIxOTQxNjkwNjUwMzkzNTkzLC04My43MjgzODQ5MDQyNzk3OSwzMC4yNTg4OTkxNzk5MjIxOTYsNS44NzM1ODE5NjA5ODA2NzcsNS4xNTE2OTQzODAxODE5NDQsMC4wMDgyMDkzODA3NDExODI2NjEsMC4wMDg5NzY5OTc1ODY1MDAyNTIsMC40NjgwNzQ3MjQzNDUyMzM3LC0xMjMuNzgzNTQwMTMzODk4MjEsMjYuNjc1NDc1MjExMTg1MzcyLDM3LjcxODExOTAyNDgwNjYyLDc0NjUuNTMzODQ5ODcwMjk5LDE4OC43NDg0Njc3ODg2NjA5MiwxNjUuOTI2NDQ1MzE3NjM1NjUsMC4wNzE2OTIxNjg3MzMxNjgzNSwwLjA3MTY5MjE2ODczMzE2ODM5LDMuNTUwMDEzNzAwNjI3MDMzNWUtMTcsMC4wLC0yLjMzNzAwOTYzNTQ2NTY0OTZlLTUwLC0wLjA4Mjc5NjMyOTk0ODM3NzIzLC0wLjA4Mjc5NjMyOTk0ODM3NjMyLDguOTkzMzY4MDQxNTg4NDg0ZS0xNiwwLjAsLTYuNjE0MjE1ODIwODMyMTU2ZS00OSwtMC4wMDEyNDk2NzQ1OTcyMDQ1NjQzLC0wLjAwMTI0OTY3NDU5NzIwMDUyNTksNC4wMzgzMjU0ODEwMTAxNThlLTE1LDAuMCwtNi42MTQyMTU4MjQ1MTIyN2UtNDkNCjMwLjAsNzAuMCwxLDI3NiwyMDgsMC4wNzkzNzMzODA0ODg3NTk2NSwtNi4xMzYxOTczMzc1Nzc0NTcsLTY1LjM4MTQwMzk5MTk3NTUzLDQuNDQwODkyMDk4NTAwNjI2ZS0xNiwwLjM3MDM5MTI2MzY4Nzc3NTksMTcuMzMzNzE4NjQ0OTgzMDksMC4wLC0wLjE2NjUwMDE1NTQ3NjMzODE3LDczODUsNTQyNSwyNDAuODQ2NDY0MzM3NzU0MDQsMC44NDQ4NDU4MzMwMDUwMTc4LDAuMTIyNzU5MzY2ODc2NzU3MjcsMC4wMzIzOTQ4MDAxMTgyMjQ3NjUsLTc2LjE1MjEyODYwOTYyMjE1LDM5LjQ5NjM2NTk2NDAwNTI5LDkuNDU1OTE2ODM4MDA5MTI0LDQuMTc2ODk0Mzg2OTM1MDg3LDAuMDA4MzEyMzg3ODUzNDAxNDYyLDAuMDA4OTAwODY2MzM5MTYxNzQ4LDAuMjU2OTI4MDM1MTg3MDY3NSwtMTc5LjkyNzE2NzI5MDY5MTc4LDY5Ljk4MDExMDQ2ODg0MzYyLDQyLjQwMjY1MjA2NzE2MDY2LDc0NTIuMTM4OTA1NTEyNzExLDE3Mi45MDU1OTMzNjA0NzU4LDI0MS40NTY2Mjc5MzM5MDM0LDAuODc3MzIyMDg2MDQ0NTI5MywwLjg3NzMyMjA4NjA0NDUzMDcsMS4zMjUzMzg0NDgyMzQwOTI1ZS0xNSwtMy4wNTcwODI5MTkxNDU5MDFlLTE3MSwtMS4wODU0Mzk2Mzk5Njc5NjIyZS00OSwwLjQ5NTY1NzM2NzE3NTU2MzYzLDAuNDk1NjU3MzY3MTc1NTY1MywxLjcwNDAwNjU3NjMwMDk3NmUtMTUsLTMuMDU3MDgyOTE5MTQ1OTAxZS0xNzEsLTcuNTAwMDg0MTkzNzA0NzgyZS00OSwtNS41Mzg2MTM2ODM5NjIwODc1ZS0wNiwtNS41Mzg2MTM2Nzk3OTQ3MzllLTA2LDQuMTY3MzQ5MzI0OTAxMjk1ZS0xNSwtMy4wNTcwODI5MTkxNDU5MDFlLTE3MSwtNy41MDAwODQxOTczODQ4OTdlLTQ5DQo0MC4wLDcwLjAsMSwyNzYsMjA4LDAuNzk0MDQ1OTM1Mzk0NjEyNywtNS4zNDQ5MjMxODkwOTA2OSwtMTI2LjczODkwMDQxMDc5NDg4LDIuMjIwNDQ2MDQ5MjUwMzEzZS0xNiwwLjQ0MTkwNjA4NjE0MzA1NywyMC4yNTY4MjkzMDY1NTQ5NywwLjAsLTAuMTcxNDc1MjI3MjM1NDU3NzcsNzQzNSw2MjAwLDM4OC40ODYzMjExNTg2NTE5NSwwLjUxNDQ4Njg3NzgxMDM1NTUsMC40MzAzMzc4NjI3MDQ2MjAxLDAuMDU1MTc1MjU5NDg1MDI0Njg0LC0zOS42MDQ0OTUwOTM2MDE1OCw1MC41NDE2OTQ2MTQwMzA4OSwxNS4yODA3NjIzODc0NzA3NiwzLjMwNzUzNzQ5OTI3MjQwNjUsMC4wMDc0NjMxOTI5NDE1MDIxMSwwLjAxMzExNDIxNjEyNDQyMTUzNCwwLjIyOTE0MjYyNDMwNjM3ODA1LC0yMjcuNjI0MzA0NzQxNjY1NzgsMTMwLjUzMDcxNDA2NDY4MzMsNDYuNTgzNzU1NTI0OTI4NTM0LDcwMzEuNTc1ODYzMzgyNTMxLDEyMS4xNTcwNzE5NzAwODQ5NCwzOTIuMDM4NzIxNjU2NzgzLDAuOTg3NTY2NzAxNDY4NjI1MSwwLjk4NzU2NjcwMTQ2ODYyNSwtMS44OTMzNDA2NDAzMzQ0MTc5ZS0xNiwtNy41NjI4ODc0OTAyMTM4NTdlLTEzOCwtMS4zNjA5NjU3MjkzMTE0NTAzZS00OSwwLjQ5Nzc5NzgyNzgzMTUxNTA1LDAuNDk3Nzk3ODI3ODMxNTE3MywyLjI3MjAwODc2ODQwMTMwMTRlLTE1LC03LjU2Mjg4NzQ5MDIxMzg1N2UtMTM4LC03Ljc3NTYxMDc5MDc0MDc2ZS00OSwtMi4wMjM3ODEzMzA3MjIxMjQ2ZS0wNywtMi4wMjM3ODEyODkxNTk1MjA4ZS0wNyw0LjE1NjI2MDM2MjE0Nzk5N2UtMTUsLTcuNTYyODg3NDkwMjEzODU3ZS0xMzgsLTcuNzc1NjEwNzk0NDIwODc2ZS00OQ0KNTAuMCw3MC4wLDEsMjc2LDIwOCwxLjM3MDAyODc1ODYzOTM5MDksLTQuMDc1MjM2MzM0MzUxOTAxLC0xNzUuNDk0OTM4MjAwNDk3ODQsNC40NDA4OTIwOTg1MDA2MjZlLTE2LDAuNTAwNzI0MzYyODMwNTg2MSwyMi4xNTA3ODgzMzcxOTMxMTMsMC4wLC0wLjE4Njg3ODMxMzQ1NDM0NTQsNzQ0MSw2OTMzLDQ5OS43ODc0NzQxMTM5MDY2LDAuMTI3MTE1MzM3Njg5NzM0MjQsMC43OTg0MDQzMDI4NTY4NDYyLDAuMDc0NDgwMzU5NDUzNDE5NjYsMjIuMjYwMzEzMjE3NTIxNDc4LDY0Ljc2MDM3NTE2OTI5OTg4LDIyLjA0OTU0NTA4NzczNTMzLDIuOTM3MDM5MDU5NjA5NTIwNCwwLjAxNzA1MjAzNjQ4MDk2MDA4LDAuMDI0MzE2MDU2MDU1NDAxNTA1LDAuMjI4MTY2MzIzMzEwMzg5NywtMjY0LjAyNTE5NzEwNjg4NzQsMjA0Ljg0Mjg2Nzg3MTA2NzQsNDIuOTk4MjE0MTE4NDUwNTMsNjkzMi41MDc4MTgzODE4MzgsMTM0Ljk3MjIzNzA0MjU1NzgyLDUwOC44Nzk0MjIxODc0MDYsMC45ODg0MjEyNDY0MzQ5ODk0LDAuOTg4NDIxMjQ2NDM0OTg4NCwtOS40NjY3MDMyMDE2NzIwOWUtMTYsLTQuNDk3NDY4Mjk4NzU4NzgxNmUtMTA3LDguMDY5MTA2Njg1NjU1MDEzZS00OCwwLjQ5NzQwNzEzMjY1MjEwMDgsMC40OTc0MDcxMzI2NTIxMDMwNywyLjI3MjAwODc2ODQwMTMwMTRlLTE1LC00LjQ5NzQ2ODI5ODc1ODc4MTZlLTEwNyw3LjQyNzY0MjE3OTUxMjA4M2UtNDgsLTEuMTE1OTMwNjQzMzQ2NjQyMmUtMDcsLTEuMTE1OTMwNjAzNTgyOTA4ZS0wNywzLjk3NjM3MzQyMzc2OTI1NmUtMTUsLTQuNDk3NDY4Mjk4NzU4NzgxNmUtMTA3LDcuNDI3NjQyMTc5MTQ0MDczZS00OA0K', 'CP35_PAIRED.csv': 'bWVzaCx0X015cixmaW5lX2ZyYWN0aW9uLG5lYXI0MCxCbmVnNDAsbmVhcl9ydCx4X2NlbnRyb2lkLEosVDUwLFQ5MCxtZWFuX1QsZGVsdGFfbWVhbl9ULGV4Y2Vzc19zaW5jZTIwXy00MCxkZWx0YV9pbnZlbnRvcnlfLTQwLGV4Y2Vzc19zaW5jZTIwXzAsZGVsdGFfaW52ZW50b3J5XzAsZXhjZXNzX3NpbmNlMjBfNDAsZGVsdGFfaW52ZW50b3J5XzQwLGxvd2VyX2JvdW5kX2lubmVyX291dHNpZGU0MCxsb3dlcl9ib3VuZF9pbm5lcl9vdXRzaWRlX3J0CmV4cGFuZGVkLDIwLjAsMC45OTk5OTk5OTAyODU5NjE4LDAuMDA4OTc3MDYxOTQwMDY1MDEzLDAuMDA4OTA1OTE0NTAzMTAzNzIxLDAuNDY4MDc0NzI0MzQ1MjMzNTYsLTEuOTgwOTg3NTcxMDc1NDMyLC0wLjgwNDM5MzY0NjMzNTc0NjUsNDcuNDQyNDIwMTYwMDc5ODksOTAuOTY0NzQyMTczMjgyODksMTY2LjEzMTMxODU1ODI3NzE0LDg5Ljc3MzkwNzY0Mzk5NDAyLDAuMCwwLjA3MTY5MjE2ODczMzE1OTQ5LDAuMCwtMC4wODMwNDMxMzA0MjYzNDYxNSwwLjAsLTAuMDAxMjQ5Njc0NTk3MDAwMjA1NCwwLjA2MjcxNTEwNjc5MzEwMzM4LDAuMApleHBhbmRlZCwyMi4wLDAuOTk5OTk5OTkyNDEwOTMyNSwwLjAwODk3MjM4MTA3MjY2Mjk4LDAuMDA4OTE0NjE2NjYyOTI4Mzg2LDAuMzg2NzU0NjgzNjUxMzU1NCwtMTQuNDk4NDAzMjcwMjE1MjU0LC0wLjYwMTg3MTA2ODEyMDk5ODQsNDkuMDk4MTE4NjE1NjgyNjQsOTQuNjE1NTc1NTY4NjQ4MSwxNzkuMTA4OTgzMzA2NTY3ODYsMTAyLjQwNDQ5NDg4NzgzNjEyLDAuMDU4MDUxMzE2NzM5OTcyNDE1LDAuMTI5NzQzNDg1NDczMTMxOSwwLjMyMDA2MDAyNDg3MjEzMDY0LDAuMjM3MDE2ODk0NDQ1Nzg0MjIsMC4wMDA4NzYxODgyNTY3MTE3MzAyLC0wLjAwMDM3MzQ4NjM0MDI4ODg4NjYsMC4xMjA3NzExMDQ0Mjk0MTA3LDAuMApleHBhbmRlZCwyNC4wLDAuOTk5OTk5OTkzODEwMDQ3NCwwLjAwODk2NjQyNDkxNzIxMjg0MSwwLjAwODkxMTE2NDgwMzI3MTc4LDAuMzM1MzY5MDk3OTk4Mjk0OCwtMjcuMTcyMTMwMzgxMzYwNzEzLC0wLjQxNjc1MjQzNjkzMDA1ODIsNTAuNTM3OTQ4NDgyMTExMjQsOTkuNjU5NDY0MDM3MjQ5MTgsMTkyLjYzMzY5NjE0MzI3Nzg2LDExNS45MjQ1MzA5MTU0MTU1NiwwLjEyNTg0NzM0Mzc0ODMzMDIzLDAuMTk3NTM5NTEyNDgxNDg5NjUsMC41MTI2MDE3Njc0MzI0ODE4LDAuNDI5NTU4NjM3MDA2MTM1NCwwLjAwMTEyNzg4MjkxNjczMjY4ODIsLTAuMDAwMTIxNzkxNjgwMjY3NzEzMjUsMC4xODg1NzMwOTQ0NDEwNDA0OCwwLjAKZXhwYW5kZWQsMjYuMCwwLjk5OTk5OTk5NDcyMjAxMzQsMC4wMDg5NjA2NTQ4MDA2MDAyNDgsMC4wMDg5MTE5NjY2MzI4NzA2MDksMC4zMDA0OTcxNDg3MDgxNTcyNSwtMzkuOTE5MjcyOTI3MzA3OTIsLTAuMjQ1OTUyNTg2NTE5MDE3Nyw1MS41MDYzMzg4MjgzNTM2ODQsMTA4LjEwMTc0Mjk1Njc0NDIzLDIwNi41NzUzMjI4MDY0ODU3MywxMjkuNjA2Nzc2NTY3OTcwMzEsMC4zMjE2ODk4NjcyOTYxMTI1NCwwLjM5MzM4MjAzNjAyOTI3MTksMC41NzE2NTc3OTk5OTg0MTA1LDAuNDg4NjE0NjY5NTcyMDYzNTYsMC4wMDEyMDkwODEzMDM1NzI4Mzg0LC00LjA1OTMyOTM0Mjc0MjY2OGUtMDUsMC4zODQ0MjE2MzI5MTgzMzg5LDAuMDkyODg1MTM5MDEwNzgxOTQKZXhwYW5kZWQsMjguMCwwLjk5OTk5OTk5NTI5ODE0MywwLjAwODk1NDUyNzU5NzI0Mjc2LDAuMDA4OTM0Mjk1NTg4MzEzMjE0LDAuMjc1MjIzNDM0NDczMzQyNjMsLTUyLjY2Njk4NDIyMDc2ODg1LC0wLjA4Mjg4ODk1NTc0NjA4MzMyLDUyLjc5NDY1MDk1ODk1MjksMTE2LjkwNjYzNDYwNTU4NTg1LDIyMS4xODcxMzg5OTQ4ODg1OCwxNDQuMjI0MTQ1MzgzMjcwOTgsMC42MjUwMjU0OTEyODA5OTY4LDAuNjk2NzE3NjYwMDE0MTU1NiwwLjU3NzIyMDE5NzU0NDQ1MDYsMC40OTQxNzcwNjcxMTgxMDQzLDAuMDAxMjM3MDMwOTEyMDc1OTczNCwtMS4yNjQzNjg0OTI0MjYyODIxZS0wNSwwLjY4Nzc2NTEwMTgxNTAyNiwwLjQyMTQ5NjE5NDkzODkyNjIKZXhwYW5kZWQsMzAuMCwwLjk5OTk5OTk5NTY1MzQyLDAuMDA4OTQ3MjU5MDA1MzA3MjQ3LDAuMDA4OTM1NzQ2NzY5MjA1NzQ2LDAuMjU2OTM0MTQzNjg2MDEwMzYsLTY1LjM0NjkwNDAyOTMwNjAzLDAuMDc2MTUzNzI1NTU4NTQ0MDgsNTMuOTYwMjIxNzk5MzA0NTQsMTI4LjgxMDYzMzA3MzU5ODY3LDIzNi42MzU3ODEwNzg4OTEzNywxNTguOTU5OTMyOTU3NzI1NzUsMC44MDU2Mjg5ODY5MzQwODM1LDAuODc3MzIxMTU1NjY3MjQyMiwwLjU3ODQyNjA4MDExNjY1ODQsMC40OTUzODI5NDk2OTAzMTE5NCwwLjAwMTI1MDI3OTQ2OTUzNDg0Myw2LjA0ODcyNTM0MjgyMjA4MWUtMDcsMC44NjgzODAzNTExMzM4NzI2LDAuNjIwMzkzNDY2NDUzMTY5NApleHBhbmRlZCwzMi4wLDAuOTk5OTk5OTk1ODY5ODgzNSwwLjAwODk0MTMyNjkyMDE5NzI4LDAuMDA4OTMwNjM5NDQxNTE5NDY3LDAuMjQ0NzY2OTQ4MzU3MzM3OSwtNzcuOTAxODk0NjMzMTEwNjMsMC4yMzMwNjcwMDA5NjcyNzAxMiw1Ni4xNDA0NDA1ODY4MTAwNiwxNDMuNTEwMzc3Mjk2Nzk3OTcsMjU0LjI1NzE0MDkxMDA3ODM2LDE3Ni41MzQwMTA4MDQ4NDMyLDAuODg0NDYwMDAyNjA5Mjk4NSwwLjk1NjE1MjE3MTM0MjQ1NjMsMC41NzkxNTE4MzYwMTAwNjg5LDAuNDk2MTA4NzA1NTgzNzIyMiwwLjAwMTI2MjQ5MTY2Njk5NTk1OSwxLjI4MTcwNjk5OTU2NzQyMjRlLTA1LDAuOTQ3MjI2MDg5MjI2NTc0OCwwLjcxMTQwMDQ2Nzc4OTQzNDIKZXhwYW5kZWQsMzQuMCwwLjk5OTk5OTk5NjAwMzQ2MzksMC4wMDg5OTMyMTc4NjMzMzg1NDQsMC4wMDg5Nzc1OTA0ODA3NDU3MTYsMC4yMzYyMzgyNzUxNDI2NzY3MywtOTAuMjY4NzQ1NzYzMzk0MjEsMC4zOTA2NTQxNDI4Nzc2ODk4NSw1Ny40MDQ5MTIyNzIwMTA3NSwxNTYuMzcyNjg2MjYwMjkyMDMsMjcyLjQ1MzE0OTQ5NjQzMzQsMTk0LjQwMDk5NjUzNTQxNzU2LDAuOTEwNzE2ODExMzgzOTc3MiwwLjk4MjQwODk4MDExNzEzNjEsMC41Nzk2ODU1MTAwOTAzOTA3LDAuNDk2NjQyMzc5NjY0MDQ0NDUsMC4wMDEyNzY1Mzc2MTkxMzI5ODEsMi42ODYzMDIyMTMyODM5MTk4ZS0wNSwwLjk3MzQ0NTA3NjIwNDUwNTUsMC43NDYyMDAwMTg5MjUxNjczCmV4cGFuZGVkLDM2LjAsMC45OTk5OTk5OTYwOTE3MzQzLDAuMDA5NTI2ODQxNjI1ODgzMjg3LDAuMDA5NTA3MzYyODA0MDU5NjY4LDAuMjMxMTc0Nzk0NzgzNzk5NTcsLTEwMi4zNzM5NDY4MDg2OTY1NywwLjU0OTQ5MDY5NzMxNjI1MTIsNTkuMzgwOTQ2NTI1MDg3OTg1LDE2OS40MDg4ODc0NzE2ODc1LDI5MC4yMzUxMDM4MjQwNTIxNSwyMTEuNTkxNTc5NzM0ODA2MywwLjkxNDE5MzkxNTQxNDM5MDQsMC45ODU4ODYwODQxNDc1NDk0LDAuNTgwMTUxMDMzMDYyNTY2OCwwLjQ5NzEwNzkwMjYzNjIxOTk2LDAuMDAxMjg4MTQyNDc5Mzk0MzUsMy44NDY3ODgyMzkzNzAwMTRlLTA1LDAuOTc2NDAyNjc4MTQ5MDEzOCwwLjc1NDc1NDcyNDk5MTA5NzUKZXhwYW5kZWQsMzguMCwwLjk5OTk5OTk5NjE0NDk3MTYsMC4wMTA5NjU4MTM5ODM5ODc0NDMsMC4wMTA5NTA2NDY2NDgxNDg0NjksMC4yMjgwMzE0NjUxNjc1MDQ1OCwtMTE0LjEzOTk1MjAwNDU2MDgsMC43MDU2MzY2NTg3OTAxMDgzLDYwLjc4NjEwMDIzMzQ3MTEyLDE3OC44MzQ1Nzg0NDIxOTEwNiwzMDYuMDY1MzAzMTk4NjgwOCwyMjcuNjM2Mzk4OTE3MDY2OSwwLjkxNTIxODU5MDY4NzI3MzgsMC45ODY5MTA3NTk0MjA0MzI3LDAuNTgwNDk1MTM5MjUxNTk3MSwwLjQ5NzQ1MjAwODgyNTI1MDMsMC4wMDEyOTkwMzU2MDU5MzM2NzUyLDQuOTM2MTAwODkzMzYwMTYyNGUtMDUsMC45NzYwMDA0NzExMDI1ODU1LDAuNzU4OTM0ODE5OTE5MDY4MwpleHBhbmRlZCw0MC4wLDAuOTk5OTk5OTk2MTQ3ODc3MywwLjAxMzI2NjEyNDc1NjMzMTIxLDAuMDEzMjUwMzQ1NzM4MDU0Mjc4LDAuMjI2NTMwNjkyOTQ4NzkwNzcsLTEyNS41MDY3MTIxNzkxMzc1NSwwLjg1NTU4NDY5ODg5Njk1MjMsNjIuMTMyODcyODQ0Njc1MTYsMTg0LjYwNjU4NDUwMTA1NDczLDMxOS44MjU4NzQyMTI4OTY2LDI0MC42NTY2ODEwNDQzNjc2MiwwLjkxNTc5MDIzMzg0MzkwNDcsMC45ODc0ODI0MDI1NzcwNjMyLDAuNTgwNzUwMTI4NzczODg4OCwwLjQ5NzcwNjk5ODM0NzU0MTk1LDAuMDAxMzEwMTI3MzExNDUyNDc4Nyw2LjA0NTI3MTQ0NTE4NzUxN2UtMDUsMC45NzQyODQ3ODk0MjcyODE3LDAuNzYxMDIwMjIxMjM0ODIyMgpleHBhbmRlZCw0Mi4wLDAuOTk5OTk5OTk2MDYyMjA4OSwwLjAxNTg3OTYyMTg2ODA5NjE4MiwwLjAxNTg2NDY4MzIxMzM1MzQ1NCwwLjIyNzI2MjUwNDA1MDE4MDAzLC0xMzYuNDI4MTgwNzYwNDIwNiwwLjk5ODYxMTI1MDI3NTkyNzUsNjMuNTkwNzUzNjUxMTMwOTQsMTgzLjY4ODQzNDcyMTM1NTMsMzMxLjI2MzI1OTU3OTU3NDQsMjUxLjA0NjA4ODE1NDAxODEsMC45MTYxNDQwODY0OTk0NTA2LDAuOTg3ODM2MjU1MjMyNjA4NywwLjU4MDkyMTQ4MDgxNDkxMzIsMC40OTc4NzgzNTAzODg1NjYyNiwwLjAwMTMxNjg1MTk5ODkyNjg2NjUsNi43MTc3NDAxOTI2NDEwNDJlLTA1LDAuOTcyMDMzNjk4NzAxNDU3OCwwLjc2MDY1MDgxNjUxOTM3NApleHBhbmRlZCw0NC4wLDAuOTk5OTk5OTk1ODI5NTY5LDAuMDE4MTgxMTU3OTUzNjgzNDg2LDAuMDE4MTY2MzQ4MjQ3MjE5NjMyLDAuMjI3OTIxODQwMTI4Mzk5ODMsLTE0Ni44Njg5NDIzMTIzMDU4MiwxLjEzMzc3MzQxMTU4NDEyMTEsNjUuMjYzMzMzNDE0MDk3MTUsMTkyLjQzMjI4MTc0MzE4NDAzLDM0MC4xOTQzNTY0MDYwMzc0NiwyNjAuMjU2MzE0MTM3NjYwMywwLjkxNjM2NDIzOTE2MTk0MjUsMC45ODgwNTY0MDc4OTUxMDE1LDAuNTgwOTQ2MjU3MzMxMzg3NiwwLjQ5NzkwMzEyNjkwNTA0MTA3LDAuMDAxMzIxNjg0NjE5NzExMjA0LDcuMjAxMDAyMjcxMDczMzQyZS0wNSwwLjk2OTk1ODk5MTk5MDc3OCwwLjc2MDIxODMwOTgxNjA2MTUKZXhwYW5kZWQsNDYuMCwwLjk5ODYyMDIwODIxNDc5ODgsMC4wMTk4NTQxNDM4NTg0NTcyMDIsMC4wMTk4MzIzMjIyNTgzODUwMjUsMC4yMjk1NDM1NDYxNTE2MDUwNSwtMTU2LjgwMTc5MDU0Nzg1MjcsMS4yNTg1OTcyMjI4NjQ1MzYsNjQuODc3OTMyMTI3NzU1NjgsMTg3LjYwMjU0NDI0MjAxMjMzLDM0Ni40NDkyOTI0OTU0OTEyLDI2Ni43ODMxMzYwNTQ3NjI5LDAuOTE2NDk3MzkxNzgwMDA1MiwwLjk4ODE4OTU2MDUxMzE2NSwwLjU4MDg2NTA0OTk5MjM4OTYsMC40OTc4MjE5MTk1NjYwNDI5NCwwLjAwMTMyNDEwMzEzNTM0Mjk4Niw3LjQ0Mjg1MzgzNDI5NDEwNWUtMDUsMC45Njg0Mjg0MzE2OTk3OTQyLDAuNzU4NzM5MDI5NDA2NjQ2MwpleHBhbmRlZCw0OC4wLDAuOTg0MTU0NDQ2NjkxNTczOSwwLjAyMDk4MjA1NTE2MDg0NDIwMiwwLjAyMDk0NTgzNzY5Njg5NjU0NiwwLjIzMTk5MDY3MjkxNzk1NDQyLC0xNjYuMjAyMjYzOTA4NTg0NDYsMS4zNjQ1MjI4OTI0MDM3OTI5LDYzLjc5MzAwMzEyMTczMzc3NiwxOTIuNDU4NjgzMTU2MDY5MjQsMzUwLjg3NDYxODEyNTExNDcsMjcxLjM0MDQ1NTY4MjI3MTgsMC45MTY0NTczNjIzNDY4NzE4LDAuOTg4MTQ5NTMxMDgwMDMwOCwwLjU4MDY5ODYxNjU2OTEyMzEsMC40OTc2NTU0ODYxNDI3NzU2LDAuMDAxMzI2MTEzMTUwOTEwMjU4OCw3LjY0Mzg1NTM5MDk1OTY4NmUtMDUsMC45NjczODIyNzkxMjU4NDEyLDAuNzU2MzczNjYxMzY4NzMxCmV4cGFuZGVkLDUwLjAsMC45Nzk0Mzc3Nzk0ODUzMDI2LDAuMDIxMjYwNDU1OTE5OTA1OTQ0LDAuMDIxMjM0MTY4MzQyNjc3MDM0LDAuMjMzNzg3MzIyMzgzMDM0NzYsLTE3NS4wNDk4MTUyMDc1NzM0LDEuNDU2ODA4NDM0NTczMTU5LDY0LjE3MjI2MzgwNTM3MzE5LDE5OC4wNTE3NTA1OTIwMzQ3LDM1NC4xNzA0NjI3MjQyMjY4LDI3NS4zNTc3NDU3MDg2Mjg0LDAuOTE1NDkwNDU4NjYyNDIyNCwwLjk4NzE4MjYyNzM5NTU4MjIsMC41ODA0NjUzMjU5MzkyNTMzLDAuNDk3NDIyMTk1NTEyOTA1OSwwLjAwMTMzNzM0Mjc5MjI5MzA3MjMsOC43NjY4MTk1MjkyODIyNjZlLTA1LDAuOTY3MTM5OTk1MDI4MzQyOCwwLjc1NDYxMzEyODU2NTIxNApvcmlnaW5hbCwyMC4wLDAuOTgzNzAzMDM3MzIxOTEzNywwLjAwODk3Njk5NzU4NjUwMDI0OCwwLjAwODkwNTkwODAzNzc4MTg1NiwwLjQ2ODA3NDcyNDM0NTIzMzQ1LC0xLjk4MDk4NzU3MTA3NTQzMiwtMC44MDQzOTM2NDYzMzU3NDYxLDQ3LjQ0MjQyMDE2MDA3OTg5LDkwLjk2NDc0MjE3MzI4Mjg5LDE2Ni4xMzEzMTg1NTgyNzcxLDg5Ljc3MzkwNzY0Mzk5Mzk1LDAuMCwwLjA3MTY5MjE2ODczMzE1OTQ2LDAuMCwtMC4wODMwNDMxMzA0MjYzNDYxNSwwLjAsLTAuMDAxMjQ5Njc0NTk3MDAwNDI3NSwwLjA2MjcxNTE3MTE0NjY2ODExLDAuMApvcmlnaW5hbCwzMC4wLDAuODEyNDU1MjI1NDc4NzY1OCwwLjAwODkwMDg2NjMzOTE2MTc0NCwwLjAwODgzNTg5OTE4MzcyMzQ0NCwwLjI1NjkyODAzNTE4NzA2NzQsLTY1LjM4MTQwMzk5MTk3NTUzLDAuMDc5MzczMzgwNDg4NzU5NjUsNTUuODY3NDg0OTc3OTQzNDIsMTMzLjQxNDM5NDg4Mzc3MTIsMjQwLjg0NjQ2NDMzNzc1NDA0LDE2My4xNjk4NjI3NjIwNjU2LDAuODA1NjIzNDYzNDY5NzY5MiwwLjg3NzMxNTYzMjIwMjkyOTYsMC41Nzg0MjU3NTQ0MzAyMTMsMC40OTUzODI2MjQwMDM4Njc1NywwLjAwMTI1MDI2NzU2MTE5Mzc2MjUsNS45Mjk2NDE5Mjk2OTE2OThlLTA3LDAuODY4NDIxMjE5NzA1MzY4NiwwLjYyMDM5NDA1MDg1NzQ2MjkKb3JpZ2luYWwsNDAuMCwwLjA0MDY1NDQ1OTc2MDEyMzQ0LDAuMDEzMTE0MjE2MTI0NDIxNTMsMC4wMTMwNjE5MjM3MTk5MzM1NTksMC4yMjkxNDI2MjQzMDYzNzgwMiwtMTI2LjczODkwMDQxMDc5NDg4LDAuNzk0MDQ1OTM1Mzk0NjEyNyw5Ni41Njg2MzY1NTIwMzY2LDM4MC4wMTAxODE5Mjg5ODY0NywzODguNDg2MzIxMTU4NjUxOTUsMzA5LjMxMjg2Mjg3Mjg4MzY3LDAuOTE1ODA2MDAyNTg0NTY1OCwwLjk4NzQ5ODE3MTMxNzcyNDgsMC41ODA3NDc3NDU5NDczMDc1LDAuNDk3NzA0NjE1NTIwOTYyNywwLjAwMTMxMDE1NzMwNzc2ODg1MjQsNi4wNDgyNzEwNzY4NjM4OThlLTA1LDAuOTc0NDUyNDg1MzQ0MjAzMywwLjc1ODQyNDA3NzE2MjI0NjgKb3JpZ2luYWwsNTAuMCwwLjAyMDA3NDExODAzMDMyMzc3MywwLjAyNDMxNjA1NjA1NTQwMTQ5NSwwLjAyNDIwOTUwMzExNzE2NjYzOCwwLjIyODE2NjMyMzMxMDM4OTYsLTE3NS40OTQ5MzgyMDA0OTc4NCwxLjM3MDAyODc1ODYzOTM5MDksMTMzLjQ5MjA2OTA1MzczNzY1LDcyMS4wMjEyNzU3OTEzMzA3LDQ5OS43ODc0NzQxMTM5MDY2LDQyMC45NzA2NTEwODgxMzcyLDAuOTE1NTEwNDAzNDkxNzk2OCwwLjk4NzIwMjU3MjIyNDk1NDksMC41ODA0ODQ3NjYyMzY0MjkyLDAuNDk3NDQxNjM1ODEwMDg0MjcsMC4wMDEzMzczNTY4MDU1Mzg4OTEzLDguNzY4MjIwODUzODI3NTU3ZS0wNSwwLjk2NDEwNTE5MDM3OTU4NjUsMC43NjAyNTQ5MjMxMjQ1OTg1Cg=='}
for name, value in embedded.items():
    expected = base64.b64decode(value)
    visible = source/name
    if visible.is_file():
        payload = visible.read_bytes()
        if payload != expected:
            raise RuntimeError("Release file differs from notebook copy: " + name)
    else:
        payload = expected
    (root/name).write_bytes(payload)
(root/"manuscript").mkdir()
env = dict(os.environ, OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
env.pop("CP36_END", None)
def run(script, *args, cwd=root):
    subprocess.run([sys.executable, str(root/script), *args], cwd=cwd, env=env, check=True)
for dm in (0, 1):
    d = root/f"full_DM{dm}"
    if mode == "replay":
        if packed:
            d.mkdir()
            for i in range(1,5):
                with zipfile.ZipFile(source/f"DM{dm}_data_{i:02d}.zip") as archive:
                    if archive.testzip() is not None:
                        raise RuntimeError("Damaged input archive")
                    for name in archive.namelist():
                        if Path(name).name != name or (d/name).exists():
                            raise RuntimeError("Unexpected or duplicate archive member: " + name)
                        (d/name).write_bytes(archive.read(name))
            expected = {f"frame_+70_{dm}_{t:+06.1f}.npz" for t in range(-50,51,5)} | {"configuration.json",f"h1_phase+70_dm{dm}.csv",f"h1_phase+70_dm{dm}_state.npz"}
            if {p.name for p in d.iterdir()} != expected:
                raise RuntimeError("Incomplete saved simulation fields")
        else:
            shutil.copytree(source/f"full_DM{dm}", d)
    else:
        d.mkdir()
        run("CP36_RUN.py", str(dm), cwd=d)
for script in ("CP37_ANALYZE.py", "CP38_KINEMATICS.py", "CP38_PLOT.py", "CP39_JOINT_CHECK.py", "CP40_VERIFY.py", "CP46_DIAGNOSTICS.py", "CP46_VALIDATE.py", "CP46_PLOT.py"):
    run(script)
out = source/(root.name+"_results.zip")
with zipfile.ZipFile(out,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as archive:
    for p in sorted(root.rglob("*")):
        if p.is_file() and "__pycache__" not in p.parts:
            archive.write(p,p.relative_to(root))
with zipfile.ZipFile(out) as archive:
    assert archive.testzip() is None
print("Mode:", mode, "Results:", out)
